In [ ]:
## Run this cell FIRST

import subprocess
import time
import sys

print("Starting Safe YARN Cluster Cleanup...")

# 1. Clean up local zombie spark references before querying YARN
if 'spark' in sys.modules or 'pyspark' in sys.modules:
    try:
        from pyspark.sql import SparkSession
        active_session = SparkSession.getActiveSession()
        if active_session:
            print("Stopping local legacy session safely...")
            active_session.stop()
    except Exception:
        pass

try:
    # 2. Query YARN for active sessions
    cmd_list = "yarn application -list -appStates RUNNING,ACCEPTED"
    result = subprocess.check_output(cmd_list, shell=True, text=True)
    
    lines = result.strip().split("\n")
    apps_killed = 0
    
    for line in lines:
        if "application_" in line:
            app_id = line.split()[0]
            print(f"Killing conflicting YARN session: {app_id}")
            subprocess.call(f"yarn application -kill {app_id}", shell=True)
            apps_killed += 1
            
    if apps_killed > 0:
        print("Waiting for resource managers to clear memory allocations...")
        time.sleep(6)
        print("Cluster environment successfully normalized!")
    else:
        print("Cluster queue is already wide open and clean.")

except Exception as e:
    print(f"YARN query bypassed: {e}")


In [ ]:
## Run this cell SECOND

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic

print("Allocating new Spark Session...")

spark = SparkSession.builder \
    .appName("MobilitySpark-Fresh-Start") \
    .config("spark.master", "yarn") \
    .config("spark.submit.deployMode", "client") \
    .config("spark.executor.instances", "2") \
    .config("spark.executor.cores", "1") \
    .config("spark.executor.memory", "2g") \
    .config("spark.driver.memory", "2g") \
    .getOrCreate()




spark._jvm.org.mobilitydb.spark.generated.GeneratedSpatioTemporalUDFs.registerAll(spark._jsparkSession)

# Acceder a la clase de Java a través de Py4J y registrar las funciones
#sc = spark.sparkContext
#generated_udfs = sc._gateway.jvm.org.mobilitydb.spark.generated.GeneratedSpatioTemporalUDFs
#generated_udfs.registerAll(spark._jsparkSession)


print("Spark Session successfully attached to YARN!")
    

In [ ]:
## Run this cell THIRD

from pyspark.sql import SparkSession
from IPython.core.magic import register_cell_magic


@register_cell_magic
def sql(line, cell):

    # Eliminar comentarios SQL de líneas completas
    lines = []
    for line in cell.splitlines():
        if not line.strip().startswith("#"):
            lines.append(line)

    cell = "\n".join(lines)

    # Separar sentencias
    statements = [
        s.strip()
        for s in cell.split(";")
        if s.strip()
    ]

    for statement in statements:
        print("\n==============================")
        print("RUNNING:")
        print(statement)
        print("==============================")

        result = spark.sql(statement)
        # Only attempt to call show() if the query actually returns data rows
        if result.schema:
            result.show(truncate=False)
        else:
            print("Command executed successfully.")
            

print("cell magic %%sql prepared")

In [114]:
## Run this cell FOURTH

from py4j.java_gateway import java_import

java_import(spark._jvm, "org.apache.hadoop.fs.FileSystem")
java_import(spark._jvm, "org.apache.hadoop.fs.Path")

fs = spark._jvm.FileSystem.get(
    spark.sparkContext._jsc.hadoopConfiguration()
)

# !hdfs dfs -rm -r -f hdfs://namenode:9000/spark-warehouse/flights
# print(f"Deleted files")

fs.delete(
    spark._jvm.Path("/spark-warehouse"),
    True
)

fs.mkdirs(
    spark._jvm.Path("/spark-warehouse")
)

print("spark-warehouse clean")


spark-warehouse clean


In [ ]:
%%sql

# read csv files

CREATE OR REPLACE TEMP VIEW opensky_csv
USING csv
OPTIONS (
  path 'file:///opt/data/opensky-2020-06-01',
  header 'true',
  inferSchema 'true'
);



In [ ]:
%%sql

## create opensky delta file

DROP TABLE IF EXISTS opensky_delta;

CREATE TABLE opensky_delta
USING DELTA
AS 
WITH ICAO24_WithNullLon AS (
  ## cleaning
  SELECT ICAO24
  FROM opensky_csv
  GROUP BY ICAO24
  HAVING COUNT(lon) = 0
)
SELECT * 
FROM opensky_csv
WHERE squawk IS NOT NULL 
  AND ICAO24 NOT IN (SELECT ICAO24 FROM ICAO24_WithNullLon); 

ALTER TABLE opensky_delta ADD COLUMN EtTs timestamp;
ALTER TABLE opensky_delta ADD COLUMN LastPosUpdateTs timestamp;
ALTER TABLE opensky_delta ADD COLUMN LastContactTs timestamp;
ALTER TABLE opensky_delta ADD COLUMN Geo GEOMETRY;

UPDATE opensky_delta SET
EtTs = to_timestamp(time), LastPosUpdateTs = to_timestamp(LastPosUpdate),
LastContactTs = to_timestamp(LastContact), Geo = st_setsrid(st_point(lon, lat), 4326);


In [ ]:
%%sql

## perfecto pero incompleto. Precisa luego crear Flights, para la cual se precisa unnest

CREATE OR REPLACE TEMPORARY FUNCTION sorted_array(
    arr ARRAY<STRUCT<eTts: TIMESTAMP, val: STRING>>)
RETURNS ARRAY<STRUCT<eTts: TIMESTAMP, val: STRING>>
RETURN array_sort(
    arr,
    (r1, r2) -> CASE
        WHEN r1.eTts < r2.eTts THEN -1
        WHEN r1.eTts > r2.eTts THEN 1
        ELSE 0
    END
);

CREATE OR REPLACE TEMPORARY FUNCTION array_as_string(
    arr ARRAY<STRUCT<eTts: TIMESTAMP, val: STRING>>)
RETURNS STRING
RETURN
 CASE 
    WHEN size(arr) = 0 THEN null
    ELSE
          CONCAT('[', array_join(transform(arr, x -> CONCAT(x.val, '@', x.eTts)), ', '), ']') 
END;




DROP TABLE IF EXISTS flightsday;
CREATE TABLE flightsday USING DELTA
AS
WITH 
datos_ordenados AS (
    SELECT
        ICAO24, 
        TO_DATE(eTts) AS date, 

        array_as_string( sorted_array(collect_list(struct(eTts, geo as val)) FILTER (WHERE geo IS NOT NULL)) ) as geo_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, velocity as val)) FILTER (WHERE velocity IS NOT NULL))) as velocity_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, heading as val)) FILTER (WHERE heading IS NOT NULL))) as heading_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, vertRate as val)) FILTER (WHERE vertRate IS NOT NULL))) as vertRate_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, trim(callSign) as val)) FILTER (WHERE callSign IS NOT NULL)) ) as callsign_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, alert as val)) FILTER (WHERE alert IS NOT NULL))) as alert_array_str,      
        array_as_string( sorted_array(collect_list(struct(eTts, geoAltitude as val)) FILTER (WHERE geoAltitude IS NOT NULL))) as geoAltitude_array_str
    FROM opensky_delta
    GROUP BY ICAO24, date  
   # DISTRIBUTE BY ICAO24, date
  )
SELECT
  ICAO24,
    date,
    tgeogpoint_in(geo_array_str) AS flight,
    tfloat_in(velocity_array_str) AS velocity,
    tfloat_in(heading_array_str) AS heading,
    tfloat_in(vertRate_array_str) AS VertRate,
    ttext_in(callsign_array_str) AS callsign,
    tbool_in(alert_array_str) AS alert, 
    tfloat_in(geoAltitude_array_str) AS geoAltitude
FROM datos_ordenados;


## sin unnest no creo la verdadera flights aun

## DROP TABLE IF EXISTS flights;
## CREATE TABLE flights USING DELTA
## AS



In [ ]:
%%sql

## perfecto pero para evitar unnest teniendo en cuenta solo los que tienen solo un callsign. Asi, desparece

CREATE OR REPLACE TEMPORARY FUNCTION sorted_array(
    arr ARRAY<STRUCT<eTts: TIMESTAMP, val: STRING>>)
RETURNS ARRAY<STRUCT<eTts: TIMESTAMP, val: STRING>>
RETURN array_sort(
    arr,
    (r1, r2) -> CASE
        WHEN r1.eTts < r2.eTts THEN -1
        WHEN r1.eTts > r2.eTts THEN 1
        ELSE 0
    END
);

CREATE OR REPLACE TEMPORARY FUNCTION array_as_string(
    arr ARRAY<STRUCT<eTts: TIMESTAMP, val: STRING>>)
RETURNS STRING
RETURN
 CASE 
    WHEN size(arr) = 0 THEN null
    ELSE
          CONCAT('[', array_join(transform(arr, x -> CONCAT(x.val, '@', x.eTts)), ', '), ']') 
END;



## ex flightsday porque esta es lo mismo con un solo callsign
DROP TABLE IF EXISTS flights2;
CREATE TABLE flights2 USING DELTA
AS
WITH 
datos_ordenados AS (
    SELECT
        ICAO24, 
        TO_DATE(eTts) AS date, 

        array_as_string( sorted_array(collect_list(struct(eTts, geo as val)) FILTER (WHERE geo IS NOT NULL)) ) as geo_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, velocity as val)) FILTER (WHERE velocity IS NOT NULL))) as velocity_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, heading as val)) FILTER (WHERE heading IS NOT NULL))) as heading_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, vertRate as val)) FILTER (WHERE vertRate IS NOT NULL))) as vertRate_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, trim(callSign) as val)) FILTER (WHERE callSign IS NOT NULL)) ) as callsign_array_str,
        array_as_string( sorted_array(collect_list(struct(eTts, alert as val)) FILTER (WHERE alert IS NOT NULL))) as alert_array_str,      
        array_as_string( sorted_array(collect_list(struct(eTts, geoAltitude as val)) FILTER (WHERE geoAltitude IS NOT NULL))) as geoAltitude_array_str
    FROM opensky_delta
    GROUP BY ICAO24, date  
   # DISTRIBUTE BY ICAO24, date
  )
SELECT
  ICAO24,
    date,
regexp_extract(ttext_out(ttext_in(callsign_array_str)), '"([^"]+)"', 1)   AS callsign,
timespan(ttext_in(callsign_array_str)  )  AS flightperiod,
    tgeogpoint_in(geo_array_str) AS flight,
    tfloat_in(velocity_array_str) AS velocity,
    tfloat_in(heading_array_str) AS heading,
    tfloat_in(vertRate_array_str) AS VertRate,
    tbool_in(alert_array_str) AS alert, 
    tfloat_in(geoAltitude_array_str) AS geoAltitude
FROM datos_ordenados
WHERE ICAO24 IN 
('a84fdc','3c71a1','3e7699','a9ceaf','a07d4e','484f9c','7c68dc','a84a11','475002','a658ee','3e7d89','ac77be','a3775c','4042e9','a8720f','4b4029','a7e67f','a7cb14','a0b7ae','a2a02a','a78a0e','a214d5','7c52f8','a77da1','a389e7','abd774','7c37c5','394c31','aa3126','abb72e','a68764','ab11f9','abf751','ab2eff','a9c782','ab245d','aba632','7810a9','a82791','a68d71','a9a74e','a1ebbd','a0a35f','a55880','a04f40','a1a7a2','ad3fd7','a0a385','4078fb','a68673','a1b280','a40278','a67014','489e4c','a33ecc','894f6b','a9be83','a98412','a39e37','a950c6','7585d6','a50049','ab2e4f','4063e5','a78398','ab0e21','a6a9d3','a81598','710126','ae58e2','ab62b1','a27d2e','ae1c36','7c4947','a65cb0','a1dee4','a1779e','a8b533','a0653f','a7601e','abf400','a22976','a71102','aa517a','407533','440461','400c2f','406b88','acd8c1','a39d49','484207','a1dd9d','a05ae7','ad0724','ac7493','a2467d','a73aad','a15f87','c040aa','adee36','a209a8','a8638a','ab37d6','a8be40','a0d45c','ad8bdd','a64a22','4ac994','e48038','a38106','ad7e3b','ade0d7','896271','ac6e7e','a9355e','a37e06','a9f2bd','ad9414','a5cc7d','478363','a06667','a64446','aa22e6','781590','a7fa1b','a6be9f','4400bd','a31b07','a95665','76ccd1','3ff4b9','a65f00','a19bc9','a26ce9','a361e5','a90c14','ae04d8','33ffc1','a5e795','a55d81','a29ca0','4b50fd','ae4eb9','3c0699','c06921','aa809f','a26004','4859c1','a38c78','a0e136','a7748c','a0bab4','aa9bfd','4cafad','ac2ab8','a659a3','acf3eb','a35cb0','a63853','adda93','3d3c2b','a19a84','a94214','aaefa1','a88491','781362','40504a','a6299d','3d0aa3','394a04','c02fb8','a6e1c6','c073fa','a0e96a','3dd7b8','ad1c38','a32877','4cafb4','aabb29','3ff6a9','738682','342587','781163','a67f9b','ae636f','ab5335','780627','4408a2','4d225c','3ff1ec','a28815','3fedf5','478f0b','a8fab5','a20f61','adbc07','ab252d','a2169c','4402d3','71c359','45b5d3','3d324c','a7c849','aa3681','adfe79','a32929','896c18','aa1950','a40563','a638f9','a25a09','7c42b0','86cf6f','ab4900','a64019','3ff2eb','010161','a13533','a0f60d','34504d','a02431','a63b72','a2f274','a44528','a66d51','ace13b','ad6312','a7413a','a7aba7','4601e0','a9d730','78068a','aba51d','c07c81','a0e249','a21635','a031a7','484561','ac1c67','3d5501','ad8b69','4025a6','a192a6','a5cf39','78070b','a52a5c','a3f085','a7cf78','c04aaf','47c5a2','7c1a75','503c38','a9a0b5','a87b1f','c01ff1','a53183','780722','7c4423','1406a0','a09922','a53385','ac5868','a29451','7809b7','a09b48','4850d0','a6101f','a57228','a0b8d6','ae47e4','a65f86','a0e97f','ac6f98','7c391f','abc582','a9580b','a116e8','aa689a','a5a671','a3213b','0ac936','c04755','a1b0c2','39280f','a15219','ad7992','aaa0d1','a46a7e','aae0af','a54a0a','ab00ba','4073fa','a16975','a8d665','a6bc95','c03d52','ad5398','7c6baa','a21698','ab9af5','a411d3','aa485b','a36d76','a12128','a54f84','a51918','780559','add167','aac540','a7c7c7','e481b1','ab7098','a9e962','7c0dab','a69016','a4c6c9','a5f021','a02443','a86fe4','a6a5e7','800401','ab49a8','a66a7b','a53232','ad5ae2','a0af67','a6630b','a9b3e8','a1d378','780c5c','a3144c','a7f398','a7c313','3ff839','a58a04','aaed3a','a865e3','38e03b','3feef4','a74c9f','a91f9b','a49105','49d10f','a63d02','a47244','ab3b98','4071b7','a6925e','aa0ea1','a3a2eb','ad5711','a4b95e','a1cc30','a93d7c','ac045e','3ff65a','a3f798','a03762','a5ac16','ac7aeb','7809bc','a27631','a9bbec','a9b7d4','aad34b','a01f1e','c05971','aade86','a7a8f8','a12355','738074','c05970','c01a9d','ac2268','aa2f7b','a1e0ed','a08bdd','a3b852','a16ece','884364','e49555','ae116a','a1af86','a738e9','a5102c','a14f07','aa32bd','ad4925','78156a','a0de90','a561fd','a6eb85','a44e2f','a5dbfd','a32eaf','407716','a46018','a1f059','a1ada6','a5ac8a','a7f774','a315d1','a4fd0e','accb0d','7c3ae0','aa50a0','aca248','a08347','4ba94f','48c8e3','781754','c06d26','ac99d8','adfede','ab7f7b','780e5a','a7d17f','780567','a9fb3e','a9df25','a9a092','a7c5db','a98d0c','a095ee','a36c89','3ff291','ad9569','a1137d','a052ea','a726a1','7c7eef','a205cc','3d1a99','c0469a','7c622c','3d32fe','a4f542','c079e4','aabad9','a1d00f','a9673c','a431d7','ac7e63','aaab1d','a9cf19','a928c7','a60d5f','a21e48','a0e094','a8aa26','a2e29c','a76a28','a661a4','a7ea1d','a29b62','ae4ed0','adc26e','acbaa8','a5df1e','ab5213','a4e786','3fecde','ae2027','a71d59','40487f','a96ecf','47c4e6','a5cf0c','a77107','a22535','05a054','ab6906','c05d8a','78124e','a3c74e','ab2b02','a6dea1','a0ec6e','aa319e','ab2d77','4b2ff7','7810d1','ad13c1','45c28c','440426','a17f5c','3d2ad3','a9ea6e','ae10e3','a17ac0','c08661','a3a8bb','ae04d9','3e63e9','a200e9','a8147b','3ff66d','aa86f8','a19e24','ac104a','ae0478','7c2b8a','ab4193','440437','a50b06','a40bcf','a9d7d5','aae00b','a5980f','a3a2b0','780c10','406d68','a79332','3e715e','04004b','a8dd0a','a9ecd6','a9e585','a6e6fb','7c6693','a95be8','398249','a6776f','a2fc70','a72aec','a1b174','a67422','ae4f14','780ba1','ae26b4','accd5d','a9b7b8','a0964d','781298','a4c7ad','a31cb1','a42f8b','a5eba7','75827b','ad071b','a0a7d6','ae4afb','4aaa0e','a91de9','ad3d9e','a535ff','a52d6a','a1c140','484368','a18a64','aae2b6','a8a0ab','a043ac','a3a110','a1690c','aa9963','a75141','a43240','a09d81','71030f','a13091','e47f84','341201','7380c1','a972b0','a3a5eb','a9300b','3e77d9','a05ef2','a641d4','a8b5d7','a09df5','800446','48f98c','a6225b','a5df10','abc53a','a39c5b','a12227','454a43','a6aa92','aa56b5','a0a08e','a0c06b','a2b8db','a867aa','a8a7ae','a23525','4494e8','a283cf','a82c95','a60bbd','44a3e6','50843d','a6b3b9','ac4208','a40164','3fed75','a66be6','4b2801','a7ce70','a570cd','a0aa00','4b1e69','ab6d2c','3994a5','a6869b','a3b8d1','424575','a3eb3c','adcdd9','a5047c','a35206','a20fca','a1fcfc','aa43f5','ad66f7','a36111','ad52a3','aaa97e','a68c34','a39d38','868b50','4d21e5','aa6337','a9d6ec','c828ce','acbf73','a1ebd4','a40a05','ae5772','a55e44','a25d2c','a521c8','ac3b92','a92480','c06ce7','adf6d8','39c499','a7878b','a43815','a3a98e','a5c2ea','e49593','a19bf8','7804b1','a5d74f','acf41e','a642cd','a873d2','488120','ade549','a63b17','a907f0','a5a65a','a65ab9','a7e58d','a41b8c','4b41b6','a1ea52','a3e2a9','aab21e','a9604a','a0dbf2','a0d94d','ae036c','ad48ea','4952b1','a6af49','424b68','ada982','a28e5d','8832a8','aae34c','3c70c5','a1bd0a','ac80b5','a88d04','a9a23a','a0d0fd','8880f8','7c5c67','78092a','ab7ccd','780078','acee50','a59ffc','48a5e0','ab5752','800b4b','a804bb','a344b1','4ca8fa','a638e1','4ac9e8','a41d87','a34230','a53caa','e480cc','c067a1','ac9c3f','a6f3c0','a0f2f5','7c6725','a18245','798038','a46ec9','780b1b','a725a0','aa5e88','a62d26','adf111','3d5502','88510b','a125a2','a54b06','a2f293','a5c2b4','ac7667','a2a140','c0795c','a84b83','a3ceeb','7cf9c5','a1b363','a02af5','a2bd19','a05388','ad2f6f','a127d3','a04aec','a2216b','4009d3','a4e709','70620a','a7cf2f','407902','a55870','a3ec2e','4b10f7','a1ede0','a86214','e8c072','3848f0','780770','a92e10','a41ef3','468157','a9ed89','3d271d','a5619a','a0b22e','ab2fa0','a3d63b','a00c82','aba597','ae048c','ac9bfe','a057d1','a007f1','add665','800732','a20733','7c6b98','a6bf7c','a99274','aa6321','aa26c9','ad9502','ac01a4','aae3c8','a47861','3ddc73','a4e34d','4aad35','a12701','47847f','a9854e','78087a','a9e631','a5e91e','a6acff','3d32bc','0a404c','c005c7','3d0ab2','a7b8e1','a2fd50','acfd38','a73143','a244aa','3ff426','3de586','8a0531','a63a26','a36f66','7c19d2','a2e35c','71c065','3ff83d','7c0307','a9aee8','a4a820','a16e46','a68352','acafcf','a1a83e','ab778a','7808c7','3ee2d4','48ca6e','a57d7f','a16410','a53473','a3434b','a12310','c01d92','a5c09d','a0c95e','c01be0','a90938','a7e19d','aa7288','a773ec','a1cfd2','add109','a64d46','a1ecfa','a9589c','a2a628','a04cbf','a310cf','a2d71e','a7010b','ab3662','ac1984','a958a2','3e3d6c','14fa34','a7629a','acbcae','abb288','a472ab','a25894','a64771','ae1e76','aab510','7c4cb7','a024cc','a7635e','a01fd9','ad518d','a1f3ac','a0117f','3ff737','a4efc3','a86371','a7ff29','4628b3','ad6029','a466a1','c04f80','a4e430','78067c','48fc0c','a882e6','76cda8','a69166','491115','ae60e9','49c1fe','800cee','459487','3e6bc9','a48e1f','a9a264','ab39eb','acb512','ab398f','0d04ce','a65cfd','ab75d7','401803','a1c700','acce3f','a4f948','acdde0','a3e221','a388e8','a01bd8','ab30f4','a35a46','a67ba6','710357','a0e0d9','ab7fde','a83a9b','a50242','aa68f4','a15ea4','a64294','780f1a','abd59a','a8b51e','a40d63','aba188','a3a237','ab200d','aa5990','a15f58','ac48e1','abeaa8','a69d67','c04b4d','a009ba','7c47cf','c0054a','868034','ab0ced','a5da42','a35488','ad0a4a','a47851','a10202','a12b3e','40388c','a50aeb','a97970','a594a0','c054ae','7c7794','a3f761','abee3a','c026ff','471f5c','710122','a1479d','a30753','a9eb4c','a9978f','781361','e8043a','a47f32','a0417c','ab07cd','ac9e20','a88d72','ae1211','aa60af','a0316d','3ff28d','a477d9','a9e5a7','a27dd4','3de585','a4deea','70c0c1','a85404','a12a73','48436f','a6c27f','aa6a2a','7c581d','a022a7','a0996f','a7c920','aaf0f9','a70284','a0bbb4','a6ee1c','0ac1a3','aa5610','a2b67b','3d2785','a870e2','a07cab','440618','a15d6c','aaf787','3f0099','a3a101','ac29d0','7c75e5','3e231e','ac3e65','a60e80','a67350','a45559','4ca820','a9b152','800c07','ab979e','a0de64','a75993','ace33a','a8b2f1','7818db','a08ec4','ab60a1','39adae','a4bff3','a1e709','a0717e','a6fdbc','acd7d1','a29ea2','7c8214','78120c','ac9c5a','a15375','ac29a2','a6df4b','39a96d','a4cba2','a4405f','a40660','a001b5','a9bc68','aaeb1b','3d3434','aa5c6d','aa832f','a7269d','a1ec08','a6977a','acff5c','ad954d','a43cce','600b20','3d0f7a','ac92fc','accbc6','a6bce3','a192bf','ad4419','885968','a3fc47','a7744c','49d0e6','a54ebb','a87e20','ac6fe9','c07bff','a4bb8b','aa1cfa','a1a130','aa7315','a84f5d','a71210','4b40a7','a4efc6','a29a26','a6896d','7c8190','ad1ce5','a1adab','3d1f18','a14b40','ab14c0','a57c19','a0ee4c','a2370e','a139c2','c076dd','a76d01','aa2402','aa6466','a43550','7c753a','a42653','a4b70e','a17e7a','a9872f','adb1fa','a7fef4','3ff302','73348e','a3b88e','ab246b','4bb154','a3c2a3','acd187','33fff9','ad7683','a6ab5a','47c0cf','a40de7','a1f887','a35ec7','a212d3','7c49e3','440261','a868f0','a5c545','a21ff9','ab5796','a9a734','3ef77f','800791','a79cf4','a14f69','a0b643','a52f61','a743a2','a59d88','a58ed4','a3f6dd','add23a','a7441f','7c67a7','0ac381','aa1455','a2415a','3ff6e6','a67e4d','a6a84a','488a66','ab678e','a585b0','3dd577','48b12b','ab0e2b','a5d333','a88559','a8bebe','add817','a2867c','aaa6d6','aa8f8d','a299bb','7c4f7a','78163f','a1c023','a9d56b','a1628f','acc86f','a2ed5a','45e2e2','a2a564','a4680d','ac6bfc','a283fa','ad5a4f','a452f2','aab5ab','a65392','acd3fd','c06679','346299','3965b1','a1bee5','a4e78c','a2556c','acbb52','400f89','3d0727','a68bb7','a46705','abddff','a97626','a306cf','a118d2','a4598c','a826c3','a1e27e','c078c0','a06449','ab8112','a6afd7','a536f7','a2471d','7c7a60','45c10b','aab766','a68653','a082fb','7c7b68','a3f188','a1d01e','39a00c','a62f15','a0a36d','ad1cef','a21e12','7c6dc9','c07d11','a23f28','a081ca','3cdb24','aab77f','a725db','acb953','a666c2','a19668','a8077e','c0512f','a522cb','ab509c','404263','a2e738','4b188b','ab7e09','aa50a1','abf505','a1a62c','a218fc','ace56b','a0e43c','a53fa0','ac959d','a7de9d','ad2d61','abd6ec','aca9af','aba281','a44f6e','a0a540','ac7ad1','3ff2fa','a37203','7809e0','a468ea','780af2','a88d6f','407734','ae080d','a14c35','ad1e6b','a84494','a57980','a15a4f','acab06','a69ca2','a58795','a7de72','ad59da','407313','ae4ed4','a0971b','a3f7da','a2564c','a2ea00','a18d08','a3023d','7c0656','a5c451','a45fb3','3ff5e4','adeae2','a1dc8f','a257d4','a6ca35','abd3d4','ac965d','acb776','a997ab','04c1cc','ac9a7a','a46d4b','a331a3','345595','a1def1','a1cac9','c03330','a4ca36','aba3ec','7380c4','a6af72','a118b2','a47ca4','a4ed65','a0907b','ab3361','ab11ef','ab1203','4b256d','e48ba7','a93b01','a3b38d','acf857','acb7b0','adf559','a64997','48d034','a3995b','a2c478','a7bf16','4596cf','7c74c0','a64310','a63e0b','3ff376','ae4e9c','a0652b','ac9215','4068e9','ae29cf','a81b86','4b4066','a38e26','a8d2c7','a04d08','a1bd4e','a3e813','a96466','48c581','acabc1','a14991','7cfa33','ac9100','aad72c','a4e181','c079f4','aba5e4','a9a524','76cc62','a2b734','a4a433','ac6c2e','ad7d5d','a260ae','a00920','a4edac','424311','a4ec28','a4adbd','a32d45','a0faec','aaade6','ad334e','a753a3','30080d','39cd0b','7c7a83','a0238e','8990de','a0ec5b','31de3a','a88ba3','a1d233','ad3e85','406e85','aaa100','a4989c','a4ffb1','ab51e1','3e651d','a603e1','a60bcb','acb650','ade7be','a1f622','aa264e','aa7c1b','a4acae','39a96a','a3354b','ab3d96','a30cbe','a9bd22','a1a68d','abe578','a0f2a9','4249fd','aa0dd9','4b2bc0','a26093','ac7737','a3fd0c','a01680','400551','a52826','a6e92e','ae4ee5','a91141','a669a8','a57a9d','a1e103','a66173','a04765','a8e069','48c074','341114','a675c0','a3ad36','c80f41','ac3967','a0816e','4b4dcb','a18679','a906fb','3ff67e','acd03f','3d7918','a517cb','3944dc','7cc466','896c05','ad829a','ac293c','45e208','39e9dc','c065f5','a3a479','7504a3','ae20b8','a9d712','a844c7','a5574d','a00a66','a965d6','40688e','a90a6e','a5f9df','a4e399','aba656','a9729e','a46e39','adf3ef','a3e32a','a30d0d','a2fe10','aab72c','a720be','3c4b30','a490d8','a6ca2f','a80a5d','a56147','780d68','adf418','3432d7','881228','780d1b','abe101','c06f13','c815a9','a1a8b6','a05b9d','3a32dc','a707f0','a4468c','a9009f','4404ec','ad0979','7816a2','c8280b','3efac0','a59f03','a8931f','ae1395','a32222','a0769a','780852','a35e2f','a168f8','a3ed40','440772','a4cf33','acc8d1','a233b9','abb6bc','a23847','781124','5113fb','4075a8','a0a622','a88dd4','a661ce','a42167','a19fa5','780b0c','a684e6','a877d5','7c7599','7c00c5','4b1855','a4fb2f','a07bd5','acaa61','a69bac','a3c586','7c3539','ae4ea5','aae924','a9534b','ab148e','78025e','49d451','a8d62c','781160','a6a8a1','a79122','ae114e','aae3e3','a58a17','a0d5d6','4b1aa5','a7cd54','abdad5','001453','add4a3','a58a67','4d21b6','a43fdd','ad8ecf','ab8ac8','a3e470','a3e52d','ae1535','3ff7a4','a29c83','ad7263','a8c5d9','c0061a','a15f14','a872ec','a02bda','86d59c','aa56d5','5000ed','aaabf6','aa8c49','a9a89a','a90c43','a9aff9','a472ac','abaed8','a95b0d','a1afa6','a3a6a3','a1d746','a3e35e','a9d766','a77b27','a08cbd','a04157','aae323','a8d2a8','7c7a68','a98fa8','c82741','a1453b','a6939d','ac8618','a8f610','a33c81','ae040e','3ff809','c065eb','a65de7','a0ddd7','ab8caa','a1c9fd','aba163','a207db','a4634b','a11cf8','a69e9a','aa914f','a4e501','ae06dc','5001ac','a39564','acd37e','a4b530','3965b0','a16cb2','300689','a6c2f9','a85af0','a801d3','ae4ed1','a7d595','a07ffc','478129','a1013a','a45f83','a3fc5c','a38071','a72e95','a4ca65','40789e','ab5607','a0fcd6','a36d2b','c07d54','3ff2f0','a41132','a6de1f','a626d6','a6b140','a63a44','a1c1fe','781690','a7743f','a81450','a2a40b','7c2da9','ad0f7f','a9ad5b','a1fddd','a704c2','acf795','acbd1f','add61d','a21a2d','ae6327','a57ca4','a08a8f','3d3615','a8d994','ac096b','abb459','a716cb','a89ba6','a120ff','4023b1','a0ee34','ab6198','aa8fc6','a2255a','a4b470','a76651','acdd2a','a0be05','500037','a5bcb6','a05aac','485788','780c73','a40b80','a28461','aafc2d','4d20e2','780418','c022e9','a7f69c','3ff33d','a10fe3','a89f81','a00d5f','a40a91','a04c70','7c56d1','a2af04','a3b943','a4db52','acb88c','a70a87','76cee8','7c37c3','4405c0','3e5f1b','a9f7b0','a1a8aa','acd50a','a2df8f','a0cddd','ab156e','a297a9','a6685c','ad013b','ac5302','ae116f','45b035','a11337','a6745e','a5c8d4','a8307d','a37fa0','aa04ae','e8021a','a1df54','4bb561','adcc6c','a25ad6','a1fcfb','a7f75f','7c4480','a66e8e','a1193f','a9d8cd','ab790d','4b1c2b','a9e298','aa99fd','a4e68c','a8983b','abdb2f','7803f3','ab8b34','ae0413','a9f461','a51d46','a396db','a0cb35','a2c147','a02655','ad49e2','a5990c','4bccb0','ae54e8','ad1c10','acadfa','a64b8b','ab2fa9','a79545','a89b2d','2e1719','a6667c','a4d504','adb97b','4952a4','aae168','7c7a58','abdac6','a0e33f','a7df4b','ab06f9','a7026c','44064b','780ccd','ae5c9f','a90388','a76a1b','800b9e','40413c','abdbe2','7c531c','a8557a','a657ab','4b51e4','a03397','a32ab7','a0ba69','aa5a3f','44a015','780cb6','ac0e65','a8b62e','adfdb6','3007d3','a62265','ace7ee','a54d6c','a4f381','780bf0','a7f400','a4b171','4ae284','a58379','a4d5bc','31ff14','c80400','a515ce','4b066d','506e1a','a34499','a80c03','ad1c57','aaa60d','add618','3ff012','ab09d0','a20723','649dd7','acf6ec','7c4d13','a8022a','3ff82d','a0a9fe','a03037','a96435','a958aa','a73696','a8d916','ad3580','a17971','a6942d','adf52e','a1c9ea','a00c0e','a3e8ee','a23867','a08dde','ae4e90','a95033','ae4c15','aa9a0a','34208d','a1288b','a3f473','4b50f1','a51102','a1b3b0','a1db35','a63689','4404e3','abe786','ab1b33','780eb7','3ff44a','a6b0e1','8964e0','a96ffe','a81530','add9db','a50bc1','e48d15','a47bc6','a34fe1','ab5dcb','aca5a4','a5ec3c','a41b1c','7c66d9','a05926','a51326','a63f5e','7808d8','a51691','ae1e63','3ff66f','a34ecf','4b51c7','a6889b','a66854','a813e7','a50490','a164cb','a8409f','3999e4','a224df','a746c8','4402e6','a8db44','a756ea','a52e81','867534','a80bf0','a0fea6','a66b3a','800d41','a71030','a68750','a944a1','abb6df','a6f577','a3bdbe','a5db67','a76f0b','a4f38d','a9a4a0','e49127','a74bf6','a474f1','8686ee','3ff3f1','aa5bfd','3c70aa','a5d626','a9d91a','a9d4d7','a99cae','8403e8','405b46','a0cb60','ac2666','adea36','a3e6e4','a7d5a5','a6c99c','440872','ad2294','a05811','a5b899','c036d0','a29ea9','88517a','a39603','a07f4d','a9a430','a4c3e4','008756','a9d4a0','ac3f05','3ff24a','a17b66','a18ab0','a9dab9','a7280d','a7538f','a5cb4b','a9eab5','44a010','a5c7c5','a66e88','a4f0a4','4682a6','a1efb6','adf63a','a84130','a50600','85d42c','a7e173','a6ec95','c008fc','addf5c','a3a816','38ab7d','a1d3c3','a0be45','a26f5f','780e2a','a8ba65','abc649','a50029','a1706f','7c7ac6','ad2775','a02ee5','a5a36b','a69bcf','3d0f01','ad1262','0090c4','a72611','a18d98','a1186d','a0c830','ad47ed','49c194','aa3cbc','78110b','406b1b','ac4b9a','aa718c','a3ea68','3d3d8e','a91c89','a143eb','3ef4e5','a96a9e','a149b9','a037da','4b4080','a7103c','a0934b','a7c750','4609db','46815a','ac8587','a011be','a9d714','06a2e0','ae0687','a8b1d1','a4cea7','a3e2c1','c06d47','aae8ba','7c75e0','a5fa1c','505ea2','a7d30a','a205c9','a9cff2','78122f','a83d1c','a5dc48','add7ec','a52dd5','800c63','780c61','adff00','a35c21','7c1b37','a7c679','a7f9e3','a0fd3e','43eae6','7bd00a','ab91bd','aae6e2','a3947a','a806d8','a036d0','7c75a5','a7136d','aa0daf','424b34','a6380b','a62b90','ac3b6b','a2ad23','a0d870','ace9c4','ac98fe','a7e52f','a6d65e','adadfb','acc2e2','a5b0ad','a2b6fd','346413','a4d373','aca98e','aa0d83','a39719','a282c0','89911e','a05855','ace79f','3ff4bd','780d90','a3b268','aab698','3d1377','a53f19','a03eb8','ace5fe','399188','ae11b6','a1d188','a34a61','a9ab79','ad2f2d','a19211','a9d40e','a023fd','acc748','ade617','adfe94','a8aed1','7c6d0b','add384','ac0dc4','a8a5ab','7c7f0d','a51381','a3e607','a85f09','a7fa2f','78920b','71c001','a1b775','7c7fb2','a81214','4aaa69','ab7545','a0fa73','0c4153','a3bda6','3ec398','780895','a096e5','ad5ca8','7c7281','a3391d','0b6008','a21612','a4fd94','a22432','ae4e91','ab4ea1','a0234f','a0df58','a80a8f','adadfd','a43704','a324c3','abcf30','a2ce47','4ca725','0a4005','ae6210','a81b74','7812be','49d453','c01ee6','ad2763','a085fe','0d09b2','c81c07','a2af61','7814c3','a1e845','a1b7b7','ab5fc9','a98224','a49dc6','a9d7f5','a2f738','7c2936','3ff161','a8d94e','a2018b','ab0ff0','aaf640','ae173c','7c46e0','40310c','a32098','acb376','a559af','a02133','ac4441','a492a4','a96561','a81372','a18aea','a011ac','3ff7d5','a46d26','a700da','4b10e2','484f81','a86e3f','a0dde1','a4e7f9','a4d83b','ab6dbb','a09c04','a1d998','ac42fa','3ff7e4','c01754','a7675b','a23ab6','38f9db','aa68a1','a38a10','a20d9c','a967fe','a71299','3d10cf','48b122','a02b94','468158','398408','406b0a','a08de3','aa5b30','ac91af','a098e4','a9c701','a1336e','463a2c','a2ed48','ae1ff5','3fecd1','4842c1','a6e3ee','a791c0','a0a7de','a8990f','a32eb0','484f12','ac6baf','a861d0','a267a1','aaed5e','a580b6','acef85','a00fbd','391ea9','485b45','c07535','a932a6','a47662','a3b00e','a263f8','a5f581','a033ff','a6de37','ab8917','c026e7','a8752f','780626','ac3b8e','7c7fba','a73540','a307fd','a05856','a27b96','aba956','780ae5','7808aa','ad59df','a931d2','ad4584','a1c627','a9c8b1','a8a72f','ac6b67','a094b8','a59c4a','71c360','a033bc','ae07d8','a30a7f','a12ad5','a80136','a95641','320027','a71435','4d2265','4248f6','a90208','a334de','a4d314','44076d','42459e','00e945','7c6bac','ab4f43','a7f316','780eee','a53b06','7805dc','39ac45','a6c456','a0b6f5','3ff47a','a6c0da','8a057e','a577a3','a248e6','a83a1e','e48e3f','aa5bbc','add7fd','a11a53','a52c4b','a63d96','ae4a79','aafd02','a22e22','acae9f','a30cdc','a2c2db','a4e7fb','78119c','424362','7c6b1b','a0b0cb','aab2ce','a89f8b','a6d7d3','781579','a4c74e','a4a8f5','a40b1d','a4c246','a8a859','a9fb29','a10167','448163','a2b2d2','48434f','a2cf2a','acc9ab','a2d4d6','c00d9d','a71689','a2f3b4','a77b4f','c06d5b','896118','ad4c05','a81590','a0acf4','424b3a','7805a9','78035c','a4ae22','345646','3d687b','4ba959','780097','a24dc8','a01e36','39c8ce','ae5773','a8f75b','a166bf','a385ca','a2c1d3','a6bb63','aa83dc','adf592','ad4d5e','7c338f','abf32b','ab4ddf','c81bce','aaf446','e493e0','a53fc3','ae21c1','a4af7e','a0104d','505e95','a64929','a0b9aa','acf121','ac77bd','a6b7a0','aaf29b','ab22fd','add10d','a4f1da','ac950b','a58101','aaf6ed','a5613e','a16757','a1545e','43e914','a29931','a69811','a6c1b6','43c5e1','aad75d','a4c6f3','ad1065','ac9e9a','a2941c','ae012e','a79a68','a498e0','7c11e2','78140b','7c2b16','a3c140','a390d7','484edd','a37183','c0103e','a3d03d','710354','a7fb29','aa58e8','a85a5f','a6bb0f','3952eb','abd50f','a01ed8','a68241','e494ff','a005ed','3d1019','3c433a','ac570d','aa4a99','780ad1','ac55c1','a41e0e','7c67b8','a0f230','c82477','a4707d','a9db95','a4e485','a7da0f','a0f010','7811f1','a317ef','3fef5a','3e7268','a01c9e','a4e6ce','4d212c','ad72f2','c00bf7','a99e76','3c4351','aa631a','781570','3d21c5','acb5de','a23409','3d0edc','300640','aaf46f','3dd797','468151','a5b682','a4811c','a368b2','aaa2a6','a693f2','47a70c','78110a','501dc8','adfe54','a149b8','aa1188','abe66e','a1ee31','7c0daa','4ca332','a9926b','a1d5ea','a86df1','a70538','acd3b3','a170ee','c051e4','a0b514','a023a5','a830d5','a32f77','a89ff5','8004a7','a56285','a1d49e','a635e7','88808f','4b4db1','7804f4','a21656','781193','abee62','a53cb9','ae1e5c','a7c44d','a88f72','a14ff4','ac9d03','4ba946','a054fa','a035ba','5001a8','3921db','acd53d','a438a3','a3c459','461f0d','344305','780f3a','a31ccf','a54d03','a62b39','7c80f6','7c79ee','addc1a','448d81','aaac1a','a8090d','8990d9','4850ad','a52b00','abf546','a814e1','a206a4','a9e526','a594d3','a2a2e2','a12566','c038a7','4243f2','a3e687','4019b4','a1f86c','3d1d84','a6a0cc','a4f835','a89317','c08195','a4c3c3','abc6e0','a8a3fe','7cf9f6','a1ec1a','a326d8','a54eb1','ad1506','aaa31f','acfc3f','7c7af7','a5cbbe','a9d706','aa187d','49d1c5','acd729','aa8c99','38b01b','341119','a614ff','7c7cc5','ad41eb','abddc4','a585c3','c800ef','7c7b61','3fed6a','a00fed','488915','a95535','a29b61','400dc2','aa6b6b','a701ff','a44aa8','ac0882','7585d2','a291fd','3d2f1d','a6fc4d','c0323e','adffe5','a43ca8','a01e07','7324b0','a1ad52','738c06','a31ccb','a7d624','ab5afe','800bd8','a97046','a19abb','49328c','a651a9','a67efc','a47706','7cf9ce','4ca2a7','ae5d8b','398c2b','7811e4','4ca242','acc8df','a99ae4','ad1096','a548f6','a05f47','4d21fe','addedf','a725c4','a902c7','a1bcfd','a384d8','a1c458','a83dc0','a8da83','a7f53c','adddd2','a04a0a','abe4a1','a2cef7','aa3c29','a43bcc','a7e894','a51e31','a19253','a2bec7','a0bcd0','a9e670','e485e0','ace72a','aa31cb','a472c7','a0f345','4bb146','ab06aa','a0f2ac','a80c9a','4d023c','a570ac','a08577','a69530','a62698','a1b165','a3c111','a82d58','a66bc3','a008c1','abc60a','841115','ae2684','ac73e7','a14a77','a58153','a7e8fa','508035','ac1b9f','7c7632','ac97fa','a0208b','a14f46','ace0cc','a71525','a89002','a1ed21','7804a9','a9fa2a','4241ea','ad447e','e48982','a63dc7','ab7dc7','ab7334','0d0511','a74a4e','a14561','a3d4c6','a5bb61','a6b4b7','abc0d9','a8bbdb','a1c8ed','a51cb3','a32150','aba90d','a3e5e3','a6ef36','a1f4c7','a880ab','a7ded2','a670a8','a4ad57','a1dddf','ab1b50','ad8e29','ac06d0','a07cb3','a8db20','a727a5','a339aa','a6acdd','ad6bcd','a1a46f','a31b81','3de688','a2865f','c01eff','ab532c','a38f84','47a45d','c023bd','406ff0','ac25eb','a0a683','4780a4','a60e0c','a41f54','7c4cb6','780561','a2facf','ab0824','a316a8','accc58','a83054','a3bf1e','496c2c','aa01a2','e491a4','a6c448','a1f7b2','a703e0','3d0b38','738284','aa4a1b','a63f20','a95037','a1fc3b','4b5d9a','7802e6','a10611','758620','a7c2c2','481046','aae958','a959e3','a3530b','390024','a061ba','aa9010','ace747','343359','ae5b68','a570ba','ac315b','a95364','0a0081','ae0210','a4b0f8','a1d93b','800728','7c129a','a16918','3cee56','a7c67b','a3f481','a9755a','ab453e','acb497','89617b','a83b4c','a1902a','4d0112','a99a58','780fa3','ae144a','a70174','acb2a7','a703f5','a56024','ab6aff','a38ac4','3ff03d','a66726','43eaba','3d2341','a8df93','a18f22','a8b105','a00477','a828d8','a7a416','a0f679','a2e098','a7f37a','a6b52b','48b2c8','4b4247','800d0a','acd9a2','a948ba','384575','a16154','3ff656','7817b5','a86657','484f29','abe170','3e0174','4ca9d2','acaf9c','4a9b59','78920a','a9eecc','a01532','a5ba10','a99684','506f21','ac73f9','78103e','aa7cc2','a1fee3','39db1c','a37553','a5ebb2','aa1946','3a2ffc','04c117','aa5202','a2dce3','c04785','a59174','8007c9','a027f9','3d1047','ad43ad','a345e5','a72359','a0089e','758414','3c6572','a96823','a9b402','7c168e','ac5222','a0ae0a','a705f0','a45f94','ae54fa','a6135b','ad978e','ab7b99','3feed5','a7cbc9','ad0bcc','a8402d','a639c0','a14871','ae11ac','a13a2b','a8313a','ac79a8','a06119','aaa1ec','a4f8fc','a3e52e','7c759b','a661d8','780b22','a9ee34','a4d41e','a675ef','a9f771','a43090','e80977','a1f5f3','a8e1cb','aa1e30','a875a9','a19cbc','a7462d','a606a2','a03cb9','469c25','a8889e','a5cf0d','3007c5','4b1bb4','a4d86a','a6ded1','abc591','8465e0','ab045f','a68d7f','a7bc58','ab6630','ab00c2','a6de3c','a9b20d','7c5954','a527f2','33fe7a','7c7ab2','ae4ea9','e8044f','3e228f','a2626f','a1a1fc','a0c773','a728ca','ab72b7','a21113','4609e1','abf421','a31dbc','7c6699','a7066e','a59529','abfa69','7c530d','78120a','a5c9b5','ae171d','a13b68','a877e7','a18ee3','40740c','43be8e','a0007f','48ca69','a3818a','ad58fd','4594ba','a8a94f','780e34','a80c78','a32453','a44f1c','a964e3','7c4a08','a95f5c','a032e4','ab02c9','aa912d','a28de3','ac6e92','a8da62','a468e0','a60202','a3b790','8964ef','aaf9c5','a9da0d','a3470f','c81e82','a6754f','47c41f','c06751','a839ac','a3d3d8','7c1a29','a4cd4f','a2767d','aa32e4','a25dd0','7c7539','00a8c2','a46b03','a3ede5','a18bf4','aba807','a2b966','a12a64','a7d669','a8afc7','a13e22','ac34a1','e48a10','a8d9ce','a6b6ca','a3d024','aaf523','a6c4d7','abbe4a','a7acb7','a4efa4','4b4bb5','c03f02','498b26','a6dd33','a4e3ad','407017','a60b90','a4c7e5','a3f763','3e3d01','a6c75e','a1a698','a0aefd','a082df','484ee7','899000','06a0b0','3d08d7','a51a6a','3cc9ce','c05856','ad39f1','ac3d94','adbd5b','a90295','a0dcd0','a63d2f','a04238','ab4527','a5deee','a83756','a692fe','a4433a','a3f8b8','a2e1ab','a7cda7','a0ba09','a95054','0ac0f5','a39fbc','a6bec6','a1c75a','acb77f','a03325','ab6a1b','a81316','a8282d','a55333','acd27a','a5daed','a5198f','a12295','7c0a5d','a5ba77','a01f97','a1d01f','5110b2','4ab2a7','a96f81','a042c9','c04fc9','ac79ea','484b40','e94b6e','a08670','a4b160','a674dc','a1fd0e','a68d44','86e4b8','a9153c','ab76b2','aa1f5e','4842a4','aa6787','ab6e05','ac701b','a9dbe2','a28f93','ab3ef8','a03415','a38c1e','aca241','461838','a144bf','a8ba85','a9beab','a6548d','a55e48','ad4304','3ff83c','acffb7','a68a37','ad2508','a5dbcd','a4b817','acb051','a9ba00','4b2d23','ab356b','a5fac9','a0db82','a2676f','3ff7a1','440918','a55a8f','abdec8','a1b277','ac76cd','a17f45','040103','344056','a480a1','a981c4','a4a7dc','a0c6b1','4c4a21','7c821d','a2fca6','a82559','a0e869','ab471d','a2afa9','4caa43','a2c088','a89e2f','a728e3','a1cb31','490d02','a3609f','a5407a','ae4e9b','a5599b','a1a27d','3d18e3','a0efc3','a3ecf8','a0e191','a02fe9','a80d68','a356fa','a1ffb4','a07c0d','a3973e','3edbf7','3e376f','aa2db1','aa1c48','7c6c83','aaaa38','a3ceb4','a9ae43','a061c7','a33f8b','ad3435','4ca257','abc453','392452','a8dae2','aa3775','a6afaf','a551e0','a3477b','a16c1c','a4dfe2','7c2e2e','a9afe2','a61a2b','ac8ed9','3462cc','a1acff','ad5418','3ff229','ad9d79','a7a644','ac9793','45a1f7','a12f9a','467891','ac3408','47a03c','a1f942','a62bce','7c7a39','ac8963','ae1179','a9ee48','a1d3e7','84c3dd','485ec1','a6e879','781173','780c72','aae0ba','a7cde1','a628c5','a24796','aa1865','a84f8b','341545','a1deac','c00f97','ae04c1','a325b4','a2e08c','4b3944','4b22a2','a9b7c4','a53f84','a3323a','adfed9','490d03','a9e6e5','6830bb','4071ce','a090cd','a2620a','4b40c5','3d5899','a86927','7c4c02','7c58e4','a5127c','a944c3','a837a1','a9dbb4','a9022b','a66e8b','ae1177','a2207b','710259','a88856','780adf','3ff092','7c74c2','ace60e','a8c855','a8140e','a81c83','a5bb10','a4be81','a1d8e1','a81143','3ff398','ace7ec','a1c09d','3d489a','a9e935','a8859d','a72842','ad8a80','ab037e','a36b51','a0dea6','407014','a12747','3ff73f','aa51d3','c05fbf','aacdfb','a65488','a75fd4','a40c82','3d1695','aa9808','a063d7','a36761','a0878e','ad4ef4','ab0bda','ad60d8','aa6fd5','a37eca','aafb27','3ff10a','0100e6','a7caf9','ae08e3','a582f9','a39d67','c04920','a8042f','a22238','a30c46','447c24','440686','a0ddda','a7250e','c08199','a4987d','71005c','ac6cb2','43e9e3','a02b1d','70620e','abeca1','a045cb','7c6deb','511050','7807a5','a9e5bf','aaa3ef','abf79f','acedad','424401','76cd0c','a21c16','a1924f','a0f880','a35966','a5c124','e49612','a871aa','4ca4e7','a828d5','ace32e','3d1665','c01ed0','adbcbd','aa1f10','407933','aa19aa','a555ad','adb627','a90e6e','a8024a','a3888b','ac9eba','a1c94d','adfc61','ad0758','a3e4e5','a66fc3','8013c3','a096c8','ae0883','a031cf','aa2d32','a37352','a8b161','a21852','ae0063','a4d8c9','a27547','a1ab69','aae18a','ab29cb','a63a42','a4c98e','a25115','a93e0a','43be87','a6b174','a11a4b','a885ad','a74371','aa463e','a1cc1b','7bb130','ab0095','a50a66','a2ce17','a83f2b','adc0fb','a664aa','a4f39b','a06e11','4b5c6e','abd30f','a4f9c3','a48475','a5e719','a54fa5','a5a37d','a705ee','780bf2','3ddc4d','780936','a3daf7','501f5f','ad1926','40655e','a7493a','a13060','acd675','aa9992','a96b34','ab1953','a57acf','a1c0fd','780c8f','a021fe','a5d203','71c044','c080eb','a4bbde','39a5e5','398477','467892','a02b36','406ee7','4b4d95','aaa178','a20c94','c004fb','47c290','a3bd5f','aa92e8','a5aeca','a7610c','a1c183','ab37a5','aa220a','a61443','3d46d0','a16aa0','a0947e','a97a69','aca122','48c485','71021e','a27c74','a3a27b','4614ad','3ff3f6','ad650c','a5753e','7815e8','000007','a4d8dd','acc517','a7426b','adcc34','a53e35','a69b8f','a51e32','a52983','a73568','ae13cd','505d0b','a0b09a','a6a1af','a0f73c','896112','ac6692','a8de43','894f51','501f5c','ace5bc','a85e2b','a6db83','a9b52a','acc939','a38885','7c32aa','300827','adb680','a1e97d','4b2788','4040c6','a4bc49','894042','040047','a9e8cf','a8ea52','a10555','a482ce','e48832','a24ade','a5809a','a255ee','70c0a5','a4f277','a4a43a','ad0aec','40740d','ab2bbc','50801c','abe9ee','a6f4db','ae1142','a0665b','a6ad37','a522b3','a16943','ac49b3','acb260','aba64d','141ebe','ace230','a93019','896318','78022b','a32ba3','a87429','a33ce8','3ff11b','a41117','aa1219','a833b5','a28b3f','a6fbb5','a19b52','a9f487','a2c5e0','a7687d','780ff4','ab6869','c03c0c','780072','ac9fbd','a6167e','a47124','a9eb15','a724af','a3c3ba','ad8037','aaba9e','a178b4','aca6cc','aa4509','7c1abd','394f31','aab311','adf36f','ace5d7','a03660','a24bb2','a0a6ce','3e6cf1','ad5619','ae1eb3','c828bb','a4b8f6','c0011b','ab06a1','4b848f','a4f2c5','a63f02','a05494','a2b4ff','a6fadd','80139f','aaf724','ac3cad','c0712e','acad53','a31c42','aa70c6','a293e8','a1dffb','adfd6f','aaa307','7c19ce','a8d9b1','4ca8e4','a7e553','a062b2','add08a','a36f9d','a5e90d','7c2fd3','ab448d','a59c75','a1d3bf','a26fbe','ac432b','40665b','4d2065','aa527a','3e0a46','a20cfd','a657b4','aa1aa1','a72598','a1a787','a14fbe','a89c1f','440670','780737','e47fe7','ae08b2','ab286e','a054fc','a16379','ac2672','31de3c','ad340b','c073cd','7c777c','a1c003','7c7cc6','a28c9b','acf7f8','a9bc93','a36ca1','abf535','7bb12f','aa13e4','a85dd1','7c0074','a82ec6','7c28cc','4b0eb9','ab02b8','a6755b','401091','a2cafb','a8e6da','a20f84','a8ea12','4405c8','4b05e2','a1bcf5','485197','a1dc88','a8b70d','a02f84','407812','a166f2','a10781','a99923','4b4126','ab7ef9','a1f72d','a67589','a14b94','a9f14b','a89886','440085','aa064d','a48226','39723c','a2005a','a54d1e','a0c47f','400664','a36480','a64425','a9836a','a86c12','a32613','aa7b4e','aa82f4','4b2c87','3434ca','a96fd8','44068a','a84414','3c64f8','a53d14','485e5d','a1c8fb','a9da28','a9ba69','a6e48b','ab90a9','ab0498','a4bd3f','c0806f','adba5f','a842e2','ac1080','c81d17','a0958d','a85511','c05a04','a98156','a77757','7c3a1b','a9dd71','45e930','a69c85','a2827d','e48df6','484736','a4e62a','a3c10a','a83044','a69072','a02350','a63954','39e89c','3d0796','a75746','a6353e','a13f3a','c05ce2','ad81b9','acac37','a66558','a54aee','780b0d','a1b1af','acbbbd','aa1611','a1e993','3e2799','a43f83','ac9b88','7c3e0c','a0a252','ab0453','a5c247','aca62e','a71160','48d116','a6b222','a33a9d','a34de0','a5b2a5','a55e76','a3861d','a18f51','a041ea','a5035e','a1c101','39ac15','a4fc2a','7c4e5d','a49a3d','3d421f','a7a28d','502d20','a6be75','aa01a6','a0148e','a0d32e','a5e76f','a54823','acb35c','a953c7','a743ca','ae5395','a2ccde','a83198','a85ed1','a828c9','a9a0f9','a6c26f','a1dfb0','a46ef4','3ff594','a297ae','a7da05','a014b5','ac9b6d','a67595','780a29','c059cb','a351f7','a5281d','ae01e8','06a0b2','a63f16','a1f900','75025e','adeb90','a89505','a14084','a266c8','a08456','a247c6','a53c8b','a2c440','aa97b9','a8f84d','a28b52','a9b34a','ab0fe4','aa71bd','acd208','a0d671','a982c4','acd119','a9dec2','aa67fa','ad1825','a84a77','4d2212','ab1fc2','ad8154','a911ff','ae51e0','a32f85','a2cf9e','aa2cbd','ad20b5','ab31e3','a0a55f','c08207','c07f37','a69a50','a0cf7a','ac6ab9','a6ae83','a013dc','a01a1e','a7ca94','0c4157','ae1153','4a9f25','c06151','a4bfb2','e492fe','a23b84','a58dc8','3d5fc2','ae10df','a9d59f','ab1218','4407be','ac3d4c','a5489b','a065b0','ae4ddd','781181','a1da1a','ac5f88','4076e0','a4bb80','80064c','a3cadf','a063b1','a2f41e','a1c5f2','76cef7','3d63c0','a2b041','3ccfd8','a46ae6','ad081a','a6c80f','a5a5b6','a24dfa','a00bc3','4ac9b2','a1952b','a2c7c5','adba82','a7d973','ac9b54','a50fb2','ac290c','3d0fb3','c1e547','ad5a1c','a96e19','a6b42a','a2cb2a','a6c2cd','a4494d','a95355','a539b8','a2724b','a65e94','a560f3','a216e3','a28022','a3a63c','ac3476','a6633f','a9122c','393269','aaffd5','ac6e5e','780fa2','a82c1b','a1f50c','7c2a34','a61af2','a04752','7c2bb1','a2f942','4cc51c','7c0841','ab1b30','a99bba','acd7a5','a33ea8','abb40e','a6e8be','a6e3b8','402bb7','4b4dcf','401b3b','888888','a60ebd','a52aa2','440967','ae1c2a','a55cef','a4d837','a92904','a530e2','7805f1','aa41c4','7c384f','a9f09a','a0fd7b','a1c7bf','a91209','a4cf7f','a04f27','aa5fc2','a1610d','a32e7e','ad48da','4064a7','a0bb7a','a45a39','7c197a','a81f5b','a2a688','3c4d8f','ad8b18','acadc3','a2b2e1','7cfa71','43be83','a0e34f','a03c7e','a00493','a37833','a23ce3','a22209','a7d678','a12ab8','ad7a88','a42a15','a166b8','ac3431','c82368','ac99d5','c03352','ac3cba','a10973','39dce8','a34bd5','addc92','a6c327','a1af10','7380c8','896c08','3d0a7e','a96e57','a38875','a8d984','a74d03','344491','484205','c82170','ae0dda','a3be6b','a3c98d','4242c2','a8a2da','a20813','abdf4e','76cdaf','a31c15','a4f107','a3296b','461f50','ad98d9','3dd37b','a3a79e','a6eadb','a5e2d5','43e9ce','adacc6','a3d742','a78a59','ac6c45','7c7b5b','abacb9','abd559','a88e3f','a6338c','a43ff0','a18194','aab187','a34b29','a46b56','a24563','a21104','a1c1fa','ae03ee','a9374a','a19167','a027d5','a2c183','adf2b8','aa96cc','3ec56d','ad2cd8','7c671d','a1098a','a4bf3d','7c49e8','4400a1','888054','ab30f0','a5fdd5','a04e73','a286cf','adff57','a16ff5','a506e1','a30739','a8b476','aa37f6','aa998e','aa31c2','a7ca6d','a80fb4','c06336','abc4be','a4d5f0','a7f590','a55312','a9707c','a55323','710419','a531e1','a6f6aa','aa7e99','a1ede3','a4fb34','a6be14','a77379','48460d','a64531','a37892','a126de','acc498','a68e37','a9e5ce','aba941','aaf4d0','a133c5','a63595','89900d','a171a2','ab4c77','a2cb62','a7fc44','a63660','3d668c','adf016','7c3fdc','a14a51','a0c6e5','c07a51','adb9d2','aa196d','ae1897','a8cce3','a590f3','abea2b','a44e5f','a03f9e','a1c857','780dc0','a23114','ae113b','a64ad4','7c79b7','ae5c91','abd1e3','a70087','a5525c','a08e17','aa8767','a269c6','ab722b','a1034e','a65088','a82795','a773a7','a046b6','a65e0e','868340','a771a8','a3cdbb','38dd7c','4078ad','a4d079','76ceee','a7d6d6','a5ac9e','c05f8f','a6b458','a57e5c','89636d','a4edb6','c07b39','a6fe1f','a6b185','c076e5','4076f1','ad9302','a87bb9','adbe74','ab7fe3','a50acc','ae0567','aae1f9','aaa43e','a9a541','a09b61','aa8486','a3fd74','aa33d9','a681cf','4ba28d','a06d48','a86ba5','ad3734','ab4476','a4c02d','ae1400','a65308','a1b2e3','a04f5f','a34b17','a2281f','e2001c','7c1602','acf8be','4b39bc','7806d9','a97530','abaed1','ae138d','a8b98e','a7287d','a54595','ac9ab0','ad6814','ac4d44','c0701b','a4841f','a431ab','a2e03a','a68ea7','a42329','ab1b79','a02209','adfed6','a05de3','3d1c35','4076e5','acc3a0','a36c20','abc9a1','7cf9fc','152c2d','781714','a8a1d3','c08515','a6e25c','adcc46','a10006','a4f488','a164ad','8990f0','a32146','ad5dc6','4d2094','a3ac27','06a1a1','aa3b4f','aa66f6','a186e8','a166f9','ae1450','4018c7','acd3b8','a4f366','a0f9f5','3e6f40','a55a08','a003e5','a98db9','a8167e','ac90f4','a1089f','78050f','a0b7f6','89646a','a1083b','ad6ec6','a3cdd5','c06c95','ada916','71c046','468152','45f07e','71c011','780eb4','a96b80','ae120c','a3ee1f','a7b0c9','a004c5','a42f3f','a700ea','a34b39','ae0587','a9d0a4','c00e01','a863f0','a4c2fc','a15f32','a0149a','0ac9aa','a83a75','a2eacc','a4af03','a5732a','a2d5a7','a5984f','abe924','a3dca4','a0fc43','a529f3','407155','38d9bc','a8e6e9','501da9','ad6926','ac1068','a7e2c5','aabc0a','a22ee9','a294c1','a90833','a7ab18','4484e1','a32c47','aac69d','43eaf5','7c1cdc','700002','44099d','a1483a','a42613','a31fa5','a2a32f','a030ce','a512ff','a3563a','3eee72','78069e','a0dd44','aa0c62','7c4a25','ab5884','a638b0','7c1768','a1ed45','a9ea9a','a3fe8c','781101','48c9f3','ad7ef3','ab9384','abadc0','aadc7a','a327bf','a6c618','ae0940','acbd0e','c04082','7809a6','71ba02','a69cd7','ab37f7','acc0f4','a267f8','a9fef1','4408a6','71c010','a27ce2','a69ccf','aa31b8','a89e79','a89ab9','a6dfc0','a584e5','a088ec','a96f3e','a39c4f','a80935','a06917','780c79','c0711a','a0ac89','c051e9','a6acc9','a1c811','ab3d51','a50fdc','aa3de1','ad0e40','a03a51','c05c6c','3d3992','7809d8','7c1b26','40621c','343687','8964a0','3d247b','ac38e1','3d22e3','1f1d5d','a5f55c','34604d','aa2425','ae00a4','a65f5c','ac0b46','a671ba','a0263f','ac9a78','a6d579','4076c3','ac95cb','3411da','ad7d57','a76e2d','ad7ea9','c08566','a8658c','a82de5','a4497a','71bd54','3d1798','aa2b44','a9ecf6','48a5e7','7805c9','a8028a','4050d9','a2457d','aa714e','a6242b','a71e3d','a1fe61','4aadc8','ab56b4','a7fe1c','7800cc','a45216','a3328d','a03427','ab3544','a42036','a188b9','7805f2','3ff177','a40c25','aca9bf','a7ded5','a0de59','a54d6b','a3c666','a323bd','a58b7e','acbdca','a864b2','3e65ec','ac293a','a9f3ba','501f5e','ad5522','a12b21','3cfdec','e14d34','a54ee2','aa108a','7c176e','498b29','7c7c35','a35e61','ab1cd0','ad1852','a5048d','a4e2b1','781581','ac7aba','a85a64','aa2039','a010ac','4b4e3d','ace04f','a74bb7','a66062','a4eabe','a4af3a','a66a68','a8158f','ad95c2','ac62ec','a388a9','7c12cb','04012a','86ef28','a8065d','ac2dc9','aaf228','a690c5','a3732b','aa3f66','a98074','a6d30c','a6397e','85c5dc','ab571d','06a209','75830a','3ff40a','a7d04d','ad22a1','48515d','a3a358','a52762','a1e08d','aa6e5c','a8b302','ab1484','a67066','a8d6fe','a7d740','4075d6','aafcf1','ad389e','a2a1a5','780bee','a326ca','3cde56','ac4dfb','7c5caf','a315db','a9a3c2','ac57ad','a97abc','a22bcb','aa7817','a7a324','a4098e','a087a4','a677b2','a5c0af','498b23','ae08f9','ae1e71','a7ea39','a0578d','39b96e','acbfbc','89631d','780250','3ee4ef','a61cd0','4aadda','aab7a0','a422c9','ae2ed4','a5bec0','c0468e','ab1852','4b2923','a9ede9','ab2018','4b2c5f','ab8e3a','adc4fb','a0aa69','acbec2','a01311','e80210','a880da','a1f891','a762b1','a2acbe','a58db7','a06a07','06a139','a1230b','a7f890','abd64b','aa1d0f','a2682a','3dff91','aa33d0','a6ec11','c820d1','a08edf','ab5668','ad2aed','a549e4','500129','a7d536','a8b499','a48163','846822','4b1919','a06092','a3fad4','a0aa90','7c7a6c','3e7a98','ae114d','a77630','a40415','a0301f','aa0d21','44ccda','a20cd8','a27ce7','47855c','adf7bf','a1979a','407267','a71f97','ab30dd','a837fa','3c41fa','a3e83c','a521df','a51ca5','a29e5f','a789ea','a09c47','a8e1f1','45cdd3','7812a4','a022b9','a00260','3d214c','a41eca','781283','aa7d09','a213db','a1cf3c','a1ce93','7802a6','a360fd','a2273e','a543f2','a7cdc8','a182c4','a51735','7812e5','ad8ca5','4898f1','a4acaa','ae037a','a67aa3','780b65','a1a0d1','a7d9bf','a05001','a300ed','4d2250','a97ac1','a4f081','0ac372','a11785','aa247b','a87d48','ad337e','a998b9','a3e4f7','3e65d1','78129f','a02a5f','a77d2a','ac2626','a2e77a','a9e013','406a34','a57dbe','abca0d','a2cd90','a9e60d','484977','485d9e','3d12b8','aa3e4a','a1f89c','a9da34','a89eb9','780eeb','a5ebfc','ada8ce','a72fe9','a9da74','a0658d','a0317b','3ff4bc','478b44','adb1d1','4b2c4e','adf0a3','a63aee','3007d7','78071f','a36fbf','3ff77c','aa9143','ae2ed3','ac40f1','aa9c34','484202','a88dbf','a82a6e','a5523e','8990ed','4b325b','a3b082','a9bd86','7814b9','48ae24','ab5eb7','a71e6f','a86488','c0572b','480000','a9dd07','a98b8b','a5281a','4405c1','868ec0','c06415','ae0583','a28cae','a07253','aa9986','800da5','a70879','a06905','407142','aa4a1a','479384','48a5cb','3e5ec7','a8f990','4d2200','a7f05e','a2f227','a82dc5','a95ac9','a69d78','ae2651','3d4881','a0245a','ab1fab','a98d77','a67587','aa3173','a89a9f','8963b4','a42f67','a0946c','a3b90e','ae00a7','a6c1fe','a4ccd9','aa841e','a005e5','a6c286','a33644','aa34cb','a2f292','ad1715','a9dcfe','8a05b6','3ff0e9','ae5e01','ae115a','a98840','a1abec','a00c72','497cb2','a5ddfd','a17820','a030a7','3fec1e','7804d9','ace130','43ed0f','a04da8','ad10d1','a4a54c','4d20d0','a5c237','a26db7','aa513d','ab29be','a7e5b6','a34e4f','a05200','a034fe','7c7a71','a05770','7c4dda','ab9078','a7b425','c806d1','a2f092','4b32a5','3965ae','a0d3f5','3ff7ae','ad4e68','a99c78','a6dd25','a0cd27','ae093c','43c1ff','a32e10','7808a9','a20ef9','a3978a','780e9e','a8539b','aaf1da','a9ec43','424156','ad7892','a31ea7','a17623','a7325c','a727f5','a625f9','4009cc','7c2483','a8a606','440785','ad48d2','ac45ea','a2addf','c82258','a0b5fa','a8bb5a','a23892','a0b6df','a19c52','a47c26','ad3ce5','a0894c','a0c22a','a0c99a','a7ecfd','39806f','a99824','a9598a','aa6f29','a69fe7','a1db1b','ae5f05','7c2b41','a1c932','abbb7e','a57c7c','780e98','a65fb8','a69bfa','a1312b','a2270e','0493e0','ab9ab3','a8a2b8','a3eafb','780cbd','3d47b8','5110ec','add012','ad9acc','a8f9f2','7c4d44','a1e015','a0217c','a0ccd2','ac6fb4','abda02','a72859','ac1058','a45e2f','aa4622','a16022','a17df2','a8180d','3c458e','a18e80','a8536d','a81de9','a0d27e','a4c326','a9eeb8','a7ec9c','884343','4005a9','a4142f','a3524b','780d12','ae115d','ae2652','ad6657','8990ce','42424d','758343','acae9a','ac81c9','a55568','a6a987','ac9a81','4b07c4','acecbd','4bca63','a55d23','c073e7','aae6ea','a21603','a137d5','780410','4077b5','a24d5e','a8f58c','a50cd7','7c49db','7bb0c3','a9eca5','aa32c4','a051c9','aa05b0','a2abcf','a0c34e','c01a52','a2380e','3d3c53','aad24e','7c2bad','a9d612','a52fe8','a758cf','aaf3af','3ff3fe','a9718a','a443d3','484806','a99054','a0bf4c','3935ab','a8c45e','a67470','158406','3d381b','a94c60','a09c79','a8b3c7','a7d516','aba887','a1b8dc','485069','ad82d3','a1569c','48c138','acbeab','a8c9ff','ab13d2','7c3a2b','7c6182','a6a759','a6648c','a3a2ee','a7590b','a12bc3','3d2b0f','a9cf00','a6827b','aa60e6','a9a24a','a0d9a4','a1c69f','48fe71','a51299','484cb4','780bf4','7c1cb4','4071a9','a1f3aa','adcf5f','a4949e','a3d4ae','781369','407931','a5611d','3c7062','a7f4eb','7c7b56','a68240','a48b51','ae1479','a133a4','a3a111','896c42','ac52f2','a252ae','a28729','aa2e41','ab9ed7','3d122d','a0d8fb','ac03ef','ae2ef3','a68acc','ad0b0c','a4f8f2','a07de2','8991c0','8963af','a396ef','404470','8962e6','aa2525','a8243e','adfe1b','780dda','aa7975','a74294','a3d3cd','3edb51','4243dd','aa7bcf','c05bc2','a4ebb0','a9da99','c1e437','a27ca5','a59b19','a34893','4b43d6','7c14b3','a4b6c2','a35fbd','a1c767','c005c3','ac858e','a0d1c8','780a70','a9ab8d','abc096','a8d4bf','a86904','78094c','a5fa5f','3d24e0','aaa5ae','aa75ca','a67e6e','c05097','ae093f','a57064','ab4549','3d9cb8','aadf86','a53bd0','a2f72f','781555','a40edd','a4fbdc','7c802e','abaafe','a42a97','a83bab','a1d9a9','a6d820','aa3aa4','a4f6d5','a25572','a234cb','a447f0','a87ad3','a227e9','a98ebb','a8ce8c','a1a4ea','48412f','7816b3','a1211d','a4b7ea','7c79f8','a64160','0ac3d0','aba7ff','a27f61','76cda4','3fea3b','ae11ae','4b51d8','4007f6','a65ce4','a1e2b2','a385a5','7585ef','4b2c58','38f21c','440536','a98159','a07367','ac8efa','a4ac6e','a10f6c','ad3959','a55a3a','a4bb89','adb317','adcb14','a1a7fd','a1538b','a6b477','a03c23','a06763','a02ccc','ac4749','3e4601','ab0a3e','ac9be7','79a05e','a46b73','3ff7c3','acb8d7','a819e2','a4c6b8','a6544d','a39519','407154','345682','4ca846','a84428','aad63e','a9de03','a26bb6','758629','a2a227','abdb17','7c0634','aacfbe','7c7b5a','a9d2fa','ae1195','489328','acf213','a28490','ac1c45','a39479','ae065a','aae3b4','a21ca8','acc960','8964c2','a3d754','a7da75','a724ce','a54d2b','aca3c6','a64a78','ae20c7','a5494f','a35a7a','aa2e7c','3004b9','a61064','89631c','424294','3ff268','a651b1','a3376c','7c7ad2','3ff45a','a5dd36','fef718','78086e','3c61f3','a88d77','3d3f14','a72ad9','a1e36f','accd11','aa8f5c','a72fa0','a57e5a','3d0fd2','a34b9c','ad939c','abab75','c07303','ae54e1','a3ec34','a46745','a3230f','abc65c','ab0c25','a03060','ac9c9f','acb1f3','a2bbce','a67a82','71bd78','4b3040','a74eb6','c00e8c','a51da0','a4f08d','aa6b88','a5521c','ad66aa','7c6b4d','c04599','402989','aa466c','a08627','424a71','a2e294','ab39da','a06bd3','78158e','adf346','a189d4','a87059','a848d6','440a49','a27f0d','aaad3e','a0a92e','a00de7','aa11be','a2fe7b','a13e71','888189','a22e6b','aa50e6','a5ecc4','7c07ca','ae0058','ab2eb5','a726ec','a2fe98','a549c6','a3459d','aa5bc6','3b7781','a38fc4','a34605','aaf0e2','ac1d76','aadf38','a9db24','a9da49','49d25e','3fef1b','a1a1e7','a9a310','0d099a','ae2fa3','a85d00','a72791','a19758','a2a45e','ac9042','aaf914','3ff171','a44bb3','a405ae','a719a3','a601de','aa8f44','7cfa32','a729b9','ae1155','a54d26','a3bf42','c04c84','a8d234','ac9cb9','a04685','ad8f4c','a42750','7580d5','ae00c9','a214cd','ad2e4e','ac9ff5','aadae3','a7ce10','a82908','aac9bf','a5e12b','c04414','a7fffa','a7fe51','ac9ae3','a3ada4','a8bc29','40263e','a3cc2b','a2a96f','aa812e','7c6728','ab31c1','a67a8b','ab2146','ab9dd1','a0a668','adb7dc','a6f16b','4b5169','0ae065','a527a1','78056e','a64ad2','a588f9','3c171b','ad8f21','ac3d15','7c7c41','3ff511','a56fee','ae1bf1','acc99b','a7600e','a16d0e','ad0284','3c4341','a3ef2a','400518','a657c9','780daa','a06153','ace754','a7c5ec','a5887e','aa3718','ad561c','7c4530','a810af','a4d239','a00ce0','a959b4','478670','abe80e','a59a0b','a1b991','ac7ac2','4077c7','a1048e','ad22c1','acf260','aa37f8','a80ff2','a78018','a683e8','a3b444','a13f10','a69415','a4ab91','a4dd42','a95151','3e386c','89901e','a622fc','a02b77','acc29c','a6ae24','a2558b','3c4347','a95dfe','7c7a5d','aacd30','ac9aaa','a4b7d8','3c4322','a00c04','3c68b4','a9c96f','a0fe5f','a94a63','85151a','e47def','ab8324','a2f437','7c72f1','ae2054','a8a1c5','3ff390','48b6c8','a6066b','c828a8','c038a3','a42cff','a8442f','a37273','a4f1af','a97e68','a17b60','40334a','ab5da8','a2a48f','a76535','a58fe0','adfd70','49d116','ae02f2','706212','7380c6','a6bb6d','a3ca69','a35707','3e6946','a6c354','a4bd24','a52e4d','a8021d','ac88a5','aba242','896487','a114a9','a4db92','a3fe7f','a48142','485611','3e1caa','a8ee20','06a100','a1180c','a861cc','e494f3','a98010','ad3368','a49a2c','a2c35a','aba54d','a52242','a20823','a2cefe','ae0878','ad9a4a','31de32','a60918','a7ff57','aca2ed','a64313','a3f816','a6901b','3ff4d8','ab0d4a','7c79bb','a34582','484c82','ab27f1','a95d13','5100e8','a82f51','3ff569','a86d38','a08bf8','adf473','a1cc71','a96eb4','a22013','a25862','a7dd69','c0028d','ac3eb8','adf571','adcc0c','ac7f1e','ab0165','a0bd61','a39adb','758617','a7d74b','a2941b','491d87','aa650d','adaef7','a152d2','a98ea2','a6db3f','a7dd15','a310c9','3857dd','a86c7b','a3e7e6','a02599','3e39e2','aaae48','aa8bc4','a60076','867fc6','42417d','a97249','a55b36','a9535e','a1eafa','a144e5','a86212','a0c9b6','ac8086','a54d73','a42d4d','acc161','a1780c','4ca24d','aa68ad','a5c4db','403ad8','a1a4f2','aa0dfa','a4eafd','a08818','a902b5','a6978b','ab9dc6','a92465','7103a7','ace4ee','a07c62','406ded','a0f112','aa12bb','ac6c14','a7288f','a4f398','3ff2aa','a86b6b','406845','c06047','a7da21','ae5b49','ab3bea','ac9a4a','a5af1c','3d3195','3440ce','a63997','484f9e','acc9d0','a0fcb7','a9536c','ae0480','a38125','a5de7c','a742a0','a7e579','44078b','406cbc','a5c0c6','ab2026','abddd9','3ddc6b','79a03f','78155a','a7ed1c','a05ff5','a6df8d','40335a','aae952','a74d53','acc8b3','a6d95b','ac9aca','a77aed','a3e14c','ac9c37','a9c58e','a107b4','a7c4fc','a41558','a4c004','a049ef','aa25d9','a152a6','aa73ec','c04a75','aa591b','a9d688','840fa4','a67074','a5221b','a6adb1','a6cbdd','ad58f6','a7d043','8003e6','acfbaf','a804df','aa96d6','a55986','a33a42','a3f1c2','a41078','a7797d','a4e3ee','a23bba','ab2bde','ae2034','a0ef77','a54f80','710226','a9e6b7','4062dd','aa1423','a90ba7','780118','3ff169','4063df','a1c7b0','a38229','a09829','781031','7cfa04','ae10b6','395a1c','3d33b1','0681d5','a9e89b','89404d','a4d6bd','7c083c','780fce','acb333','ac6dbd','a8b939','a5bb5a','a5e5e7','896ecb','a387f4','a3bc8e','ad5f11','31dc55','acde5f','3d47dc','abd7e0','3004f7','a1e4d9','a70a89','a1304f','a666fd','abc030','a9d770','a909da','a6dd36','a684c7','a6c855','8962ea','a66b8e','ac7333','c1e405','a13290','a0c462','a4f57e','3949e9','ace8e1','aa153f','ad6282','a1641b','e48cb5','a1e038','aa4a65','a9e4bb','7c7a67','a097a4','a3a69f','a038a5','a53810','ab4fea','a4f8fd','ac2e48','a796a7','49d1a6','ad512c','a03b8d','acb54b','acc72c','a0e913','aaa96f','76ccce','ada4ba','adc1cb','a955d8','a4f31e','a8c2c3','adeca0','ad5b7f','a7dc92','ab0001','4ca614','a51bd6','a1b70e','a18b90','a8ad10','a0373e','780246','a1092b','ac7112','a39105','aa9bc7','406ddc','a98d9b','a6a7d6','a12724','a480fa','ad3410','04c118','c01c5b','acb850','aafb95','a2a76c','a5bec7','493285','ab57b0','a12fbd','4b404f','406d47','ad3422','a2d549','a2cbc9','a28e15','a08842','505d27','a448db','48ca6c','424932','460933','ae10b7','a50381','c057b3','a144e3','a68dee','a966a3','ac49e8','ad105b','a55ac8','a310c4','a1f307','06a2e2','a25f1c','4b3a81','a02b97','ae0ce1','a35ae9','ab3525','a0cc0a','ad60fc','a8c439','a4db9c','a1e37f','a3314d','a6c13d','402b12','a6a6b4','ae10e5','a02fb1','a14b06','a575f4','86e890','a62817','ac8368','a54347','a0388b','a32460','43ea15','ab63c2','a0d9dd','a57650','a8506b','a1e7a8','ac4d43','ab5f53','a4fc32','a6d158','a3b724','3ff50c','a050f6','0ac91d','a5611b','a97b80','a256b2','3d78f8','a72722','a50419','a3026f','a7220a','a9a3c9','a5e89f','ad2d3d','7c6a62','aa24e1','ac57ca','a8170d','399c01','a02c50','a91268','a21c53','e48003','aab364','a7137b','407780','8404f4','a54959','a3e6b8','ae01f7','acd52e','a649ce','aaf5b6','39667c','a20dd0','7811f4','a59776','a8771c','a3952c','a3625f','a9df77','a051e3','47929a','a3c24a','aad287','ab49f2','a16317','acaab7','7c2e48','a197be','ada0ff','a7cbb7','3e747c','a6d446','a100c2','39b246','a64c4a','30015b','abcb9c','a1fc60','3d2c37','4d20fe','aa8f1e','a84836','a98f5d','abbe51','a0357d','7c4434','ab68c8','3e6db0','ad4062','aaecb8','a81435','894082','ad12e0','a75ce5','4404dd','c0220b','7808cc','ae0581','440965','010000','a66a33','7c1185','aa1568','a99197','a193e0','a2f5b6','e494ec','ace376','a03b68','a1e13d','a4e619','a08e2c','a067d7','346082','a67b3d','4403bb','896571','a99565','a32bbd','ac32f6','a394a7','a030b8','aba802','ab07af','ac9aa0','a1cb72','a59ef0','a380f1','0101d4','a54ad9','adc93b','a1b7d1','a44a4b','a27ff8','a84742','a2609d','a1e3b4','aa2026','ac3c9a','a6eb12','491d81','a83673','484497','781585','a0c82e','ad2062','a36c65','4b37ed','aaf9cc','aa93b7','a77c8d','a36cac','78042c','a88cf5','a7d1da','a66050','a5a038','a45e47','ace3f7','a2c8b3','a95528','a8690e','a2a307','400cfa','a7009b','a65f68','39cf03','a63649','aacfe9','a82ca7','a0153d','a39fbb','7c7acd','4d2242','3ff6c7','a50eaa','49d4a4','a18e6b','a2226f','a57666','a994f9','a0b522','4d2225','a87134','a14821','39bce8','a0f5e7','a03b8e','0d0791','45899a','ac2fe4','a58138','aa7e51','a841f5','a14b26','a48467','48ae21','68327b','4b365d','a195f6','780b88','a64d82','a4fa41','ac950f','a0d23b','aaf575','aa74e9','a8ed9b','a196e1','406a7c','0ae02d','7817b6','ab02de','440017','3d41b3','ad875f','3eda73','aa83f3','a29e6f','a728d1','aa0cc3','a70023','a7e0fa','3ff141','aba4ea','45d069','ab8a47','3ccd44','8a0286','ad8111','a08cea','abe496','406a9c','e49318','a05a16','a6e6ef','7c6aef','a35329','aacd50','a26434','a165e4','aa6cda','a66011','a4d471','ad8fbc','a793a3','424b42','85c970','a013ae','adaf98','a3d0c8','aa4bc0','4b85cb','a09e52','c0072c','ac2e66','a137b9','a712da','0a0097','ab037f','a2e220','a63f77','a0cbc7','a38293','a2e070','a4f436','a0521e','a10171','ae4edc','a102fd','aa1e08','adb34f','a408db','506dc6','a4e99a','4b2ba5','a2a5b2','3ff3bc','ae13d1','a3b796','acb105','3d0855','ae0892','a5ff17','7105ae','a18781','896489','3d7f46','a64cdb','a658d8','a9dfeb','a91486','7811a0','0ac365','a3176b','ac33ea','a2ed5e','a8ee5d','a1a7c4','a0e600','a188c0','400067','7c52fb','ad9cd9','a87b8a','a0b674','a88c23','ac3083','a3f841','a4d58f','a80626','a61565','a3e45b','869122','a9d87d','a14824','ab5caa','a1397d','a75633','a3486c','3fecd5','ac4e92','a3576f','ac9ba0','a89b30','a6d19e','4b064d','a50398','a3e86f','a4b827','a65ae8','ae5f0e','a35c27','a0f1ba','adfdb8','3965a5','ada6d3','a874e1','ad3ae3','a6b414','aa6d0e','496c26','ab4ec5','a7c06a','a2bdb6','4d2249','a2e6dd','7c19cf','4b0ecc','c05ce8','abb58b','ad05f3','ac7d4d','a2990a','780293','ab04e6','3954f2','4b1bf5','0d09f1','a6a71f','76cdb0','aabd2b','ab1abe','a7622c','a1f7a7','ac80f8','a43c51','acdb1e','502cdc','a14ea8','a22dc4','76cdc1','abe872','aa779d','4b4c47','a82173','3ff2ed','a095ef','ac8d66','a32015','a7fb52','ad4d5d','ac95f6','a0205f','407766','a67a34','a9c9a5','4ca1b9','a3b85b','ac18fe','4d229b','a18328','a0ebdc','49d0ad','a8a955','adffa1','a5a3f5','acfae9','a54e98','4851b3','a9d6d1','abcc04','e49437','abf6a4','a241c3','7c2590','ae1e61','a42035','43bea5','a6bf98','a08021','aaa5ec','a39104','a02fb0','aad835','3ff034','ad022f','a906dd','a197e4','ad7606','a17a43','a31ffc','a1051d','ab59f2','a392b8','3ef854','a24636','aba7d9','ad2174','a46791','406de8','a80b31','750257','01015e','a82c1a','a88b3b','3d63bc','aa7a54','a89fc1','7807b0','c817bd','89617e','accf2c','ac852a','a12adb','48ca66','aa945d','3ddc69','38d61c','484c4a','3e6c7c','a1dc39','a085f9','7810ea','a43494','a9a749','a204e5','acef2d','406842','7cf866','ae571a','aae82f','a1a0fd','8004e0','a952b5','a90a01','a39d7b','a48d2a','78921a','406f65','0b4040','440396','40740f','3d3c85','a1219f','3fede0','ac3d43','a8c689','a20bba','a06dcd','a03651','a0b3fc','3d07b6','acbd20','abcda6','a12177','a0e429','a05aea','4b07cf','aa79a6','a7d930','c01910','a33be5','a752b3','a49625','a71ea3','a2f5b0','a05582','acb4f9','a5731f','a712e6','448444','a206e7','a5b6a5','a9808f','a0b815','3ff828','ad9a71','a99ab8','7c1a99','a8953c','c02876','ac5af1','ae54dd','ac11a7','a37cae','a18eef','ae2916','48aca4','a1fda8','a65f37','4b3733','a4d4f3','a8132c','a3db0f','3c5426','c00d40','a6321f','a1f44e','300682','a31894','a2a447','a0f47b','a97e08','a31ca9','406dcd','a042ba','abdd30','3d1b88','424242','a6af03','407468','a21028','ab7b67','a9289d','a7d316','a05480','79a061','a5f121','a9d550','a5b850','a51980','a3db24','a661cf','a6fea9','a1bff3','a3e68e','7c0bc3','a769b0','a0aa50','a4e2e9','7c5766','a3defd','aa0695','7c6729','a02d32','78086a','aa43ed','a7a5ef','a3fb8c','a704c0','0d0a9b','a1ff05','ae57ca','a7280f','a3e4ff','a27dd9','a7c57f','78069d','8001ab','a76c55','adb25b','a36cae','a53bc6','ab0b93','a5575d','c051e6','adc02a','a51358','aca760','392997','a03143','459e8f','addf59','a69c57','3e5a90','ab5dbe','a8ba64','44a7c2','aa3579','abe30b','a4d834','ace190','a3de62','3d22da','a90b7e','502cc3','4b3abc','a6250b','a3380b','ab1fd0','a668da','7c6aec','aa019c','780995','a97130','a78e2e','494104','3ee707','a5fee9','a0413d','a88ccc','a2050d','aa3fdf','a565bf','add0e8','a006d2','a5b4e1','a04b7d','aa99ab','440768','c820f2','a5a2ba','a130f8','a3a179','a6daf4','a5bb40','a18331','a3fb40','4d2296','a25678','a1419c','a9e41d','ab0d7a','484201','ae5df6','ad7c38','8a03c8','484bbb','aca20d','a5006a','a4f4ae','a15739','a0db0d','a3e9f3','a74909','780e7e','c0223e','aa8eba','aa27fd','a54d0b','a358a5','a7a0c8','ab7995','a39559','a7a151','ae4a28','ae04ea','abaf1d','a6a068','a963cd','a14034','e49210','ad11d9','a638bc','78161e','aca61a','a95a60','a0c881','a7f283','8960dc','406c79','3ff690','a1334b','ab0fbf','3434ce','a50024','3d0d57','a09ab9','a5ba6c','aa1b0e','a69f2d','a1188d','4074c1','a6a607','a398d2','a06ab1','78068d','a36aab','781552','78115d','a1f05a','71bf00','40718d','44069e','3d362a','aa42df','4ca781','a33102','780fb3','a9ae47','a442c4','a32806','a14da8','acd65f','a67e04','ab4be4','a88bad','a3627f','a5fe17','89644e','503dba','406d18','407685','a9f038','a82727','a7118c','a4a739','abb622','3f0205','ac9c49','43c8d0','a62732','3f046e','c04438','3ff1a7','3d148d','7c6334','a2cd50','a04f20','3ff344','a6741e','a7d395','c06824','a0360e','a20a24','3e69d3','a36acb','a26279','3d3537','780b42','398435','c820d4','a9a6d2','7c678c','406f7b','a555e2','a56634','a46e0f','a3f071','ac12d5','a521e1','ab178f','acf866','aaeae9','a838c0','a5b769','a446f1','ac1f1a','a944eb','a55473','aa4d12','a62151','a84841','406c3b','aaf264','a64a8e','a358a9','06a1bf','ab5dc6','a8efd3','a25cd3','ad564e','440a3f','a8a2dc','acd253','a8082f','3ed1f8','aa0d95','abf522','a6bb17','a1ff0a','a123bc','780b34','3d1618','a8ae7f','4ba95a','a982c2','ac876b','45acaa','c025d2','ad01ab','780e76','a0f92d','add102','a95ef3','a9bde6','a7004d','ac26f8','a54d22','8a01eb','a740ca','ab034d','a67f37','a5930f','a82b8e','a7032e','a4e46b','a2b86b','7c0dac','78025f','ac1c98','4b1918','a54fe6','ab2d65','031000','8411a4','add692','ace5f5','ace3b7','a064ed','a81f28','a2529e','aafd93','4b0660','a7d212','a43221','adfff0','a0037f','a64854','7c6b11','405bfc','a289a5','a1e605','a3a36c','ab6277','7c2ba6','40712f','38395a','a4540d','40720a','a86b5d','a55aa8','a45570','a462a4','a3e573','a092c3','a94f86','a01e1c','3d45f4','a9df7c','a4cf0d','ac1057','a9581a','ab5a1d','acb0e2','a8aa7e','a711c6','a54a8f','485344','a33937','a815a1','a4db51','a0d561','a02dd2','a9384e','8964fe','a4523a','a3b3dc','aa4932','a6b2ec','a7e8e1','a32af8','a41043','a02373','a20d14','a15b12','c059ca','aa8548','a38238','a75b49','a4f5d2','ae5a4c','4bb1e5','a56fd9','ae58e7','4aab02','a6b83f','780d59','781108','780d39','a7c93e','aa8741','ab7e69','a001a0','a12760','a6d7c9','adaad1','ac440a','a247dc','adb55e','ad42ea','a9559a','a54e73','a3e5e2','c063ce','a0979d','c02098','a35e18','acc2d4','a75afd','3f0129','a65f88','ab6391','aaa054','a364a8','a6242e','a61f45','7c800e','add847','a7b09b','a2f966','479173','ae0389','3951c1','ac031d','7cf86a','ae4ea6','a00062','ad3b74','a64c5a','a63a85','490603','15407e','ace391','a4bbad','a9db1d','a8c058','393423','402e83','a1ea6d','a1b621','a9778d','a68b47','a81ac4','a4b47f','acb3cd','ad5c29','a31609','a04a61','7c79b9','a60751','a025a8','a15e20','aa50b7','a13e85','a68b71','c82846','a9d1ed','a3bad1','aa01a1','4786e7','a69582','7808ca','c008d8','a1a669','a9dafa','a97791','a0d99a','a4e368','a1f33a','3ff638','300661','a1b9b1','a1b109','a4d826','a4a9d1','c023bb','ab54ec','a8b74e','4aaa4a','a6ab99','a2f3e6','4406b5','a2e942','a4cce5','a6558f','a1a0fc','a4c7eb','ad0bbc','ac1baf','7c1987','14f11f','ad46d8','a45f37','a38ffa','a198db','ab355b','a4f179','a375f1','a76d37','aadc74','5110c8','a16417','a00680','06a1cb','a5d04c','aa57d2','a88ecd','ae009e','c0737d','a46a1f','a96d35','406d10','a9ed95','acc4bc','a6aef7','a6612c','5000b4','ac6ee4','ac78b0','a5ead0','a3e8d2','a6fe8b','a81320','aa2452','a1e4d6','a1038d','a70b43','a661bd','a05632','a8ae2e','a9da27','ae07f9','a6690b','ac3c43','a5b52f','a4cb5f','780820','a1aef7','3d2371','a832aa','c045a5','a78140','ae2043','406a79','ae4f15','78059b','a6a564','a681c6','a1cebd','ae1116','403166','a10bf3','a8135e','1f1d5b','a54cf6','a26a2c','a1588a','abdbcc','7c6af3','5113c5','a4b50f','aab0a1','a7c9ca','a596b1','a3614c','a009b7','a6e32b','800700','a857f1','ad836f','ac5356','a0bd62','a70a79','a03cb1','a794d3','39d228','aac209','acd333','a1949f','a3ac2f','ab829c','a4ad9c','a1ff22','402dce','a07ee3','a59766','6831f8','3d275b','7c80fe','a3ffed','a6d5ff','a0dc72','780e85','accc70','abe299','4b38fe','a85e95','a5ea1b','a18178','3ff57d','a36cef','7c19bd','78149d','aa3eaa','a8900a','adb428','ace112','abc544','aa6fef','a2e997','a99eb2','780a2c','aa2f7c','ad2c9f','780a43','a4034e','ad8f00','aa13ff','a1968d','a40700','ad4778','ac9bf9','a3cfa9','ac4450','ab0a32','a08b2b','a2f3d4','aa0166','4ca56d','a1b098','a96480','502d26','ae4a7d','ab030e','3ff2d5','a9d3f6','c019a9','a54892','3e1688','3e5c96','3ce02b','a9da07','a60833','a2fa4d','a1044c','06a0bd','460555','4b4bc5','a65917','4b4c54','ad47d8','aa253a','a1f36f','a53136','a8a4a2','a7b9e3','a9e65d','a90b3c','a4c37e','a8b79f','c063d0','a46d19','a4ae53','ab4545','a1f52f','7c7b60','a82d92','a33757','7c6060','a63673','a5123c','7c6aad','ad5658','a384ab','a9cd8f','aa3b30','a3910c','a3ebc4','a8b092','4bb142','a7d1ef','c05c11','3a2641','a6847e','acd8a3','7c5744','a32b7a','aa3bc2','a5d308','a92d22','aa1aa9','402cac','a206ec','a1bb83','adbca5','478747','aa0e26','a8a386','a0235a','c81d73','406de5','a9b5c2','a5f6bb','a1c65e','896c28','a63993','a5794f','506e38','a29405','aaf8cb','a07162','3c433c','ae4ee2','a54b09','a358e7','a9adc0','780d84','a7456d','7c7c34','ae1f82','a96e78','ab2120','3d1bbe','ac1988','aa9bc6','ad3e16','ab7d09','a35e4b','78080a','a9ea14','780d9c','a26001','a7adb2','a9605b','a8adc4','a20005','ad4f2a','a16616','78080b','ae04a2','a32427','a98c1e','a588b1','ab51e5','ae0110','adf1d3','a91ea7','a9e668','4cc3f1','3ff461','a5b4e9','a44248','a3b49b','aa6756','a10adf','ad3fbd','accb65','adda0c','a057eb','a7e09e','ae4a81','a347a0','4071ab','a6c7cf','7803be','3fedf2','a07e5e','aa30e8','acbbbe','ac9388','c803d4','a4ca66','a5dcd0','ae4a2a','aacd83','acbe10','a4a024','ab09b9','a3b8ad','a9e552','aa6955','addb3e','a5bdf5','a28992','3ff057','a3431a','a1bfba','a178b1','a3e377','a120d7','484c39','aac236','a5e204','31ddd7','a5cb51','c07f94','aae86f','aa8798','7c5b76','a55a7d','abaed2','a6432e','ad5ad8','4b5224','49c218','ad2c63','781011','a777b8','ad0e34','ab8ea3','a5ad00','a355ed','a4d248','a07448','3e66c7','ab624f','a065f7','a7c315','0d06e5','ae1457','3e723c','a954d8','a2bdf6','a8258e','acd25b','a2a7c4','a27d5a','a7a9c5','a780f8','3c71e1','a7d01c','a0d82a','a54dae','3991a3','acdfdf','7807fd','a95c58','ad4a0a','3ff727','a43b41','a0c32c','a89814','aba6e6','7810a1','780f9d','a91eca','ac69dd','3ec1b4','a5b329','ad0b87','ad4d9d','3ed438','a8c79e','a8358c','781589','ab89f2','ac82bc','404fbc','a64046','a1d81f','8880a3','a80c4e','3aabff','a03368','a64f0e','3d66c6','a50f0e','adb99e','75043c','a5b81d','a052b4','acb89b','7c68b7','401cc0','a5745b','405257','a8a106','a7f405','aa8243','a19741','ace194','ab6c01','a56122','a190b5','3ff240','4b43b1','a844ae','780d23','a6db55','aca9d4','ad2114','384684','a052b6','a63c0b','3f050b','a6d3c2','a15f7e','a2a67c','4b3578','c020ff','a0239c','7c1065','406a0d','3450d5','a4b1f4','a1e14a','780370','c8809c','a54c6d','a78cb4','a0ba51','a3e615','a3b601','a0fc0a','a88e29','a1e798','ac79ef','a9ea6a','a362ae','a10efc','3ce735','a88ea1','a13ec0','a6e6ae','aad117','c04a2b','a6b11c','a03401','3d42ff','7c7803','a175fc','7c1695','a50a91','a9824c','a0e1f1','a34b70','a9b3fb','a80830','a05b29','a99fdd','a00e0e','a72d0c','ad878a','a813dd','a50833','48ada3','acbfd4','a17268','ad7de5','acd8d1','3d27d2','a96329','a2c1c6','a6701d','a2ec0a','06a2c3','ab172f','42445e','3d0cbe','3e120c','a1775b','aabb95','e47e28','80073b','a97ded','a55708','a60a48','a05777','7c7ba9','a22068','48fa0c','a2d695','a4ea8e','7c7ba3','896452','abfded','a501e0','3d06eb','3aabfe','a44e41','a5af37','728666','48da90','a37fef','adf4a4','ab4ae9','a3fcfb','3d5506','4b2b2d','47c304','a0b9ac','3d32d6','ab1f53','a70243','7c7c32','3e537e','ac44f6','aae6b7','c040f9','a21c18','7c80fb','4d2131','a815de','a4a40c','abbdaa','a8b516','a3e763','acd203','7c4dd7','a83b3e','4b27d8','a79104','4aa506','a4e1ec','ab95cc','a9d5c2','a4654f','a03303','4caa36','a74d8e','a24cfb','a0cf99','460688','ad2d46','a2f430','a9e988','4aaa47','a6bb15','a3827e','489e28','a8a568','406c28','a30361','4bb1c6','3cc249','a38768','a70d72','a9ab07','a6bbc7','a63a72','a4ebf1','a39ac6','ad93c7','a12921','44044f','a4589f','a835af','a1a71c','acbc89','a4fa17','a2542e','a891b7','aca608','a73aec','ad4453','5110da','739483','a5074f','a05db9','ad6478','3d1948','ab5b30','a04669','a59e1a','a4b288','a35ed0','a62d4a','a29f6d','a09616','a7f6c2','a13470','aaf4a2','a04af7','a92082','a9dec3','a46b3d','424767','4ba284','a9198d','acb83b','ac148f','7c37d7','ae0088','a60435','7c7c3d','a02412','39c01c','4d0131','acb319','ae173e','a03bc1','aa6b1d','a804f8','4d2224','ad40e1','4010c3','a70abe','a44b5e','406d69','a0e247','7c3f03','401e36','ad52e0','8013a3','a9227a','702c0a','478787','a3b2c7','a6c294','a89cdc','ae045d','4d2279','c06883','c820ce','a52ad2','a0875d','ac1f70','c0722e','a64ac6','a5d3f9','a35383','acc776','a56cd5','a38b9d','407742','a31013','a18d79','a29434','a0e766','06a256','a2340d','a0d40f','ac7f4d','a9e40a','a5823e','a5f550','aa3a19','adfe90','accc83','ae47da','a9ab40','a1d074','c01759','a9b330','ad8bec','ad177f','4b4352','a807be','ace306','3e1a67','aa83ea','ad4bca','040039','7802fd','3988f4','7c6d76','a38c9e','a3ee0b','a263eb','3d0cf8','acc5b4','508003','a49819','a08346','a18ed0','a62ca2','3ff7b7','ab9677','ae01eb','45ac32','a22275','aa4d0f','a38bf0','ac8097','aacfd2','a9dfa2','3cc1db','a291ce','a2c467','a2d6b6','7c7ad0','a6ebee','8967db','a37e48','a1b324','a036fc','7c07a4','aa08dd','7c8130','4d2264','a98732','a97089','a59b50','3cd9d2','a63ea2','a0ec07','a79618','adfea1','7c5dd0','a3e88a','a1f19b','e4919e','a190ab','780c4a','a3b6b6','7c390b','7c3e84','3cc82c','3d3f9c','a6b439','a2b269','a2ea87','ab9126','a1b175','a0d2b6','7809db','a60814','a928f4','a11edb','a3e421','aa6628','a8d3e5','a902ba','a807cf','acdd15','a6ff8d','a6073d','86d68f','7c2114','adb8cb','ac6856','a02d16','abfbdd','ae0507','a5fb93','a1b75f','a05b89','a09bbd','aa9dc1','c07b7a','a9347f','86d98e','75043e','4007f2','ad64ab','ae4a25','a9a983','3ff7b3','e48424','a5c0c3','781499','ae093d','ab7e30','706214','a97690','a411b7','344505','a8d238','a77ff3','aa7544','a3f518','a1752d','a3a9d1','a6016f','a1d80d','7020a3','a4f6fe','a82025','a9702c','a969c7','a5f4be','7814de','a4adef','ac70e9','aa1f0b','a7d8d1','ab20ae','aa38d0','a1c78a','a6aaaa','461f53','a398c3','8964a7','78133f','a555d7','a2dd2a','a53508','a98c7a','a87370','a9c79e','3d5535','aa95dd','86ef8e','aba05f','a568c9','a0b6dc','a92b88','a6d505','a07fe9','a956c1','780f58','122436','ac916e','a7d406','7c37fb','ae04ef','ab0bed','a7c3f9','a7906b','c04b54','4008ad','a130c0','acb5f4','155c02','8006e1','503da2','adfed7','3c542a','a0a233','a755e9','a309b4','4481c2','ab8a9b','a4cbc8','a0d1eb','a06f5a','39b9cb','a067fb','ad5262','abe3ce','aa66d0','a713b9','3ff69e','abcc01','a8097c','42492b','a066e3','ad8b41','a4c036','a3bdb4','a2ae1e','ab0339','aba772','a1e8d9','485324','3d3371','a4ce11','ad1717','a2c793','a4970d','7c162b','7c1ab9','ad4f81','3ff15b','a6f12b','acbd82','a9ea05','406d0c','a4bed7','a372bf','a0b2bd','406ea3','78064b','a66be4','a637bc','a551c6','7c394b','76cdb2','3b76ae','a65161','ae11b1','a096b2','a61f42','a2b564','491d86','89644c','a0cbd6','ad942b','c08176','a3c68c','a15c41','781684','c81d25','342314','a77c00','7c617b','a682b3','7800da','a0269d','a3712f','a0698b','a52ed0','a08f6d','8a0241','7c60cb','43ead5','c8280d','a37f98','a083b2','a0cf9b','aab186','c049f0','c063cc','acfa3d','a91597','a69c80','a2373b','a0b42d','a727c0','ad28da','c04779','aba8e9','a55351','a44dd2','a5ec5c','a9d3e2','a5deec','abd2a0','a692ea','a21692','a19d4e','4ca4e6','800b67','a22e59','8001c5','acbfa4','a71c25','402cf2','781860','a4191a','aba798','a07ecb','a03ad1','ab6d88','a8a3ac','4b43ad','add981','4d0010','acb88f','aa195b','a9a429','ade0bb','aa9876','a84034','a6fb15','a6192b','3c43bf','a6eae4','a2ff72','a82a5c','a69f09','a4ea9d','a5de4a','3851e4','a647e6','a9e648','a8b22f','4409fd','aa5bc7','a804a7','a3bc45','c0202d','aa454e','440036','ac71de','a13f4e','ad2d6d','a5fee6','a0e9dc','a968ac','500026','ad1c0f','a31ee0','a1973f','a4d1e1','ade582','ace621','a3b7b3','71c362','ad111f','3e50f3','aa6750','a721be','5051c7','a49892','3e7a41','ab2ea6','ab20ee','a50757','a456bb','5113db','8990e1','0a404d','a1e4b6','4d0218','78158a','780794','a0ac7b','a90dbe','a4e4e9','a3e520','ac742e','a0e934','abebd4','a58272','4b1b00','a219d4','a40b37','a20d5d','a6c35d','a7626b','ab005f','a24ceb','a3572a','a1746f','ae112f','aa6fd2','a84a69','7c44ae','7c7f8f','3ff4f3','aaaa07','48bdd5','3d0e57','ae268e','a68dbb','aba820','a181aa','47a402','ad8e52','a8150c','a2f28a','ac3dde','aa22d2','3ff646','a20a32','a47994','a45935','a3fa72','3c4338','4ca98d','adf521','ac3c9b','a417ad','7c6c17','a20639','a19d34','a88b33','ad2169','7815ca','a26df3','a0cfec','a36d37','a8fd05','a81018','acaa56','ab7d92','7c76c0','abb377','aa9748','aa3bcd','a80f91','adb706','a31a56','a11aa6','8964d6','478454','3c70b6','c01b6c','ac4102','a4414a','a7c2d8','a9e5b0','a1acd5','a1ed2e','4b43aa','3e744c','a0af91','a2187d','acd4d9','a9928d','a4e1d2','a449a7','a0e047','a52d91','a63863','484208','a82baa','a2859c','a1ad90','3fec90','a0d06c','aba33c','3e7456','a3f1e9','3007e2','a1b100','458d70','7cf9fb','a7c7fb','3feb62','e48b2f','a8afd2','a1aa43','40785b','a0434a','3d4893','4481ae','a9c5d4','a0bc06','a7afdf','ae60e8','3d24e4','a8f615','a5b3da','a3f0e2','aa688a','a03367','7c68d3','a73c75','c01b4f','a482d5','a3aff0','ab3c34','a6f49d','a6db36','3ff102','ac00d0','440921','abba25','a1c5f0','a050d5','a3183f','a15460','a693ba','a1e6a5','a0c4f8','4a9c4c','a3ba9b','a13adc','3e6a5d','a1afb1','8991aa','a4ef67','ad9d42','a06310','484211','aa17e5','a063df','a68245','8963b9','a0308a','a67206','a0b6eb','a69347','a9d659','a04d7e','3ff759','47c404','a32045','a69ba8','7c019d','ac24ab','a49846','a19340','a3ad88','a3076c','a2a1ff','a09c69','405f7d','adf068','ac4381','a7f642','a2dff5','a2760f','e49481','ac3221','ac11eb','a2a995','a950cb','a71106','ab4e9d','a95d08','440691','a24f04','ae593f','ad9e5c','a97a71','a5a378','a6ffcb','a63baf','a09281','706217','780343','abfa36','a63563','a37a59','38e9db','780b7d','505e8d','3ff534','a10e2d','a44262','a1863a','780d8e','3ed61f','abc865','aaaaad','a09646','a64df2','a70c47','a35e47','a026cf','a0006a','aaed58','49d46c','a87804','ac3dc8','acfc05','aa3ee4','c08321','ac5828','a9a346','ae223d','add297','3e623e','abd294','71c007','ae2eda','a23195','ad6c05','3df5ab','ab0ff9','a7c646','71c078','7c2df1','a0c069','780c1a','ad7224','a0798c','ab1d1b','a36c7d','49d213','ac1eef','ab5063','a4dc56','7808a1','ace285','ae1143','a95147','ab138e','4b1af2','3ff370','7807fe','a26910','ae1447','ad79bb','a196d1','a1b9bf','ab362c','a169bc','ab2551','a8ca0b','a5714a','a21ca9','780dc6','ab5168','7809e4','ae1ba9','a1b8b8','40621b','4aaa14','a8ee8e','ad4b35','a6b979','484359','abefba','a6d0af','78159e','780a89','a8f63b','78007c','abadf6','ac1ab7','a80f15','a1db3a','a5c4e2','ad8e5c','4bcc22','345542','a0144e','a06744','add952','3fee16','3ff823','adf1d5','a1fcf8','ae5f06','a7ef40','44055e','4ada93','4842a9','3d09fe','a361f0','7811a5','a81ed7','aba5ec','39464f','ae68bd','a82d8f','a9bd87','3fea79','7c0c49','3e3c92','a0b5a1','780006','abbd69','a64da4','ae120a','ac28fa','a7bc4f','781005','a31c39','ab9a29','a2e7d1','7c24be','ac9ad4','a59892','a1956c','a95156','a4b996','a33b07','ac2aa9','a9631f','a54d1d','4aaa15','47b9f4','a9f047','a0317e','a0573d','a69f9c','4d20f7','ad8d72','a11971','3d77a7','a9ef35','a7c6d2','a90185','4b3510','a68caa','ab7768','a206d1','ab2b5e','a3effe','406fa3','a8adda','7808d7','7c33f5','ab9896','4ca9ea','a51290','a0a258','7c1b2c','aae26d','7c4316','a8c0ff','ac7368','a660b3','abc827','a0a5d3','c07cf9','a34077','ae1393','aa1c21','a4f95b','a22f51','adf261','a00102','a49bf0','a3dbad','038615','aa8e4d','a80d80','a06ded','a9ef2b','4b06f4','3d3194','ac7b33','a9d66c','a3d541','a83ce5','780c4f','c81737','a52979','a07bb4','3ff487','a922ae','a9cdfc','896c1a','a0dc03','4d010c','a5297f','ab89c8','aa5aef','a1ea39','a88baf','3e5cda','a354a1','a9541c','7c168b','a8132e','a15a43','4aaa41','a9a36f','a6aa4a','a5324a','a12cf3','aae7a2','7c6cf1','8964a3','ac71d0','a978ea','3b7b9e','84b779','a067f3','7804fc','38b0dc','ac1709','ab5b95','8007de','a73960','7c7fe0','a6ab8d','a5749c','4b1bb7','a1b998','899144','8403a4','a96def','a8c1df','a92846','acf823','a4533a','a463fa','79a04e','aa32c9','a73242','a5ebd0','ada160','404f04','a486ce','3e67b9','7c1689','781695','a3317e','c063c2','a119df','a97b09','a33815','ae07de','344141','a89887','a87ef3','a8c693','a6dac7','a984c1','a781a0','a2036b','4c3daa','7c0a71','ad9eca','a18ddb','a41627','a21690','ac779b','a1b61b','a04a26','401669','ac1d64','a9f1bc','a3ad64','3d5f99','a001f9','abaf47','acd5ba','a39d73','aa688e','acaf1b','a22dca','a88354','a3238a','a9931f','a76dc6','ab0789','a950a9','a30c8d','acb23f','a6ddbe','a90a8e','4d208a','a9be70','aaf001','a980fa','47c007','3e5c29','4a822a','a081d3','a5600a','aacf28','a00c59','a1b632','a3bbe0','aa8859','78058a','a18d10','a968f8','a252a7','a0d499','a11ab2','ad2d99','a21411','a6d1ac','7c0c41','4b3905','aae16b','a5ed02','c01a13','ab148d','4ac982','a66cd7','3ef528','40457a','a27cd5','a92b2b','a7ee43','ab43b2','a9e63a','a0387d','a1adb3','8a020d','a03e08','aa3637','a985d7','a6800a','a63052','a1b9c9','ab4ece','c045df','a5be3e','a6dec4','ac754b','3d1229','a666af','45a96c','acbc5f','a7e18b','800cc7','a5e7f3','a19b0f','89611a','aaaf84','3ff579','a9e759','a7c28d','a0bb07','407672','0d0b2a','aba5e7','a1e1e5','a6f713','a5396a','7812f2','a589bc','39a5e6','c04347','acd913','3d7e6e','a7c624','c04aee','ae174c','aa787e','a5123a','a10941','ad409a','aa244a','a8e6f7','a112d9','a90c5a','a34997','ab9217','a6b8f7','c02031','ad342e','4850f5','76cc6b','a938d5','a02074','a696bb','a4db2c','a0f921','a6be1a','4ab19a','c08870','a0a3aa','a62a80','780dc8','899081','c82090','a4c797','ab308b','a72834','4ada06','ab3fc0','a587db','a4d0e8','a8ed86','a89a67','4851af','a1c1ed','a9b406','a6b1c3','a4cb6f','abb0d3','43c6fb','a9528c','a6c60a','4ca739','a739a6','a1c144','a585ca','a63677','adbe80','a10835','a2191b','7c7c33','a001b0','7c0fcf','346114','a6eadd','400dbd','a24188','aa1f2f','a0fc23','ae68bb','c06644','a87586','3006b1','ab872b','a9bfed','48c50f','ab3e8a','4a9ed4','39a1e0','a45297','a01333','a128e2','a09d4f','a477e6','a1c39d','ab0eb3','a3aa0d','a87fe4','a896b9','a59b56','4d2192','42457d','ad66eb','a3e35c','39f05c','a6c616','a558b6','3ff770','7808ff','a043cb','ac8ff4','adf337','a617e3','a11882','780817','a1cb5c','aad058','78023e','a97742','a7e9f6','a0dbcc','a9a2d5','a69d43','4b40e4','ad8937','4481b0','c02262','506f11','00b2e6','ab5722','ab7e66','a9ed2e','a71484','89611f','ad4690','ac5fd6','a06082','c04b8d','15407a','a24d86','a00af3','ab8a2e','ad8926','a79de9','ad2b92','a054fb','a7dc9a','acdbf4','a890aa','a2d081','a81db1','a3d10f','a01f84','86efb0','a1a76c','ae4a60','ab4920','a9e67b','a184ff','3ff492','e09596','885963','a80518','38c59b','80039d','ac6abc','a24cc4','a129ba','a6554b','a30029','a0826a','a06fa0','8005ed','a90af7','4408fd','a60da2','a82a20','404b78','a204f3','a8b5b2','780673','a2dafe','a9dabf','aab276','a4d16b','a465dc','aa67f1','a17680','ac9ee1','a3cd7d','a65527','40166a','a7ff09','a520f2','a1c483','a2be59','45a215','3ff10c','3e7c20','a19344','a02a77','a257ac','a6e93e','3ff6a3','a4c6f2','440395','a55a7e','a47969','3ff63d','780a00','a96fd5','8967dc','ad55b5','a7f7b9','a63559','4840e0','a4a986','adafb5','a61e3b','a7704c','a759a0','a39ff4','a6a45e','484b62','a276b9','4b28e3','3c43ba','c07982','aaf968','a15c4b','a3a67b','c07752','3d1b07','aa5380','a38d34','a92bff','a056bc','a1f50d','3965b3','acdd38','ab026d','c0637c','a4cafb','a1a4fd','a5158e','a8634d','a9555a','a0fe4a','3e77e8','a5b278','a39e8a','a82e00','a6a8a5','ac94e5','a43b21','c05f31','a3fa03','a3a1f4','adb8c0','ab6d8f','aa8536','a02d68','7c02d2','406e32','3fedbf','ad57f1','a3e5cd','a5c564','4d2244','a045bc','a1a3da','a99eac','7802f4','a9db94','3ff3cf','06a131','a2d6a9','a29111','a69ecd','7380c0','a671d6','780932','4ba285','48aca5','add627','a963a3','a4b158','a77b2a','a8d844','a18d8e','48c9e2','a6db05','a3677c','a80f83','a6c44c','a4e2da','aa027e','a06f65','a6652b','0d83be','a19062','3b7b96','a1aadf','7c8106','a4ec3a','a4758b','3005df','3ed37f','a8b898','7380ca','c07712','e49284','a72b7a','a394fe','a9f757','ad98f7','a62f27','a7cef9','3ef1b3','a605f0','a463fd','addf5f','acb4e5','a8151d','7c4e36','76cc6e','a4d0cc','a4affb','780096','a43c7f','ab0b11','a4bc0a','a34f45','300621','7bb138','ae2911','a7c3eb','e02604','a07c13','a6e342','adce43','aade46','a4003d','ae6486','a54e87','4d217a','a93fa3','3e5fa1','a8b784','a1f8e7','3ecbb7','3d24fe','a0c20f','7c6651','a68cdf','a69c72','ab112f','344508','a6b69c','a9f67b','aac7f5','a47fd9','a1bab9','abcb63','a526e6','7c6b4c','a400b2','a2ee91','a81be0','a77706','adfff4','aa8662','ab3fec','a6c1e4','a4ab44','4ca224','3ff3d9','c03b5a','300695','ad24eb','a23333','a44438','aba961','acaf82','ac5249','a447d5','aa68ee','ab6b02','01808f','a1be6e','a34bc8','a348ce','ab2f78','ab20a4','a9b10f','a5392d','4d02c1','ac80ae','a51745','aae052','a828aa','a1c740','a39e50','ac9366','a2dc22','a1f043','a2429e','3ff0c1','a69e3a','a6867a','a0f24b','aabbe3','a83340','7c7a47','a155ad','3848bd','488078','ab543f','a5e020','ae06e7','a03f64','a59012','ac6283','a80561','48ae06','a5622d','a59cae','aa4978','aa8e32','a5927a','ab5f8a','a6f0b7','a5911c','75828a','ace31a','abfa4d','a24609','7c7ca8','a55b0a','3dffa1','adbd8e','c04fdd','a3fce7','a6cb9e','a97df6','a0d0a0','896354','a5ca47','a954dc','78067f','040102','a9c537','a254f8','7c6715','a3f944','a1db63','abbdfe','a27e20','71c345','a9585b','adfd72','a65449','7808fa','ac234f','a54d48','780ff0','a103b7','a4f55a','a39d57','780bef','ad1ec8','a1b051','a20c88','a663fb','a84093','aa457b','ac258e','a79b64','abcd31','a817ac','a43cc4','89405e','a0490b','a9d267','a17c62','a5a3a6','a1ca3a','a0de5a','acbb27','a4e788','a3323b','a3c1e6','aae65e','a14c51','c055d2','a6b39a','a61b73','a7bd16','c02682','a9fb67','a07874','aa0a1a','a1b8c5','a19398','a92aeb','a1cdd8','a0272a','ad642a','a631db','ae60f8','a1f0e8','a19fe9','aa2269','4ba26d','add184','adb014','a13234','4076f2','a126c7','7c6333','4ac9b4','7c4229','a110a1','ab4a99','c01a5a','a269fc','a0ba91','a6fd9d','48ce42','a96f91','a9662e','a18e8c','7c15ff','ad9637','a5a654','a1cf5c','a47166','7c293d','a6a407','a2fcf4','7c52ff','ae06e6','a6982f','a66c1c','a3301a','5113f8','3e120e','a13df9','a84c2a','424403','a32a35','7100be','a5391e','a6b5b8','ad2682','a51a49','a6a059','ac5466','ac3c0d','a7d732','aa3b04','a8dc3a','a6978d','adff70','a5d034','a8ed0a','a4e38f','71bf78','4a9b53','a40c60','3d1837','ac97f1','407238','a32b28','a912de','a76ad7','a83d67','7804c9','a351fc','7c168a','ac409a','3e1c9e','407803','30038f','ab4090','4067c6','adcd70','a6065d','aaf591','a2f839','ab6458','a5cd55','a72bd1','4caaec','a79d16','89641b','a7127c','a80352','aba841','a2a77b','a109b5','39c428','3ff56b','a63ba1','4ba892','868078','a85755','3efd3c','a05f11','a9d5ba','a2cfa8','aa2050','a96f04','7803b1','3febf8','ae0136','a9b310','aca137','ab6eb6','a7fb77','acc1fd','ac9b93','ae080c','a739cd','c0488a','ad18bc','a41d1d','8405aa','ae074e','a2d70e','7c33c6','a8d891','a355eb','add546','a2131a','4aa6a4','a989b8','aae98f','494109','a4f5c7','7c1685','43c174','152c29','a5cc1e','a03405','140ae9','a5844a','a7677a','3fea58','a705af','4aaa73','a7a2b4','ac710a','40645a','a03e09','780063','a1304b','505cd8','ae0232','a198d6','a9beb8','7c474c','7805ae','ab635d','ac1c2f','ab063f','a154d3','a59103','a2e48d','70c10d','a2420c','a476e9','ac9f5f','a14e56','a1bc1f','a33cb6','8004dc','a28975','a56ac6','a23689','a19783','a5d3fb','abbf44','3e60b2','3ff0f2','a02781','a04410','a8f506','aa1fd6','a0b5d1','a9e614','86dce6','a9b9f5','490d01','45ce50','adffe6','a5df46','a4bbb0','3fef7b','ad59e1','a81325','a06147','adbabd','a9cf0f','4849b6','ac2a95','a4ee7e','406f52','a02bd8','abf1bd','a65b27','a278b2','a83d95','aa781a','a6069d','a42b73','3e0173','a4b1a6','a4ea77','a0806a','a98cb1','ac9e4e','3ff4e8','a0e1cf','a1fb40','a7f289','3d14c1','a431ce','46c8c1','ab2c5d','7c7741','7c1bc3','a89e00','a8130f','a46b3e','a8e7d7','a1aef3','7cfa72','a543c0','a964b0','c022c0','a023ec','a0cd81','a8892e','a33565','ae04cb','a449dd','a5045b','4243c4','ad2823','8963b7','a7d5cc','a50f4d','3d202f','ab2e87','3ee45c','7c1bdd','3ff71b','3d7b2a','49d371','aafa35','a6ee69','a42df0','a95eb3','a62a1e','4843e7','406b4a','a7d90f','a89853','a4dcfd','4c3d8a','a9a037','a1c798','a10417','ad1a8d','a8373a','a1ebc4','ae01f8','ac9de6','a41794','a3f43c','a8ded6','a3fee7','850b30','a65692','a118cb','a196c0','a467de','abff66','a02bc8','a08e2b','a3c274','a1b153','ad1cb6','ab7e99','3d29d1','ae1eba','501f5d','4ca9c1','a8eb8e','a99580','71c025','aa02df','a880e7','a66a74','3d390b','a8e603','a44592','a36108','5113e0','a69b84','7c61c3','a0482b','a5693d','a6b45b','06a083','a4ce43','780819','3ddc6d','a9764e','a7adfd','e40087','a568a6','ab3409','a88b1b','aa9000','a281d5','acffbb','aad94a','a5b55f','a18b36','71c218','a158ff','a129d9','7c7018','3ff0cb','e4818c','78075e','505e2c','ab2044','ac263f','a89e08','a64194','a6601b','a4a30e','a984f2','a24b94','a1adfe','a10f41','4d0207','800bd6','ac9b57','a7a99f','a73b17','a18032','a242e5','a86cc5','3ff36b','490d04','a95c38','ab2d2e','a184d4','7813af','aaf3b0','4ada67','ab389b','4b5134','aa8aae','a6bf0f','800b23','ac1595','ac47dd','aad32b','aa50c0','ae0662','a6e24e','a9bb6c','a4651b','ac1020','a4cea3','7806a2','ab3f9e','a68483','accd09','ac8217','ab6e68','a8fbe8','7c6af8','a6ed6f','a700a4','a4cc29','ab629a','a6ff6a','ae2ee5','a7ec07','a55e4d','aab1cb','a200df','a8c267','c0785a','a820f8','aa3f58','a5866d','a70988','a38219','a41da7','a691c0','7c5c10','506e20','ad451a','4b5143','a70812','4ca4f8','ada9df','a4c735','4b303f','3dd821','a86f42','ac0bc7','aab9c4','3e5e5b','ac5569','4b255d','3ff54f','ac4e5c','ae2ed9','a959d8','a61a28','44078d','a4b57e','ac703b','a87b97','ae004c','a373c4','a6197b','a5f875','a83c5d','a018c5','44ce83','aae0fb','aafacd','a6f37e','3cc158','a82571','aa8050','a9da63','3d1876','a89f2d','aa763f','a3e3ee','a42b54','aa28e5','ae638d','ad0673','a2a548','a22273','a8fb5b','a86558','a735c8','a9c151','a8e2ec','c07efb','76ccd0','aa5d1a','a01412','a98ed4','a5dde6','aa30d3','ad6f3a','a800b5','a4dc0d','ac91ec','a58401','acc79d','a8e0b5','a4ca56','a504b3','a6108c','a21432','a1090e','7504c6','a91247','a9e6cb','88804d','a41845','ab9b13','a80a9b','a269e3','aa834b','a81593','a139d9','a95cb9','a41bb6','4d228f','c01d46','a0f03b','899143','3ff620','300808','c009d4','a3d833','7814b5','ac75b2','c1e525','ac160c','a39d11','a7d1ba','a3b0a9','a5c86e','c03b63','a6e1e3','a666be','a7a133','a1c9ac','a1d16a','c0737b','a1a849','aa37c1','a6c690','adbf31','3d413f','ad1590','a5db09','a91fd7','a109b3','a11e5c','a3442e','c00ce8','86d215','a4fc91','a05dfe','a6420f','ab20b9','e47eb4','ae08bb','a63a23','78169e','a4f469','a4df69','4b51c9','a26934','c81d80','a0c04b','c8222f','ab37f0','aa8496','406b25','a98cd9','a864ed','a68b79','a6ba3c','a442db','a5aad1','a9df2f','aab5b5','a35e8a','a4eeac','a4b8e4','abd982','a1ed4e','abb44d','3d46e3','a70b4d','a9c95a','a8b22b','a7f390','a4a538','a24c63','a04bb3','a179d7','71c336','a3b4a4','a38213','a842f8','ae06d9','ab7e80','a5963a','a64e35','c00101','a9bc36','3ff5e1','aaa5dc','a02279','a5c74e','a8cc5e','abb00c','a3e4f1','aaa26a','ac6f73','aafe80','aa7af1','ac9840','a66fa9','3ff3b8','ae1e66','4895eb','7c7ad6','a96fe8','a5eb61','adee45','4ab06e','a98203','a808ba','a5cbbd','ae0362','780110','a4e23a','a14f60','3965a6','401127','aa7114','a1d61f','aba0e1','a0dde9','a4f26b','39be20','a338a6','a26f79','a2160f','a5e08c','3ff483','aae911','adc7e8','a984f5','c044bb','aab65e','adcae3','a53e3d','7c3f4d','3e63e3','a3bc03','a00049','a11061','a2aa81','abb98a','a0219c','a3d415','7806ae','a6f2fc','a59026','42476c','ab79cc','39b9a4','a09f2b','896c23','a8b556','a17fbb','470d48','a2612c','a9eb4d','a041c4','4860be','a81ef7','a07514','3b7782','ae1394','a19f99','76cdab','a25880','3d3a1b','aa181c','a0953d','3c6647','a731e1','a234d7','a127a1','a9968f','a2562f','a853fe','aa63f1','a94ef3','a3cc36','a6fc66','a0728f','a1d486','aaebdf','a4c891','a1ce74','aa7115','a97c52','3fec52','aafc52','a01194','78193e','aa7fa1','a6853f','a326f3','a43297','4ca87d','a7a557','ae1ed3','71bf34','a03600','a43ccc','a7254e','a77e86','c05737','a1e3a0','a8b84c','7c6c76','a32408','a70003','a6d5e8','a3e7bf','a166ef','780cea','a6938a','ac643b','a26adf','ace753','ac9ae8','ae04b2','ad0d7d','a6686c','7c33d3','3007c8','3e5c6b','ab834e','a9cb43','3c425d','497cb3','a569fa','a2dd17','40750a','a27fea','a66a5c','7c3fed','a6670c','a4ae86','a30c12','a1683c','a3ebc5','a14b05','4bcced','3e0a2e','aadb23','a39f0d','3c4336','a1ac70','a4b239','a882dc','a7ea7e','731a62','3d2e1d','80067a','a056c5','c02070','a59930','a8956e','a16b45','4241e9','ac7fb7','7c60c0','a83b67','acc85c','c073a1','adc940','a52acc','c020a9','a96214','7cf895','a105e6','ab4770','a26056','a36b81','781864','4ca1fe','a98bcb','4b2cd3','760981','ab7d9e','a6d1b3','a6c641','a2d031','a4ce50','4c5820','ac3dda','ac27c1','a5ad74','3c4354','a1ec9a','758502','ab5aca','aa3cbd','acb86e','a03454','4b3145','3ff00b','a4b96a','780de2','a939d4','a32b9c','7c5348','acef7c','781167','a84f0e','a97775','4b510a','aa9d60','a40acb','8003b4','e493cb','ab5993','a2c4fc','ae4edd','a07197','a96c72','a1922e','a24ae8','7c19e4','a4de30','a29a2e','a09560','a782c1','a33c24','a506ce','393ab7','a6716e','a5c74a','a4601b','a8ad74','a1cbee','a31ae8','3ff795','ab7758','a56b1c','ac3cdc','781344','aa27b5','7c579e','a7de89','a81296','a6dd7f','ab04f3','a7a1b1','ae0197','acfa52','43ea7c','a01c46','a294d3','a88e14','a5941e','c824a5','7c6ded','a38d8d','a90155','a21199','a5d194','a10d7f','a95584','ac7a89','a79603','a5a260','a73329','404d30','a7d308','4057c3','3a2cbc','c821c4','a6ad56','a54933','a9db5f','a9d270','42444c','a3998c','a2f6a9','c82731','a6620c','a0bf9a','a7d665','accdee','ab3bd1','a56b2c','7c1697','a26462','a1c237','a12122','a06464','a1a978','c064c9','aca287','a49671','7cf86c','a13bc1','a664fe','a0c406','a48f8c','a5dbba','a6c826','a01cc0','a7a568','a7d70e','4400ae','505d78','a95699','a9427e','a5671f','4d2021','acf870','71c008','a797ea','a15374','a14eca','a9db14','a32151','a58a02','a17150','c06c08','4074e9','a1e9f7','ae51e9','aa0c12','a36c44','a425e7','a77571','a6ce0b','a76761','aa9828','a1cda9','a1c796','3d2550','a2b1d3','3d2765','a88e08','ad7344','aca21e','a22da9','a0b68f','a4ae2c','780f56','a1813b','abe637','a6ae50','4849c6','7c6178','a3f753','7101ac','683200','abdba8','ab29a6','a6ad80','3cc2bf','acabcc','a42080','e49470','3d6a7d','a12f27','a021e9','a35a3e','4b0cfc','a664e8','7c6b5a','3dadab','a062d2','ab2651','a725d7','ac5e19','a0c165','3c5432','aaaa8b','ace2d4','4ab061','4ca762','aaf4d6','ac8a06','ac76d4','c8216e','a3f6b2','a140ab','a1e395','a9a266','a388b1','461987','a898c2','a7cdee','a504bd','a22b85','aa8513','a7751a','a02596','406c8e','a2e919','a88ec8','a6f8c9','3d06f8','aa45dc','4785bc','a395d5','c03d34','aadc41','a374cc','3d0fcb','a05671','89901d','ab9092','5083c3','ae4a0f','abf025','a27d92','7c0616','adaacd','a6a266','a3624b','a2b2d1','4406a3','4bcc6e','a9eecf','a55627','a33bc0','5113fd','a3dc75','aa8380','a47604','a032b7','a31b47','a9ee09','a15052','4ca262','a384f6','ad8d1c','adc563','3965b4','a3a86e','a8ea6a','a195f0','abae8f','a186fb','a77b6f','a05201','c00b72','aa1a56','a1d347','a11d24','780e4a','7813f9','a24274','acfe9d','7814fc','a60c8d','a9b227','a01cd4','a35394','a26dda','aabb88','4caabd','a07421','3c4097','a25680','a21fc4','a222ab','7c58a8','a7d736','c02d7d','40755d','a8308b','c04d07','a17d80','acb248','a2de28','a08c0d','ac3f97','a53ff5','a91111','ab2495','a7ee35','a3bca9','c01a61','a196c5','a9c097','ab7c5b','3e1859','a612c1','a55920','4b288c','aa9d2b','4b4e93','7c3381','5113b3','a8dfa2','781062','ad2dff','ad3737','ab3eb3','345341','7c4a00','a4e345','4614ae','a15045','46788d','a3d093','a100fe','a8f935','4407d5','aaa236','ac7f80','506c30','a4d6c7','a1f23c','a80577','0ac195','7808c4','30024a','a923a7','a5b2e4','ab7b5a','a5cc8f','a1412e','4b520d','0092c6','40108e','a30c81','ae4ee6','a59167','adffc8','80078f','3ff23e','ab4392','a3be28','a2eb9e','ac6e5c','ac6d85','a88c0d','a0d93d','ae54db','ae4c1f','aa3060','a11ba7','a0f079','a0aa84','ae206d','ab237e','a4ab11','a0f279','a0a601','adccd7','a662b8','a6e480','a59062','a466d2','ace785','a04718','7c77f4','a5aefe','a383e4','a500e2','ab3d9e','ae2028','0d09ec','a4c0f3','aafbaa','add0f4','a33354','710392','a69bd7','a1a467','a6a5ea','47bf37','06a11d','a28b3a','0ac31c','3ff7ec','a98c5c','4ca97c','a4d118','a32b29','7c7796','8990e0','a39b82','a07c57','a76d90','a1049d','78049d','4074e4','a82286','a4a93d','a4df10','ac26e4','a21f0a','a9b276','abcf58','7c79b5','ae01e3','aa7c64','a43100','a08965','ac2210','ac178e','aabff5','484bb0','a9767b','a0ceed','aa6878','3ff27f','a7a778','a7134c','a29677','a982b1','a4e33f','a07e3e','ab1aeb','adca7a','aa7759','abde34','a67777','3d0f2e','800b79','a1fef3','a8f529','a636b0','a14ef8','ac3fc5','3e711a','aa2e3a','a52f89','a18842','a4bf4a','a65f63','aaeaa9','a5977f','7c39b0','7c49f8','aa0a92','a7ee69','a01bfc','8960b1','acb833','a1b79c','a2cf43','3d20e7','4b24fe','adc6da','a8b6e2','4b06e9','adc526','ac9c0e','a3ef0c','51009b','a10b36','a254dd','a9f0f0','7c5321','abf396','a05d48','a784e8','7c1b0c','a66b52','a2ee48','407605','abce32','aae0d4','aa9f70','ab2ae1','a96477','7103a3','493288','a6fd40','06a0e6','a5504e','aafb18','a7f37b','ae2901','a7076c','aa8fa7','a98166','a23873','a109ef','a3ad99','a838f8','a55985','a1057a','ab99a8','a93d3a','a59397','a1e881','ac4f01','3ff12d','c079d5','a2b7c6','3ff1b8','a2e465','06a122','3d28d8','aba538','4caaa8','460818','780709','3e4a53','a93844','a26063','a5a8d3','a178a5','868dfc','abd429','a9f8e8','a81d7a','adee39','a691cb','adf601','ab0b41','a361ff','ad9d0f','a23993','adcb66','49d38a','ad0698','a6abae','ab652a','0d07ca','a6fe60','a01012','3d1847','a0ae57','e490c8','c03268','a0592e','3d7aa5','add2bc','ab121a','acf8bd','a391b5','4ada6e','a4c337','a8be7a','ad0a00','7c4270','a5aba8','c01fe9','406db4','3ec55c','a58af5','ada06f','aaa5af','a17858','aa24b0','aaa25f','3fee9b','c050d5','ab5e71','a52632','aa2bfa','a69c7f','4240e7','7c000c','a10547','a58177','ab7d00','a95c84','7815aa','040062','501c05','a54601','a617d2','a37211','78139f','aa3fd3','a0123e','c00cec','488c2c','ab6cac','abbb30','a81f3c','406fcb','a9b969','a4e0e0','3d0ddb','a00204','ae04da','400690','3d0ef5','a8023b','4b2517','3ddc6a','39ba67','a12efc','710313','a7ade5','ae036d','a8b6cd','4b5c6d','a837da','a6ae15','a24ce2','acd430','a8b9ad','a950fd','abf4b3','50006d','39cdd1','a7faad','a0371d','4b4cca','a6ec10','4ca58d','4b28a6','acd412','a69c91','780cfc','78048f','a29af9','a1fef6','a06a26','c05e2d','a46186','aaa379','abe5a2','aaee96','a3fbf8','ad363d','aa2701','add191','aa5596','ab3568','aae391','a536d6','a5127f','a0f5d5','a79d6c','7803eb','3462c9','71037f','a091e0','06a062','3ff2d4','abd237','a9a7aa','4b4de7','adf563','0d0a17','3ff849','aaab73','a6c63d','a1da13','a8ed24','a1ef41','a06624','a1085d','a05d47','a4abe6','a23e46','7c6f69','a49594','a9bd2c','a3098d','ae1f96','a59574','a33a56','a02aeb','3461d2','a16f75','a2cf18','aa8e65','a07357','a74635','a1ce0e','780af4','a69abb','ab479e','780dfd','a5991f','a05293','7c81cf','a49dcb','343282','5113df','ab40d1','a8a344','80073e','a57576','503cc9','38835c','a1f2cc','abc60e','a863f1','a79c22','c0823d','a304b6','a2100e','a9dee7','a635fe','a2b8fe','ac9113','a677f9','a8b0bf','48411e','49d390','780e49','8880b3','a7d029','a1c0fb','78085f','a08b5e','a6d91c','a4fe6e','4078dd','c08086','aa6783','ab78b1','a37666','44002d','7c0678','a1424f','a31064','aae2cc','a4a534','a97932','3d2d56','4b43ab','43c8e9','a34358','a1573d','ac576e','a1b6e3','a2af75','a5fb3d','a51f0e','a48e3a','a05b45','ac498c','a05e9c','a0079d','ad87a8','a85506','a1801c','a12a95','a555ff','7c4e41','a61971','a17c50','a37d49','48b2ca','a822da','a67345','a575b3','a190af','a20d6b','4b38a9','acfb7d','86d660','a9dfcf','3d2cf6','c027dc','34230c','ab79c2','86e4fc','ad8b8a','abd8c1','ac9ba5','a7618f','a7093c','a59890','a03b5f','ab8383','a3a2a9','a387a2','a0b656','a9162a','ab1c04','aa49ca','a6d241','a5435c','44066c','a3d05f','a578de','a1fc82','a0b338','a2f694','a6587b','781038','400cbd','ad36a8','ab0727','00b432','a698fa','424554','7c7a4b','a390d3','a21caf','4b3707','a67a86','47c225','89406e','45860d','a65ef0','780fd9','a69d92','a54e16','a9dd54','a80931','aac1e5','a480a5','895057','a3668f','abb6dc','7c7c9d','7800fd','7c61d3','a002c5','406ad1','a1f0af','ac7395','0a4006','7c6ae9','ab3497','a38961','a25f06','86d1f3','a5c173','a600d9','a31709','780ca5','a59ed5','a6fe65','a346fd','a19107','a9bac7','a1bad9','a25555','ac400e','a082d9','a17ac9','ae1458','a6faa1','c00856','aaa61e','407520','a12143','a23ec8','4b0626','a6ce21','71bc15','aaf82d','780647','ae1144','40761a','ae4e9f','a83dc8','781114','a3d7e9','780a98','a6a59d','a55d6a','ae5b6c','a76fdf','a1c39a','aaeefc','a21f25','ade63e','a22124','3d2c99','adff8c','a47d4f','a4071d','a883cc','a01fbb','a4b259','ab083b','a3f4f2','391e59','48b126','ad0ed3','a3e2f6','a8ea3a','a7c53d','896455','4b4ab6','ad4c8f','adffd2','a851ed','a840bb','a4fec7','3d124d','aa8051','7c639e','a8d1ce','aa0a1c','a56fdb','ab09d2','a32576','c004a4','4606d4','a7d7f3','adc867','a4f693','ad79d0','abe475','addea0','abe19d','a80113','a4667d','a0d90c','500022','a98fa6','090001','3d04b3','a33090','a88260','a519a8','a9fef5','3d2caf','a373a1','a66cf0','a3ec09','3ff411','a00825','abedd9','a7286f','a2a270','a171f7','a2f449','48b048','43c6f9','abce31','acb9c6','abc62a','aa3f0a','a22327','c023c8','a0e4b2','a7da68','a6a5ca','780eb1','3ff386','a61d31','abdb97','3e7c38','ac2aca','3e71d4','a3c9e2','ad5b4c','ab8f51','a2759c','4408f3','a4a567','a54db3','a37e71','a4ae34','a59e7e','a4992f','a09a01','aba800','abd096','ad2104','7c35f2','ab93a5','a8b6be','aac8f7','a7f08e','39be91','7c5b71','acea96','ab313c','a27a35','c07a35','4009d2','abafe5','acceb2','a26071','aacbb7','4ca98b','300817','a8d977','a2981e','ab2e0d','a84023','aca36c','a2572b','7c4c68','8013a2','a5726d','a6a2b2','a712eb','ab2ad9','7c122b','a143bc','aaa88f','a9ee24','a19cd9','aa83be','a492ea','a9aeb8','a9da02','ac50fa','a0e9b7','a0af08','ad2dd1','a74410','3990e4','a079bf','a7ac0f','aaee47','a72899','aa7032','7c2bca','867592','4b1f5e','3d117a','a4fbfc','ae11c0','ad49e3','ac4d58','a16703','48ae22','3ff70d','c016be','a8f316','a1e751','3c5827','aa158f','781809','a417bc','abcd51','a4f963','a1b06d','ab654f','a6baca','a8d817','a4421d','4407e8','a4d7e0','a7aa92','aaedb4','a941f3','acca4f','a7d196','a389ca','a9fa55','ac4941','a8b1e1','c81a68','a22de6','aa9826','a2ef7f','781009','aad8c3','394a00','3ff7d0','a516e5','a32052','89907b','406cf2','a82527','a8139f','ae63f8','3d2402','a9d6ac','78040b','a6910d','a4c92d','a939d2','4b3cb8','a5b56c','a8609a','ad3834','a54d4e','a88e3a','3ed442','a8401a','a61024','a23efc','7c4a26','a017c9','aa7319','a60dfa','a20d3b','a54d87','a2a177','a7a451','a75707','ae08be','abe147','a6d3f2','a4132d','a199f4','3d247f','a9ef5c','a1f014','ae00c8','a4df1c','4b1bad','3ff6a6','a4bfc7','a14f8e','ac7ad2','a29322','a08dd0','c063cf','a0e16c','a14b0e','a51f10','a38d77','485d92','a4e20a','acf1da','a539fe','a82643','a00a52','a687e8','a5e709','c021af','a2e7f4','a19b75','a9e3f2','a98a0a','4cafc3','32000d','869254','a0e7ab','3949e6','a5c51d','71bf05','a57161','a163a6','4b066f','7c4a28','ad51e0','ab360b','a8796c','a313da','a05cd7','aae0d0','a81288','a9560f','a4a924','ad0bd9','a709be','342315','3d0c2b','3d124c','a21c61','71bd55','800b2c','a8994e','ab54c6','a30872','ad1136','4bcd05','a62550','a11d18','7c7f8d','adaf12','ac1c4d','7c7512','789215','a3f7de','7c7181','3e5b33','a6a3c1','a4ca9f','780e97','acf045','a7bac9','a08bca','ac2bd5','a3ff50','a85575','a890f3','a0e07f','a6e10b','c07211','a588a9','a57258','ab115d','a2d88a','a2cb13','ad1f48','ae64af','a5d7a2','a51da1','a897cb','aa0d24','a87ef5','ac7f12','4ca98c','4068f7','40797b','a1260b','a8b43e','a1f5ba','ab7e0a','a9aebe','a1959c','a454dd','a583aa','a0d655','e493ba','a37b60','a983b7','86e79a','a5c983','accbbe','a88bee','3c4202','aa68b7','a5e070','aca99c','a7140a','78047c','aad97b','4aaa48','ade58f','a202ab','a1e556','506dc2','aa49c1','a4ca8c','aae2fe','ad2ee2','acf4c7','a88b46','780d33','c061b2','ab6101','ab75db','a552d5','a0bf3f','a39142','ac1c34','a6765f','a68c9a','a1d9e0','a1c869','aacbb5','a95111','a9387f','7c2f11','3935a0','7cf874','ac44f1','ac0c90','ab341c','a6b324','3a164f','a46b6b','a0383c','a75184','a6bc0f','a490b0','4249ad','a60cd2','ae0080','a531a4','a3e5c5','7808e0','a54a32','a168ba','ad3b4b','a716b5','7c4e23','abb1de','a33dd6','c003da','a31d5e','c06d96','c04fbb','a41723','a69e68','a6dea0','a3234f','4d0002','acc327','c063cd','a80c25','a43305','7c77fe','aaa7d0','ae1ec2','a31eb0','a4d3b5','ae54da','a56ff0','407607','a2f9f5','ae2063','a2eaa9','384a3c','ad4405','3fef85','a32038','3d0968','a42b37','3ff25c','acaae3','ad1b0c','ad3fba','a92528','a82ae8','780e99','aa75e3','a54d6a','ae5ad8','506f47','a5d4aa','a491e3','ac8641','ac7c21','aa698f','a9e737','c0257f','a18007','780b67','4b51fa','48dd8d','a7db02','a44eb5','c0013b','406e0f','a9dc8f','ac1d24','c821bb','ab465b','a3e989','accea2','a4e4fd','4b3f92','a36e86','ac2555','a818cb','ac44f3','a06f83','a1903f','a6d70c','a3bf34','40709d','345585','a85b52','a5893e','a121b2','ad0b02','a775bb','a9dc11','add92e','a36b99','a7d040','ae1748','a7a61e','acf76f','7809d6','a13e56','aa8ab9','a47b0b','7c77fc','ab2f83','a6bc25','a60eb5','ac3d4a','3fece0','4ac93a','4045ee','ac7a61','a18340','a0b4f1','ad1514','a26238','aa929c','5113fa','a0c625','aaf04f','ac071f','a91920','5113fc','ae1ecc','a069b7','ab83f0','ab7260','a7300d','a09f95','a8db0e','a637fd','424433','ae4ae5','ad48df','add13d','ab3cd4','a4eaf5','e06589','7c2b08','ac9646','142330','a4fca1','45d4e8','a07d2a','c07720','a53afb','ab1469','a3da32','a428a9','4aaa16','a9dd41','7cf9f4','405200','a71a8a','a06903','a432ff','c009d5','00b0f1','a22644','a3257a','a0ca3b','71bd79','aa0ccf','ab35d8','a5494a','a9a80f','a467c5','3e120d','a40e06','0d0ad1','a79b41','ab6511','acae8e','a13eab','a05939','3ff1f8','40740e','a13704','a07fa5','a9af48','a9516b','a1bed2','ae1e47','a41014','4d02e2','a8df75','7c668f','4b8228','a5f2ae','31d90d','a6c405','780d3e','39958e','a563e7','43eafb','a1d918','aaea83','a00439','a52ae9','a8423f','a31723','aa0d93','7c7acf','a1624e','a31a34','44091f','a6cb2b','800740','a1763b','3ff63b','a2cbde','3ff5f5','a813f3','a01395','7817b7','a72c43','ab2610','a04422','abcc86','3ee123','a7ceb0','7c1cb2','c04a25','a2dd0c','a4dbe3','a98042','a60963','a9afd6','a02162','a2a13d','e47f08','a5951e','a02af9','a02518','a55b69','a6511d','a1c16d','a338d0','a9df74','a097cb','a982b7','ae046d','a217e2','485e49','a3e618','780f06','a98085','a8a2c2','c02222','a11795','ad7a2b','a2eafb','4b0cdc','ae08ed','ac380c','a5733b','ad099b','aa8eed','4b8d41','a1cb1d','424574','a2b1d4','3c458c','ac9c5f','a89597','405c56','a92a84','a33193','e495e7','acee2f','a380f4','ae1251','a2fbf7','a2f086','a9dd65','a61ded','a678d2','a3e657','a3becb','a3bd50','39e93c','a35bbf','ab0d8d','a1df43','a530dd','ae1e6e','adc95d','a6a1c6','a5d73c','a7d645','3984e2','7811a9','a571b1','ad4d5a','aa9ea8','a18ecc','a37e32','4ab548','a86043','896211','a1f794','405f33','a9dd0c','7c331f','aaf0c0','ac8518','a39460','5082e4','a85e74','7c60d8','a4bf70','aa3f33','a6795b','a35ea8','a21d77','a51717','a716fd','a31efd','a30626','a90e04','a28d5a','a41fe7','a08273','a32718','a1b766','781155','abcfa8','780c23','a68981','a025bc','3ff68f','3e63ff','a10b02','a44651','c05f57','a258fc','a0dc6a','aae280','a0675f','8964e9','a521e9','a0cfb6','a64c93','896453','c086bf','aa26c1','a9c2de','a33f78','acaa80','adddc2','a5bf7e','a165ab','acbc1c','a39498','a3285a','354555','abfc92','ad6dd2','a4bbb3','a95531','a7be2f','a69c5e','aadf96','a7c4e9','a02ed0','a84f41','ace39f','7814d3','3d477c','ab6380','a54fee','780e00','a505ac','a3aa5e','a26573','3efc61','ac1db0','300797','a63dc3','a9562e','a3cc6d','4b26bd','a9ed9d','48f020','404fdc','acd837','7c7c11','3c40aa','a51a23','a461b8','7c0582','49d101','aa8f99','a4c7a2','aa67d5','a32b43','ad0898','ab2772','448445','aa495b','a11e05','a9797b','a547e9','7c3f34','a12260','aa6e04','ac0cc9','a490da','ab140a','a9eec1','a58136','ae2908','acf181','a046ee','a6189c','780c69','adfebc','394a13','300401','a9e55b','a712f1','a133d5','ad7f97','a1e5b3','a8da0a','a82abc','ad83aa','a275fe','a47e01','ae5f38','3c42fa','489e4d','3e7841','c2b3c3','ab8bd7','3dadb0','a8103e','a1276f','a69c66','a030d5','adbdc6','a4c357','510112','a08fb2','392083','ab2fc7','a6b377','a37045','a359bb','a98e4d','89644f','899093','780e42','a1db12','a0d9f2','7c0d3e','4d02c3','ad7f0d','a88ce7','ac836e','a67337','a117b9','a9d5f6','a05c3c','ac65df','a1a47c','a4e638','a6231f','a506bc','a2a4bf','78110f','32002e','ac6e5f','ae114a','a5404a','a7275f','a0664b','ae58e5','ae2683','a7011e','c055c6','aa66f7','a7e09b','a12002','7c769b','0d087e','800ba1','ac7573','a56e0c','3d37d8','a6134b','a57572','4ca247','a85f17','a2c189','4b2837','a1bc8b','a18536','a3ff0a','344495','a6aa63','a197fd','a576c4','155bf1','4aaa44','a6a32d','a894d0','396d7c','ad0d35','c01077','ad8c37','ab54fa','a3a474','aa5941','4cc524','a190d1','a75db5','aa5531','3d3460','3d05eb','a99039','a8347f','abcd30','ad4ffc','a25670','a743b6','ae138e','a6ca0d','a54da8','a81d11','a481a3','acd09e','881052','a97c19','406de7','aa7345','ab22a2','a07a11','ac510f','a7e5e2','ac097f','a4dcdc','a0450e','750266','880c47','ac3ff3','aaf96d','a8175c','a3b2c8','a07075','a2ae6a','ad06d3','ac1847','a0eb67','3febfa','a52d5d','71ce39','4ca894','a585fb','a6b3b5','a3c53d','ae04c6','a02c7a','a054ad','aac189','a7a9d4','a9d789','a7c237','3d2665','adc15a','004006','a3952d','a26439','a05b51','aac2e5','ae6497','ad06ff','44ccd8','a06332','a42ba5','8003a8','a23ebb','aaf006','780bc7','3d1c26','a27820','a370b3','a88b44','a7ee6b','a02ff6','a6ebc6','a3a9e2','7cf922','a1f17d','a02088','43ea49','a23002','781858','780d2d','a81f58','a3332e','7c5324','06e00f','780c2f','a66268','ac9374','3efb16','a056f9','a90868','a29eb6','a7403e','710113','3ff266','ada70b','a465d4','780e0d','a9ca82','a17a5f','3ff76a','a332e6','7c29cf','aac45a','a3afe1','3ddc74','acdb6f','a607e1','7c79b8','a7247c','a5c7b8','a5fc42','7c5cda','adc5e2','780a5b','48469c','a64d9f','a1cb75','a4e3f8','ab4ea9','a20dd5','407388','ac2575','a92631','a040fd','aa735a','abf07a','c04e84','a9b1ee','a15ba8','3914bb','a8fa21','a5376b','a008c9','a4d8ee','a3918f','a9f332','a4e6e6','7bd005','7c3d57','a6b790','a60e37','4aa5b9','a6717b','7806f7','4d219e','a28d75','ad9130','a94661','485a55','478fa4','0d0c71','a34a94','ad9792','a54d69','a5cfba','a03224','e20057','a848c0','a13767','3e692c','46a098','a73dfc','a7a84a','040078','48f9d2','ab0cab','4ca251','a5c95b','0101f3','a0a1ee','a11f3d','a9ea32','a1f216','a1a9ff','40428b','a4095f','aa3226','3e22f0','aa5cf8','a91153','abc592','a9e9b2','a6e4d0','896119','a9a868','a28786','3f057c','a34de6','46b8af','a149e7','a829e6','c02d3f','acee71','a3d066','7105ab','a16ed2','8964f0','06a1c6','a499d2','a40b10','a647e8','ad291e','a82688','4b0f72','f21127','a8bf0f','a5a21c','c07c55','7c62aa','a9ba31','a6ba79','3ff449','c02ee7','aae0fe','e2005d','7c779a','a819dd','a66ad7','a72c32','a7f6f7','aac721','aa2ada','a65650','7807c9','ae54f7','a9d159','7c0bd0','ada908','abb179','3ff3b5','3ff5ab','a9026b','394a12','a74efc','a666c6','345398','4842c2','4b50d2','800877','86837c','7c7b89','aa344a','733072','3c434a','ae1123','ab8d84','ad4040','ac971f','ab69e1','a44f67','a513ac','3d0bce','343603','0d035f','a34b13','adcef3','ac6f74','a91225','ad5c68','c0808d','a578e0','a69e45','407907','a6b6a6','a99a1b','a4a991','a078e3','867f60','a6a445','7c35c6','a9bec0','a3e895','780e89','a9ee0a','5113b6','a712b3','a72736','a297f8','a913b1','a43818','3c422f','ad6e86','acb92e','abded2','22ff33','a6fead','a0b2d2','ab2c11','a5e728','a45587','a2237d','4aab10','a70a8b','aa47cb','3dd796','781698','ac79e8','a24b17','a89fcf','34524e','ac712d','394830','a0e2f6','c02449','a9d4da','4ba890','a2ef4b','c0175a','504e43','ad1175','3ff81e','a4e670','a4dbb8','e09651','a11031','ac1176','a32798','a0d33c','aa2e0b','ab40f8','abf53a','a242a4','4401a3','896116','a69139','4b444a','aad21c','a7ef25','abf2c6','ac7f27','a4a918','a9a050','a284b5','a0d30c','4b43d7','a6a5de','ab1100','a3ffab','3d2287','a78dd7','a6a705','406d26','a1eb41','a9ee47','aa5d60','a2f452','a023ee','3d349e','ab5e76','ab20bf','a129ae','a150d4','aa9d46','44d88d','3d4a79','ab7ca0','aa4618','c06c8c','a1c9a6','7c6714','a3f885','a9af67','a41f19','780fb7','ab5cd1','78b45a','ad9e42','a23af3','a8dcb2','ade73f','ab6704','a114bb','a987ba','e48e98','a5521b','a535c2','a3fb94','c03624','abd474','74358b','a23b0f','adeca5','398675','ae4be1','a918b6','06a1c7','ac7e56','e4818b','a7014b','c063c8','a291f4','3c1c82','7c3439','a89bd9','a36049','aac9c4','7c14fc','780083','a289f4','4b264a','a66e9a','a190fc','a3f119','a13f2c','8005f0','aab485','a2f2d9','a992f0','a06d88','a0676e','a1fb9b','4070e6','a830db','ab42d3','440870','c05ed8','a859c2','a1cc7f','a9c1a4','ada628','a0fc3f','a7cfcc','a7d6c2','abf59b','a73c19','a25631','a1f4c5','a8a26e','ae5963','a6a757','a10d68','ae04a7','8962e5','adfe4d','ab098a','781795','a4b7d2','a6becc','406b05','a8c815','c069a9','a27c59','390d25','a93152','3ff2e8','a3f436','a48d94','abf0c8','a8a91c','a9d5be','a75a24','a55d52','aaf621','a1c486','a07c56','a6ea3e','a4fae1','a8017b','a02cbb','ac6e68','a22f1e','424778','a03e33','a5b4c0','aa3db4','a60b23','a382e0','a9bd5e','ad07ae','ae1ed8','ae0a75','a3095b','4407ca','a02e09','78119e','a92053','a6880d','a23eda','a42afe','a0b892','a9ea43','a7330c','a33d8c','7c753b','a22966','a642ed','a00956','a1fac1','ad0e0d','a6f63b','aa7d2b','c2b553','a315e9','4408ca','ab7d44','7c6ddd','c8280f','a5ec84','840fb1','a53cec','a1b37e','7c1663','a145ec','a1bb3f','ac972c','a69d3c','a963fc','a1d825','a5ce0f','aa08a9','a1cac8','adfed1','3d2fb0','a50b0e','a24267','a4d8ec','a15bfa','4b089d','a57337','a50f2d','a456da','3d204c','398186','aadff1','aa353f','ae2029','c04930','a87b3e','4ab549','3f03fb','a396a8','a59d90','accfac','a4ef3d','7c32db','c057ae','4952ae','aa5963','e493bd','a66e43','ac9b79','780af3','a40a28','4ac92c','a45bdd','ac4597','ac5e02','a54d0a','770591','4b3f75','7c1522','3d6008','ac9e75','73945b','a50f57','800b45','ade862','a2f351','a4f021','a0c7fb','a4ca7d','e48a49','a486c4','a3eb0f','48a231','aae61c','a7ec59','aa0c5b','acb826','a66069','a769f3','c81c31','a4177f','a3f02e','8a0104','a95d42','a10308','a52fee','a64e0e','addbb3','a08f4b','4c4849','a0a2cf','ac899c','a5f1c5','3ff0a7','ae1254','a5ba26','780c9e','a439e6','a2bc2d','ac06f0','a9bd4f','a24ef7','a8a32a','a96705','a11e38','a03329','a6f831','c04b92','adab66','a16b66','ac5fd7','a6f80f','a3bcb2','4d02c9','ab0299','a7d8e1','407759','a7622d','a0d597','a9bcf9','a20d66','ad1a02','a085e5','a4bcac','a0504b','a7d1ea','3fee20','a5e6c9','ab7647','a1e7dd','ab7ece','ada01c','a23047','ad4e3f','ace5b7','ace4e4','a65006','a1e8b2','c02ebd','a6871f','a1d636','a77da4','acd40d','a77104','c05af1','4018bb','a7132f','a2f230','a04b8f','a72eda','aae1ea','c063ca','aa9169','a3b74a','c019e7','a3e8c2','a2dc13','aa5abc','7cf85c','a66528','a67389','a8f172','ac6d0d','ac4890','a0e5df','a20519','a153b4','ab7e9d','3fee5d','7810b8','c0309b','440084','a6b8b5','a8c971','ac494e','a34b0a','a0506f','a2cf07','3a344f','3d39b6','a8a07a','ab7a0d','a3ab7e','c827a0','acd028','a01cec','780f6c','ac9256','a61919','a9da1a','7c0da9','a65e4e','a8b5a8','781029','a5f90a','ad263e','a73c8c','a88603','a01c1c','acdf9a','31ff27','a70432','a67ded','7c5346','a17eaa','a7fb8d','a6c8cd','a0261f','a63098','a97c83','adfee0','7c79b1','a1e2cf','adbefc','a5e6f5','a593e4','a1165d','a9df1b','a2983c','a5faa6','a2ddfa','39ada0','aa0ce4','a32c19','3d1662','aad73d','a8863c','a7b177','7805b1','15cbba','a195b7','a1d14a','a7fb00','896581','ae5719','30073b','a8ef77','a68382','a7ff34','780839','a73c0c','30074f','ab3b5f','a78557','a4fc36','a326e2','8418cc','a376cc','a1a615','ad9f7f','ab7c21','a69c9d','ad337b','a6fd53','3cf476','a6640c','78030e','3d2a8e','a7ec39','4b418b','ab4718','a0c982','a65ca8','3d3aae','ad2b5f','4b0618','c04631','a676dd','4ada0c','aaeaba','add2d0','abe412','a05aaf','48fc17','3ff2d9','a1e83b','a7e1c6','a4e078','7c5343','a4b07c','a5d902','a5e96a','8990f1','a5522b','a320ee','ab0efa','abc1ef','c06cc2','ab5125','a05879','a4dcd3','a499dc','a497f2','a3bd78','e4961d','a39eac','a554dc','acb111','ae27f8','abf079','404de6','e48d43','4481bf','a28878','a70c08','a8f547','a4ebfa','87cc0a','38ebdc','a6ea9a','e8040e','a6a977','320038','478708','a83a66','a57142','a8ea1c','a02367','a8e5d0','ac5763','a94ddc','89901a','43c171','adb906','a66835','39c9e7','ab6225','a20c44','a5a84e','a1a447','a47182','a26008','abc55b','a1fb50','a7742a','440641','3ff657','acae10','ac90a0','a1c32e','a00244','a9b62c','a56e8d','ad91e4','a47b7c','780095','3d1b27','a8a602','a5f6cc','484174','adeaf6','a9ac0e','a6bf7e','ab959e','ab2764','a8f682','a5724f','a44663','a1b725','50018d','ac467c','a86ba6','ad5b14','c03a0b','a897d4','a0601d','a3e542','e49298','ae5693','a69ca9','a3d867','400e77','a8fb06','a5fa63','a53d43','a0c998','7c11b9','a52164','ab7ec8','a08812','ac82ea','862d50','a50680','c04b7a','7c4321','ad556d','a4a8af','c017fa','a81173','a1bce4','a54d04','a8bb0e','a360d2','a066f8','aa79f9','780bf9','a65f48','a59025','a7c42b','403fea','ab10a8','ae2edf','abffca','a0b0a7','a20173','aa915f','a6378b','aae81d','aa405d','a20120','a1bc3b','a09aa8','a9d62d','a16707','a14425','3cee5d','abacc4','a694c4','a74d1c','acbf57','c082f2','48ca40','ab574d','a7b7dc','a832d4','a65095','a02f2d','a08c90','899064','7c534a','a83bc6','ab5b66','a401f3','a98397','c05faa','a89543','a714b6','a56c8a','a1c29a','a7133e','a9db1f','a6acf6','a5e759','ac44a6','aa8c76','a1e068','a05688','49c1f5','c08883','ad0fcf','ab5a2a','3c5424','a52b55','a2fcb2','a1b9ce','abb507','a66ddc','aa2f28','ad16f3','a947af','a49750','a225a7','a1a62a','aa4bcb','a1eb7a','3ff7d4','a393e3','ace5b3','a1be65','ab2325','a86d51','a698d4','ab3632','a04576','42442d','4609e0','a4cdc3','750259','a23359','40731c','ae20c5','a9073c','780cfb','7c404c','3935a4','3ff556','a9b78e','a40682','a07361','ad1fd8','a85b07','a6be4c','800bbb','a31961','a12b1b','ad4835','780ee6','a31448','780022','06a136','ae5d59','ac1099','3ff27c','a61649','a0d1f3','ac6d73','a7cdc0','a6ba69','399074','a224ef','acb5ac','ab20a6','a602e6','aa4414','a757b3','a6cecc','ad690d','ac7225','3d5ec6','a81ab4','a54dfa','76ccc9','7c79ba','e48fa0','39982e','ace212','a7f02d','a25e73','ad0e22','a13359','a0f869','a97a67','a19c4b','a2ccef','ad6553','a32456','a248e0','a8493d','a31be9','c2b3b9','ad99fc','71c081','ad26c8','ab0091','a1e856','407371','a73e16','a6d6d9','aaffd1','a51a24','aadfc8','a45cff','a39c1e','7c7ba1','a8865e','a69041','505de3','ab2d35','a3ca0a','a1826c','a1c7e8','a193df','a0b66b','a19fbe','7813ad','43c1c6','a05440','48d90c','ab8f12','4070f4','780320','a12a83','a0fdd3','e48855','a2f9b7','a6b301','a81141','a20a4a','a689f0','7c7ecd','a69711','7807b4','a33605','33ffae','ac9f51','440713','aba7de','ae4d69','ad3178','ab5156','7c3fe7','7c7c2d','a4f2f6','a60a59','a2431f','780e6c','a7b7fb','3c60ee','3c5433','a819b6','4b27c0','ac9ad8','a8755f','3d1171','a79c1b','ad58fc','acf841','a27816','ac8fb2','a5da44','510114','800b7c','a026a5','a3e9cc','a96e6f','ab4add','aa34e7','ac9c1d','a22233','aa18fd','a3fea9','485789','a9ec98','3ff40e','ad8f74','a7ff6a','a1c782','7c4248','a7746b','a559b7','a55c67','a352c9','4ca7d8','ab58c1','39d08b','ae5f12','acf149','a2f98e','4ca86e','a1c273','a515f7','ad3da8','3d19d5','ac8a0a','ac8b2c','ada0da','43c8df','a80120','ae5f9f','a6a5c3','aae5ce','a141d4','ad5a5c','a735c9','a0ca8a','a800d0','a7fff9','a83d1a','a570b0','a8ad11','a9d796','a6373f','aa34e8','85d7c0','a5f047','aa9ec5','a7d876','7c7533','adb5cc','ace60a','a3581f','a06bfa','ada138','a6a627','a19c58','a167b5','acc20b','a663d2','399582','a79433','a40658','ab0361','3ff0b6','488183','43eae0','a87039','a78ad9','a5b859','aa8214','a5760c','a90a5f','a3e3ac','a3235b','781237','a34169','a821e4','a8d9ee','a9f0f2','a34b5b','89408c','a24014','a88d3c','a60a0b','a320fc','a66bbd','a5b764','a3ec46','a255e2','39ac43','a8f848','a5ea14','406df5','a65f8b','a06ec9','4ab586','475001','a9c0e1','a08eb7','c811de','a3b147','a2274d','47c4b5','aaaf8a','880848','ac6ca9','43e83d','ab2ad8','a8b1c0','ae1ffe','ab742b','aac708','3d0ede','a97628','300796','ab68eb','aaee0f','ab17e1','a76d5a','a356c1','a224fa','a37a4f','7cf9d3','aa8510','a45f45','abda04','a7b4c7','c8277b','acd318','a72808','a52617','a4bf7f','a66090','a1709e','a8da40','a3696e','a5bb14','a6e458','a7d9a3','a31122','78097f','ab0d47','a5c68c','a000f7','a6aff4','a897e5','a85c8d','a80c74','4007f4','aa58ea','a68344','7811de','a34046','a2d20e','42417f','a7d607','ad5a8a','ae1e00','4c49e1','424266','ac1599','a92c2e','a44360','a7d676','780ec8','c81d93','aa11ba','acff34','a000aa','4b521e','a27923','7c4347','a6f845','7810f0','0085a0','ac3d60','440144','ac9197','adf4c6','a8fadf','a7e944','460944','a1d433','a16dd7','a1f335','a68014','a0c875','a6267d','aa842f','a03933','43e907','7c0a92','a1a545','a84c86','a7b08d','a2919a','ab7203','781159','a3f8ac','a1f526','780adc','acdef1','a94805','a000fc','aa9bb6','a2ff65','a0257d','a0d724','3ff84b','a9c6bb','392d43','4073e4','3e6bce','3e5ec0','a696df','a8baa1','4bc9ab','ada257','a6308b','a9f2c6','485355','a3619a','4849c8','a112b6','0101c9','401153','c0526c','a70404','a98098','a86383','7c2981','a4dbaa','a2c211','a06a27','a23ea5','a2d699','ada21a','a9d107','ab7074','a56ae6','781018','780bab','a67934','a7d545','a7fa26','a3d01f','502d21','c06e28','a6d833','a1bfeb','a0017d','a1e99f','a948f6','a5fbce','a9410c','a01b9e','a44d62','acc857','a1b2d3','a10edc','a4fe83','a9502e','ad4db3','aa178f','aa70a9','a9b04f','4b2534','aa67c6','48d247','a4e34f','a35877','a4345e','a4a856','3d7263','ae0665','a1cdd4','a7d5e8','a2e58b','a7987c','a64928','a21e9d','a80e52','aa216c','7c0ab0','78920d','a2ceb1','a05ff8','a69ca0','a3e145','861fa4','780be9','a3f65c','a34e63','a645ff','a0b20e','3fec21','a857f2','a013c6','7bc009','a7d7d6','a1d3e9','a03ee1','a0a798','a4b0a9','adfdc3','ac9d1f','4016dd','7c175a','a10441','86cee7','a50ab5','a133ea','a2a536','780969','aa2547','a7c587','a57b66','adf160','3460d4','4071d1','addb15','a3c454','ac6a98','47a20d','c05dab','c8277e','ace6bf','aa884b','3d22ee','a63939','a68dd5','a724a9','a967c9','a21f57','a355af','abb155','aa670c','3ce629','aac087','ab1159','acec7a','aa8aa9','abc4b7','acf8b0','aad4b8','3ff7d9','a814f1','a913c4','a4c785','440256','a3c153','a17f8b','a5cb05','a575dc','a22936','a9e558','ae4ee7','ac99c8','7c6d53','a52461','7804f3','780a2a','aa0e1e','aae72a','a6e4c2','e483ba','a2c346','a3f79c','15407c','acb0c9','a11136','a7aea9','a375ca','4d20cb','43c919','0d049d','a6efa5','a0ea9a','7815d0','abe030','3fea63','3ff800','a23759','a4eff1','3fea5f','aac9a6','a3426d','aba0d3','a0e310','3d20a9','a9e5c3','ad3777','a68849','a96166','a5708c','a29492','3d3610','c05231','a86665','300735','a6f0c1','a3e5b5','7c1c68','a96172','80139b','ae138b','a812e9','a9ea06','407612','a43511','a5b876','a00f8c','ac6ced','0ac995','a0ae0e','a691b6','ad422c','a2f5ad','aa198b','a72453','780e51','aa06bb','a33ad3','a4dad2','a58869','aca7f5','a213ac','7c1c6a','a373a2','7cf9f8','a15c54','42423d','a8089a','a2834c','a83b9a','a4b4c5','a548ca','40453b','7c19c4','a84a29','a6b554','a42c83','a7ac18','a32081','a456e4','a7865a','ac2a05','ae1eb5','a17f70','ad050f','adddd5','a45a2d','ac783c','ae002e','ad35bc','a814eb','ae60f4','a5f763','a2a40a','4ca702','a18fe3','ab2ac4','a3991a','780667','a02ae9','ac43de','a8ffa1','a28dc8','a7ed08','abc66e','a9d320','a6ae9a','a10181','a22171','ad6b83','a0678e','89649f','a959c4','34634b','a7f609','4b1ba0','abc04e','4ca647','a05e50','a0019f','abf308','3ff29f','a1ce16','48558c','acb5a9','a9b2b5','84cb66','acae6f','ad23f3','ace3c6','a73ceb','3eec99','a33db4','aae9af','a0435f','86d66d','a31730','ab6624','503d1a','a9eec7','c0816e','ab096e','a126bb','ae4ed8','acb741','a8efba','a6f95e','a34f2a','0c4156','a65e2a','acb223','c03f67','a335dc','a431ad','abb926','a7c14d','c0806d','a42fe1','a9d836','a2d70b','3ff2c6','adc9e1','a6901a','3e5d62','a1279f','aa6a02','a10e50','4b4bcb','a0312d','a66ba7','a06253','7c6695','06a1a3','a8b6bd','38739b','a015f1','ac348f','a0b35c','4b17dd','344612','0d0c0f','aa64ca','78170e','a74989','4b401f','a09cbf','a05b7b','06a144','a0d124','a014b2','a8759a','a3f986','ace5cc','7810a7','86787c','a30d0f','adb5e2','a1212d','a61066','4b436c','a996a4','a88a7d','a1139f','adf447','a5b35d','a6f0fb','a4bf7a','a99f92','a04e6a','47807e','4b283e','ae1169','a1ebb7','ac530d','c03121','a669f2','a4ffca','aaf356','adcce3','a9879d','a06db4','a35ebd','a9b1a2','c04ba4','342317','780eef','aa209d','a66292','74f95f','ad6f15','aaea15','51301a','ab3e38','aba89e','49c1e9','a125ea','4ca5b9','a9bcbc','a90a19','ad5c7f','ab8019','7103a8','a2c5b1','a657aa','a385a2','a21f18','e08091','aa5bcb','acaafb','3d3708','a1fe89','ac2579','424936','3d2efb','7585a2','a66822','ab0ab5','a9efb1','ab71ba','a97d85','a234c0','a21127','a4d361','acced9','0d0868','c02129','ac72af','a2b2da','7c6b82','896469','a67928','a0aaee','ad78d9','aca498','a1f10d','a3f1e8','aa3d03','00ac95','a4ff4f','aa752d','aa5a3b','a7b315','49c105','0a07a7','a4328d','a57047','a28099','300750','a315f9','aa6c6a','a40c4e','acc647','aa71c5','a9b640','a62f85','a8ae35','a50070','ab8b62','a8f97c','a153f7','3ff06f','39a2a4','a25fc0','780771','a17e3a','a57c8e','a7d1a0','a0224c','a53e8e','a1175a','43bf85','3d323b','a1419e','c8238c','7000c7','a171f1','a685fc','a3d39e','3d352f','a932df','aa19b7','8880a5','ad4395','a8aa84','a8dccc','aaf3b7','a92d4c','3e060a','adfe97','a9b045','461f4f','ad861a','ac117b','aa6844','3d2044','a886e0','71bf88','aa3d2a','a982a9','a6228c','ac47c1','a35d48','a980ba','7c308f','a81b8d','aa1c6f','a6628b','32003d','ad7c57','a7bc8e','ac96fc','adda24','a1af06','400f86','a3ef3e','780026','a1f649','a6b86e','a17b94','ae2f62','ad3eb4','a3e337','7c3349','ad9515','780c11','a7d9db','424615','ab2056','a6f31d','aa8604','a17bba','4b191d','3d374e','aa2374','a77c89','a9d70d','ad074a','aafc0c','aa8e7f','a31f6e','a5c779','78100a','a10da4','a83c45','acbdc8','7817e1','acd8ac','ac2d52','a7599a','a16fe3','ad9c8f','ad1f3c','a002a8','3ff79d','a36411','a46f4c','a994be','aa8bae','a65b97','7c0d89','a4f65a','a2dcb9','a3621a','c038ac','a20d0d','71c088','7c74a0','a1262e','a69bc3','a2136e','39c9ae','a91e68','3d6a3e','a2096c','e4903f','a16113','a1d340','a62b5e','a606b6','a4ea7c','a3971c','0d09e3','7c3635','789208','780ba9','ad7f99','a7f527','a38970','a0a378','a3a878','a41fc1','a6ac5d','ae27fe','a6f12d','a88da9','44ce85','ab1fea','7c19c1','4078f8','a5121c','a170b6','ae54f4','a49610','aac271','a2fa59','8007f5','adee38','a02656','ab20a3','a03888','a20293','a72016','ac4aa1','ad64a9','a2b4e8','a1f237','a32940','a65487','a1cead','a68077','39a1ee','89648a','acad4b','a1df63','a4fd6f','a19c0b','c00378','a63c37','4b25c8','a33646','406f72','40738d','a7e453','a862e8','a952ea','a4f783','a30f15','a609bf','ac9710','a34c8e','a3bc7c','a1bcd4','a3a1e7','3d442c','a8fee1','a3a7b4','a09e2b','a0379b','4b82c4','300812','a04dc2','7c28ac','4b4005','a229b3','3d7f74','ac2b75','ab1192','a7c74a','a16234','a090e7','ab915b','a6178b','c8195d','ab2148','ab8902','79a046','a1416c','a5d2bb','a1494e','aa47da','a3d874','0d0019','aaf3cf','a54f11','a2179e','a1292a','3e6fdc','addd7a','a7ec89','a7c769','ad1850','a9d3bf','a80b39','a1f0b1','ac6679','a44e7c','a04aaa','ab4259','a66fd9','899022','300795','7c06a8','abff3c','ad4ac0','a73d50','4b8687','a02dd0','a53ad7','a0b676','a483e1','ab416c','aad5d8','896c22','a24915','a34c34','ab2398','a7fdf1','a19360','ac2f17','a36fe7','3ff19d','a25661','ae4eae','aa85d9','a8fda9','a6456b','a6bb31','ac983c','a9ef24','a0bef0','adfffa','a30a7b','ae4ed3','a36659','aaa449','a5ef5a','a9a767','a51633','aab168','a80860','a2b87f','a0ad16','a0dbde','a051ac','7c14df','a3276a','a55fbc','ade106','a07ccb','a955e2','a4df74','a3a263','7c6ce6','ae596e','ab1deb','490d0a','7c3add','a29d25','a0347d','a1a534','acc37d','a16862','4ae354','7c38d6','a74cc2','4ba674','3ff306','a2f43a','ab4531','add36d','a96f22','a95380','a813c8','a12114','a9336d','390a04','4ab28b','a54db1','aaecb9','a66a4c','3d2d09','e490a5','7cfa08','a33870','a184a8','a9ebe8','a5bef7','ab8272','a0273a','ad3ec3','a8f149','ae4a29','a3c67e','a89861','a341dc','3d361f','342250','addfed','a72221','896eee','a4de08','a18ea9','ac9c6d','aa8499','344081','a6a29f','aa7abd','a4e0df','a49c7c','a4529d','a148e6','e48eca','a4518b','899105','a707ed','8964eb','a2132b','a5f894','780a35','a36d0c','a60e92','a6b173','acdd1b','a9cb18','a73e7a','a3547a','a29752','a7522e','a088bb','a72aa3','a06e4a','a957c5','a47a19','c03b57','c07c71','abcedd','a7ed48','a65d2f','a524d8','a0a610','3d1736','3cf3a9','3fee64','a0fca8','a8bbb7','781602','a87b47','a4c958','a47ac0','a45338','a1c879','4bd036','a81833','a638f5','4b50ab','4b1f2f','a9e5ff','a6ea20','a01d97','a68877','a23e44','a66d8f','7c4493','a7da0a','adb658','ac6791','a6ffa7','a154f0','a3a78f','a91331','a8a380','aa72fd','a9b837','8403c6','ad96ce','a436da','a4da03','a17dee','ae2062','ad38ad','e8cfc8','c059d2','a365f8','a1c852','ada81f','ad449b','a44d74','a3807b','a283af','a53187','aca738','ad9efd','ac7dfe','a5d6db','c80379','a34ace','aa3764','a9f0fd','a2b469','a4c9f3','a588c1','a1732b','a3ff4d','aa8f28','a19386','3d26e1','780a6f','aaee27','a17bc3','a0b876','a47fde','45ce4b','aa55b5','394f32','a18a03','4b8225','14fa0e','ab0a07','a09e70','47a9a9','a117e9','addba5','ac5bb9','a8dbfb','7c4e42','a38288','a1a1f4','a59d63','4bbc73','a6fd77','3ff246','aa9aed','3ff714','ae54ef','7c7af8','a906f3','a7263d','a635d4','a8db46','a6b3fd','ad4076','a9e8e3','896115','a1ac11','a6dc36','7c7c7a','48e804','a432cf','7c19c9','aa5b05','a4dfd8','adea97','ad9dc2','a974c9','a02000','a3def3','a1b11c','aac403','3ff757','4ca766','ad7153','a22d28','c07446','ab71e4','a988ef','781407','abe6f4','3d1cf0','a0dde6','a0069d','a4ad42','7c8197','a047f1','0d0b20','4ca9bb','3965ab','84b7df','a199dd','ab394e','a602a0','a3244a','abf086','adc7be','a69c49','4243a4','a4630f','4b829a','a67046','aad99e','a81c69','a06abb','a40bf5','780d10','3df79d','a55ef3','a1f940','3e1bcb','ab3d46','acae88','ac9b82','a52d3a','44a00f','a1665c','a12c23','7c110f','ab13ff','a1cfec','3d115b','a27f97','a36d3d','49d3f9','3d184e','ad3570','a405e8','a43ce3','ae47f3','484219','abed10','a58ce9','acf1f6','a96877','4aaa4b','a9d932','a3c900','3de520','a41f03','a7de3c','a7d2c4','acff97','ab7aa2','780f12','ab2139','a9e134','a34d10','a0b7f4','a720ff','a2607d','ae690d','a72832','a6a41f','a6b730','467895','3d3605','a67bd0','a07ed9','a63b96','503d1b','89650a','ac6b74','ac50bc','ab5875','a82ac4','a875a7','a0383b','a5f07b','7c37ed','3d0628','a9ebd8','e80450','7c6ca2','aad485','a4c45a','ae4a27','a1d63f','a527f9','a0a530','a479d2','a339fb','abc6ce','a6ef40','a326f6','7c7a44','a76e37','a208ce','ab1c42','a9d2d5','a96844','89644d','3c4329','a19969','acdae0','ab19d9','7c12b8','a3ba94','a86520','a1d9da','abef2f','71c245','a099f0','ab37b7','a71209','a70b36','a1d868','aaba64','aa67b5','a0e62c','c01049','885967','38913b','aaef10','407937','a4a211','ae2ef0','89652e','a81b45','e494a2','0d08d1','4b255f','a7c6f7','a6657e','3e7182','30080e','a69f22','a9aeb3','a4a231','ace3e3','a55d72','a05534','a58912','a95234','a552f8','a8058a','a3ff61','3d0640','39802b','ab42e6','aae2fc','a2f41d','aa3e5a','3d1049','4d02b9','a81f64','a87dec','a7ad84','38e9bc','c03f0c','a9ce21','abf73a','a7acd6','a42390','a03aa0','3e48e5','a37b75','a27189','a1f9d8','ab85f3','a990b6','a67040','ae4ee4','781307','7c6aee','3eeaa6','a8a5bf','a4e677','3d1704','a61988','894f72','8960d0','a6eb93','a3f48a','a066a8','aa2601','a3a980','aa8fdc','ad4c2a','ac0418','c05c22','a72816','a243d7','c01081','a9536a','4b8299','ac550d','ab839c','a11daf','abdb2b','a9ea2f','aabe53','a03240','aa50c9','a56a10','a40b26','491221','ae2045','c0cb64','a727f9','ab6a2e','a332fc','a78eaf','a7226b','a4b82b','a548fa','a61bb1','a23797','a58213','ad7de1','ad1db4','a10e6c','ab3a07','a68717','a4782a','ac7034','ace8c9','abe555','aae95d','78114f','ae123a','a5983c','8964fd','736834','a68f72','a7a46d','a49e0f','a12d51','3cf571','a6ab73','a14f6c','ab4382','a6acf7','a2cb33','a24b56','a4bade','aad242','390db0','a3a870','a7e68e','ada25d','3e0610','a0b3fa','a34e7d','a512f1','a64ab0','a7279d','ab34ee','a3d150','a5b166','4b088c','40408d','aaec4e','a6a966','a4e6bf','a33a40','a0ab51','a793c9','a8c5f6','70c078','a1aab8','a05819','ad5cd9','3e7943','a85052','acd03a','a93adb','a33df9','4cab37','3fea0b','3e67fd','a9c290','a32160','888176','3d99c1','a56d52','a34626','a2d567','440001','a89185','7c80bf','49d23d','a9a156','a8a49b','4b2c7a','ae4a16','abb426','a97b92','a144cd','a105ff','a33734','a3f7a0','a0a6e5','a1eb92','a68da5','894f56','a85024','a109b6','a49bdc','acfc74','ac85ab','a336e4','a230b1','ab76eb','ab42f5','a885e5','a123fa','a7cbdf','710310','ad1095','71030b','392eaa','a3e4fa','a106b4','a8a1e5','a0f46f','a1cccb','0d060f','c04fd7','acbe62','c077b1','a21f58','45ce56','485ac8','a9c5a3','ac0940','a68db0','7c4e90','c0767d','a003f3','aba789','ac9837','a2eca0','a35a62','acb524','7c47ce','4b437a','c078da','a0c980','4952a9','c01e47','50843a','a9bcdf','49d2a8','3007b0','a8cf2d','a0a9f6','4d221a','393cc0','a8be49','a66e03','acad76','7c48a4','a6ab26','780263','4d2102','a572ea','49d1c6','ab6bc6','406dea','a7b4d6','a30657','4007ee','a9a650','3d0fcc','a89e2b','a15bc3','a34a16','a423b7','a68bdb','ab536e','a21cb3','4b368d','00dd58','adee1c','780476','0101db','a4b8bf','ad0bf4','780a19','a9c2b3','aa54f8','43e9af','a92ab5','a6d932','3df79c','a8068f','a62d04','c05ea2','a4f601','3d08d9','a57e0e','a70a84','31de8a','a975d3','a5f7e9','a2879b','ae06e1','ae6018','a41f72','a0ab14','4b8664','155bc4','a7a233','a2e99e','a1a094','7c7d1a','a6f1f9','7810d3','c04e9d','a1b12a','a65a03','aae215','a70046','a7ee2a','3ecef1','adb903','3d25a2','3d0fca','ab124f','a730ec','88813c','a4b1e8','a61752','406b3d','a11a27','a9b1bd','a24e1e','accd21','a364cc','a618b6','39b119','acbeac','a4b64e','a36c64','78118c','a139a3','3fedbe','a1be82','acd3e2','0ac9be','a56d59','a20e37','a0697d','a04983','a3eec0','5113f9','ac8514','a6a435','a4c897','a4bd31','8963d3','a00ba2','3c07fc','aaa20a','aa4019','a2cbda','a66959','781571','a256a0','a9ce44','a7d061','a174a3','a05eb4','3d3742','ab0df5','a3677f','c827ac','a16d80','abb2bf','3ff203','aa2a7a','484646','3e4896','7c62e7','45ce02','a30849','a94626','ae03c4','a6386b','402c83','a06091','aa829f','a54c0c','4401ba','aa9318','a340fc','a02d4c','a7b79e','ab0dc2','a721ef','a1a376','780a74','a6e216','a24714','71f882','a420af','adb197','a053a8','a68fb9','a5042e','a426b2','aa4093','a5bcc4','aae448','458e96','a4e071','c073a9','acda99','ae68af','acc02b','a4d33f','addeb0','4b2c00','a1bddf','4ae28f','c032a2','ac15b0','a8c31b','a615d4','7c42e5','780aff','a1c893','adf596','a15e7d','a56079','a7881f','a134de','e94b40','a1e24a','a344ab','a0ea44','aab300','a47fe7','a147b5','485341','ad1cfe','a48004','abf5f1','3ff6b3','a172d9','a955dc','a0f01f','a806f5','a0b82a','a6b4bc','a798c6','a6d781','780e22','a966b2','39031b','ad8e23','505e70','a6740f','447c8e','a06266','0ac0eb','c99999','a2e70b','3d3a54','a87be0','a4e48c','a69396','a28302','a65539','896454','3d129f','86eb72','aad375','ae1e90','a8a080','3cd7e5','c050b2','a23b36','a078be','ab486e','a8aa24','0d0a56','a3361d','ada56b','a32c20','a7b9ce','78085b','a22f42','ab472c','a4a93b','a49191','a157d9','4078cf','a0e69b','71bd56','3fecae','a56d30','a10d82','aa7437','592000','502fa7','a4e0d2','ad5d39','a2b12a','a659be','c065e0','4ca8fb','a57aa5','ad79d5','ac6b16','ad234d','aba601','a608d7','ae141e','a7c8ec','a53cbc','20101e','32002f','a4db91','a9a063','a12160','a0ed25','a711db','ac0e42','71f808','a6a9ce','a11992','a7bf21','7c3084','a516f1','ab3bc6','a50bb9','a77445','7814e7','aa0955','a4ddca','40165b','0ac204','ade412','7c1201','a05458','c063c0','a106bf','48436b','a34aaf','a133fb','7c4269','aa3c8c','a4ebec','a5e82e','a2606b','39b10b','adcfe6','ae1899','a4c0a3','ab8adb','ab4f9b','a1cbd3','3feef3','a0e648','781717','a49197','a767bc','a32055','a0a529','aa0d9b','ad1462','ae00fa','71ba08','a8651b','a250de','a3cdfa','a036fa','a45ac4','a0b5a2','710361','ac69d6','a02c8c','a72f88','a85f3c','a69bb7','c80b20','a65e9a','a3914b','a1afd7','a0ee0b','a735af','a9f3d0','39cf01','aa0863','4d20af','780856','abaa8f','a960f8','710415','abd19e','a71b0a','a66f02','adca77','ae5b45','3ff065','a2cec4','a00c4a','a76735','a2feb8','a8def9','a53490','a4a97c','a6743e','a4f24d','a82e3d','a37cbf','a45319','a1b3bf','780bd2','a7348f','a027ab','a07b8d','003e53','ac03e5','a4e14b','a70a9e','adfdb7','a09329','adecd9','400d74','a8ee5c','ad26b2','a6666d','a5413a','a4f8e0','a8b283','a3dc89','acc2a0','a3e5eb','a3feae','a4eb9e','a0191d','a68b85','ad467b','781014','abceea','ac3ca6','a6aba5','a4eb8d','adf342','a6e26d','a57395','a80b24','ac18cd','ac6dee','a07966','3df743','31de88','aa9a59','a98c51','a0a066','a2666c','a5e64b','a2bd5f','a77d1b','a721f7','aca85a','a227d6','adcc10','a33ac2','a82f90','a9bd41','ace6b5','7c6719','a223c9','a16926','4ca1d6','a5d9fd','a26daf','a07d59','acbc64','ab1bdb','a5d042','a18cbb','ab18e8','a202c7','aa7e43','a0c125','a3a463','407521','3e6881','a41911','a01eeb','ac84e7','780b68','a96c1d','7815d9','a812f9','a09fdf','7c6214','78081c','a1e9ea','80087a','505c53','a9ddde','ad1b69','a08f76','ad48f9','7cf9c8','758226','a3d5b0','a2dddf','a9bf4b','a32be6','7c2bc0','a1e5b5','0a400a','ad20d1','a3f428','7c0a57','a9e23b','a0febe','780ec0','c08550','896c1f','888194','a63a3e','a321e8','a687ab','44ce86','a36c6d','8005b9','c070dc','a5b8a8','a8793d','3d134e','0d0760','78189b','a03a0c','a96147','a72783','a80cc4','ab8c17','acdb53','7c6d59','a98109','a52bbb','c07df3','a2d1a8','acb103','a525d1','a9a6ea','a1c203','acebef','ac99f4','a3d1e7','a17a35','a14e50','a2d079','ab9cf8','a07cc7','800482','a8f3ef','ab1352','3e29ca','a0e833','a6db9c','a93682','a6290b','a1f265','49c06c','7c627f','a40442','ab5082','a4f347','a597bb','a78af1','39c902','a6490e','a6312b','a7f05c','a2b1fe','a89f09','a9e976','a15aae','4d004f','44b8b9','a9e5e4','ac7d38','a9e4a7','a1b027','a121be','aa21c7','a5e1c2','a5a64b','a97303','a11a50','ad2296','460e11','a593e2','a2e1c3','4b4c98','ace6c1','a9f14d','a245cb','a00198','a5725a','a3059c','a096b5','a0c40e','a920db','a2d83d','a22b87','a04173','a61b4e','a6c58d','a37960','a81426','a6e1e6','a197a4','3ff799','a60806','a89cc6','a067c4','abf4b5','a5c892','a4d4b3','a6570f','a4c135','3d31f8','a25373','7c6691','c81594','ac07f2','a4a801','a58927','a97693','8980a3','a9804c','a4d89e','4783d7','a6c93c','3ff7f3','ac43ee','a24317','a062a4','393d11','a5e70a','a3b3a0','a79612','a485c4','4007f5','a1e986','ab3d16','ab030a','ad3cde','a0610f','a78a27','86e800','a20d16','a14802','503cee','a8c573','a6d6b4','a9fb4e','a2e0e5','3450d8','ae1e6f','a703df','a14e8e','7c29ca','ab8260','ac1dc9','87304a','aa90a7','a6e0db','3ed23e','a233eb','a0a09e','a5dad9','a190c7','aba058','adf1ef','ab10cb','c00f1b','3ccb72','ae4ea8','a9fd29','a91666','7800dc','ac3c0e','a57ca6','a0026e','a31cf7','a416e7','a1fa48','7c8056','a4d186','aaae5e','7c802f','47add2','a0391e','45a159','a4039a','a2cf14','484bae','a81e4b','a1f884','ae1e94','45cc94','76cdb9','a18484','a1eb03','75807f','ab4336','a313b4','a7c4f2','a4ed6c','a1bca8','3fee04','ab34ef','a515f9','ad6073','aa7238','a06b2f','48b562','acd0f7','a87509','a21ed5','3e6be0','a3c1c2','a44eeb','a6c45a','a868ba','781448','a100c0','ab01ee','a8ca8d','ab663f','aa0f8d','506dee','3d0eec','39242e','300741','ab307d','a26bac','ad6d83','a06cb6','adc182','a4ae9f','ace44f','acefc7','a4fda5','abd2b3','c8286d','a63fb4','a53249','ae0630','a69a4b','a10931','a6d6a9','e94201','4aae47','a3dd80','a7e310','a529b7','a69d31','4d010a','a727b3','a45866','ab7965','a25fe6','a3ed96','ac7c2c','ad49d9','a12167','ada296','acecb2','a9d8c6','a0b6fd','48dcc0','a16ba1','a5d57d','a54a9e','a1c797','7cf86f','3fec58','ae172d','a6c5a2','a6b6d3','a0a761','7c16a8','45d9d3','7502df','4b1868','a529cd','a31870','a6af30','a16519','ac4905','a4dbc6','a1706b','a87e93','7c7755','c01aa1','aae0c2','a12180','ada302','a5e27f','aaffe3','a9e52f','39314c','a77c4f','3e47a6','aa94c6','a8e445','a6a960','3ff0b8','a8f04b','ae1c35','a74a0d','a00137','3ff2bb','a9d71b','3932ee','3e191a','adee2f','44058f','a8edcb','89617a','a197bf','300824','44cc74','a95c49','a70a0c','a5dc31','7c6696','a67e3c','780607','a41b68','a314a9','ad7b69','a2c604','acbea2','a854d1','a6de16','04c11c','aa97ae','4ab54a','abddef','a6ba61','48aca3','a6e7ac','aba3a4','a000bf','a395d8','a59ae1','39b96a','a523f8','aadeb0','a14132','a03a81','4b5018','a067c5','ae00f0','a34b8a','ae04ad','ab92ee','ae53fe','a033eb','a9dce9','ae1735','a77487','a709ee','acfd83','aa1851','a6c1d0','a661b7','a01e78','78158b','a0c15a','a7d0df','45e350','a642e3','a9599c','8728c3','401bdb','a83ba7','adc69d','a76cb4','3d3fc4','a82980','ad4236','a3f413','a0c993','a96805','a4a8e1','a62bb7','780734','7c4c4d','a9d75e','a7c75a','a04a79','ab5bd5','4850f6','4242f5','ab5f8e','150690','aa3b2b','3c436a','a22e49','ad67f5','4404af','3d0501','a073c9','a9cb46','a93f7d','a63a9f','aa2ca2','aa1f16','a7f2e3','a185ca','4b1ba4','a9ea58','a4f68d','ac3214','ace5ce','484f89','a38195','a05627','a40bb2','3ff832','ad7d4d','ac0e62','ae1e7e','a19604','7811f3','a389ef','a011af','7c2496','424480','a0dcd6','ad8f7a','a5a8b7','a58217','a05467','aa9ebf','a53601','3e0a38','ae04c7','ac7363','a63592','a1afe6','a022c6','a22995','ab310e','a54fb8','a2e897','3e71f3','40727d','a7cfd2','a29088','a26e92','a39711','4ca515','a559d1','ab5ec0','a20b35','acc31a','acb963','aca150','a5fe23','7c7f44','aa36f7','4846c3','aa74d0','a70dbf','acdcb4','ab4165','a4e746','a32ddd','ae1149','a3cc30','4b5109','a2d905','a01ac7','4d20f6','a82300','a66c5d','a27921','a7351b','ab0d62','a65560','7c585c','c001da','a2cfcb','ac6c83','aca4a5','ace9f6','a49995','06a081','c816d6','300803','a96719','a02af0','a262eb','477ff3','a1059b','a89b64','7c175e','a9b283','a4180a','a982fd','4d2190','ab86cf','a08ef0','43e7a2','e4923f','a8ab18','a34fae','a39fcc','3e0229','ad87b2','a07b9a','adf63b','aa48fa','ad6ef9','44b434','a0fa44','ad927c','71c308','a38847','a007e8','7c8067','a0407a','a641c9','710312','a93418','4845c4','a38252','a28578','484639','a6f6d4','ac2190','7816e3','e48e74','a1185b','a0d049','a14907','a626ae','3fecf3','a9bd1d','a69988','780e10','8007e9','aa1aa8','a83741','a9d6eb','4071aa','a1c86a','a1823c','a8898f','7c46eb','a7287b','3ff6d2','a8e083','3d1ab9','4a9099','ac7076','a59276','a2439c','009425','a55aba','a2aac9','3d3b94','a4813d','a0f0f3','ae2edc','a0031d','ab893e','a8a1bf','505e66','ac297b','a9e8ed','7c6c6f','4849c3','31dc5f','a2709c','a752a2','a08d3e','c01442','a61b14','a03bc0','7c6dde','a3fc1c','7812b5','780201','ad85eb','a00037','a7c465','a99ba2','a965a1','4247cf','a39d89','a1573a','a9bebe','7c7ba5','ad264d','845f78','a03007','ada795','ad78be','ab1aca','401139','a128a5','780dd8','a941c0','a8890d','a8dd64','7810b5','4b253b','a4e704','a84151','ad8c5b','a9db3d','a62d8b','a5c1a0','a0de3c','aa546c','ad5dd0','ac5da0','ad0e28','a06465','a1a470','ad3c20','a2bb08','a9eb99','a793b2','3d057b','a9e822','7100fa','ac51ac','7c60c8','3e5b0d','a00945','a84305','a9b65b','ab79de','a221d4','a83869','3fec6b','3d2c5e','ab667a','a5ec0f','aa2487','a24ea4','396f7c','aaaf0f','7c80f8','ae020f','a6df12','a2e6a7','a6464b','a3d0e3','c04d50','acc38b','a96eb9','a75d7f','a4d9cc','a2918e','a0f1b1','aaeae2','ac98ab','a82a58','8980d5','a9567d','a3a1a9','a687af','ad4eeb','3d238b','aa0d2f','a20891','a38d9e','a919b4','a97162','a53b19','3e2942','a008c5','a5c16b','a59014','ad5200','a6027d','a4aa82','aa228c','a8bcf0','405987','738071','aaa593','a5ac6c','c086b9','a8fc71','ad6900','aad288','3f06fd','a4e5a9','a2d626','a02a91','a97163','7cf9ff','ae123b','a6b5b5','ac0de6','440687','86ddf0','a413f8','a8def1','3e5c84','a496fa','aacd4d','06a0b5','a1b782','a86427','a05380','a390de','ac3d88','a9c0e0','3d0fb5','aad386','a72999','a535a3','acd733','a8d2ca','780c5f','a18021','a8e382','a1d76f','a7ccce','a2b34b','49849a','ac7a43','abdece','ab3e9f','a890d8','ac925d','a69bfe','aab40b','a9d334','a0cb39','a9221f','a68749','a24c99','ab3b1d','780970','a6cd25','a021bd','a37c00','a077e7','7c4e49','a1e857','780f13','7cf8ed','ae1e98','a45a78','478b61','ae0750','ac1c7a','a9d3a9','a68a03','a54bb3','a93593','a3a1c6','a20bc1','acb772','a4a46e','aba917','aa2cca','ad2231','781703','341202','a005c8','aa02ac','a39de5','a35be4','ad3e82','a727ce','a3f1cd','a297b2','a71ded','a5fb70','e48b4c','3cc194','a15186','adc88f','a65ba7','a992b2','a8107d','3df746','a69ea3','a8517f','a21c0f','4d21ff','513024','acc8e1','3fec17','7c7a6e','781808','aa1add','a7b70c','30070b','896328','a0ba0f','adbe0a','a13804','4cc4aa','a3bd64','a556a4','a87e26','aa940d','ada554','a071c8','a22fa0','ac84b4','aae30b','a9d83f','a7272d','a1b508','c07b42','a4ca8e','a54ec0','a0b5f5','a9d60b','ae61f7','a62b13','a5487a','a515eb','a74873','a28baa','abddf3','7c1abb','ab2b22','a23ee7','40150b','a99514','ad5885','a5c166','a6eb66','a09737','33fe92','a43f6c','a88b45','aca57b','a81fab','76cee5','3df79f','a69e29','800536','adec02','a5527f','3ff7a5','a635b1','ac9c61','a61afb','a85e9c','89653c','a6a282','3cd439','ab8b1f','49d3c6','a2473f','aa2a83','3ddc78','7cf8ee','a48957','a437eb','0d0ac2','abf5da','45e313','ac5c20','7c7a91','7c7ca9','0ac94b','396063','a56bcd','a2cd45','4b4e70','a23473','a4b964','7c7a85','a2af7c','76cef6','c07923','ac9af4','a0c3c9','a9643e','ab13dc','a07da6','a031bd','398df5','a509db','800733','a07844','a9bcf5','a026a2','a9a0e1','7c25e7','a96fb4','a482b0','ac423c','a6611e','7c19ca','a2b1ef','ac1c7f','a5b923','a5a370','ab83a2','a63e04','a5a764','a2d1d5','342316','c07797','aaf67e','780760','abe8de','780e56','47b372','a07f4c','aa7f5c','780d03','ac95c1','a2f4ad','a3e298','8964a1','a481ec','38dc7c','a2fd21','a5d487','3fedd2','a9dbfc','a66a10','a69bb8','ade24e','add3a0','a0676a','a7b6a4','a29dee','a21476','a0761e','346318','a0e995','ac6fa8','ad0250','aca21d','4407fa','a944ea','acb733','ac6b86','a23714','a006dc','a1bd75','888139','c80604','a7c676','8694fa','aade7f','a43f2e','aa301a','7c444d','aaf9f7','a476e6','4d211a','122136','a4a80e','a99796','a19d14','ac2233','4b254d','a1ed0a','a7df0e','a6ff7c','a1c623','a39c8c','a14586','a1885a','a9ba86','a57edf','a9da08','c05977','ade04b','ad56c0','4b3f93','adbd46','a700eb','7c0831','a25f0d','aa5b6c','adb8cd','406f2b','a6a9ca','aa2fe5','a97da1','a7890e','a9b317','a31528','a140c5','aa35e7','894098','150692','aab6ff','a788db','a5c2af','a3286b','a152d3','a9b7b1','4b8472','a8a9de','abb69b','ac70ed','a13f19','ad7f33','a046e7','a36b39','ad514e','ab6748','a1dd98','ae58ef','ad806a','7c0c40','8990d8','aae61d','a6e0b2','a1b502','a5103f','a83e7b','aa6847','a0da7a','c0512e','a5ad46','a544de','a0f9b5','a0f8f9','a9923b','ab563f','a68c54','a61f40','ab4de7','3c0cdb','a7d247','a08712','a54ec4','a11f73','ae2fa6','a00d48','ad4629','781277','abbbbb','a7f90d','4077a6','a19a7c','4b4e40','3b776a','a10f85','a4533e','a0f428','a34207','a9e602','a9de25','a841a7','a44d81','a70589','4aaa19','ae54d1','aa2744','ad8252','a383e8','a3710f','ae1ebf','a67767','a5184e','ada999','ae570f','a2eaf8','a2b8b8','7c7ca0','ac1751','ac67db','a88b65','a73ab6','a63ddf','7c7b63','a5579d','a231fe','a2afe1','4242eb','39a419','adff56','a7c8d8','ae1ebb','a1e74d','aa384e','a7093e','a242b2','ae08b8','ad9c6c','adc8fa','a181fe','7c01e7','7c1293','a601fb','ac5b62','401eda','a955c4','a3b822','a02033','84b860','a20691','a43b45','44080b','ad3c55','a3df4a','781127','a97654','a8a2ee','ac1fb9','aaea84','a75ae5','a69185','a66420','a7cc81','a88467','a69b98','ab5383','7c298c','acd03d','a185bf','458c92','ab5797','a310f8','a05e91','3c5eec','c8284a','a04412','acf856','a9d316','a9b8a7','a622aa','a70a67','a4eaa1','a02dd5','a4ea27','a4fbb7','a5ec0d','a368c2','ac65a5','ad23ce','a4ff7a','7c593a','adbac9','a066c7','3007af','a0aa55','a77774','a26574','86e5ab','ae145f','a6683b','a1d151','abcdb8','aade33','a0026c','aaaf51','a1ef65','a54045','ad40c7','a68307','a24650','a570da','c01d22','a6bb9e','a4dd85','a065ef','a34e2b','a78b17','3ff574','ada9bd','a9d15e','a19e34','5000ce','a77326','a1c299','a0beab','405e94','a4f8df','ae1145','a9d66e','a135ba','ad4cb0','a04f97','4ae2af','acb854','7c33d8','a6935a','a50711','adab25','a2796c','4950d1','3fee2b','ab132e','a20d1c','a009f2','aa0e0f','780b1d','a25bf2','a1cd1d','3423d4','adc023','406f7c','aaf3df','ad6539','a434e3','ac4338','abf692','a4f75f','ac979e','a88978','a4ae2a','ac2668','abb392','a4a1f0','ac5514','a8642a','a832de','ad574f','3e5fd7','a0abdc','4855f6','4d0202','c025c9','a506c6','aa3476','3d15ee','a7bfb7','780305','a3afcd','a69f08','7c6cfd','ac3bf4','a64193','8a07df','a9be43','40393c','3feb20','a529a4','e490d1','3ff7e1','a3891a','781509','acd464','a4ab61','7502d8','3e1757','ab8265','a8d6d9','a51b96','ae291a','adc5fe','ac53ac','c1e5a6','ae20c4','ac605d','aa1edf','a6d5d1','aa6ceb','4aaa70','79a076','a3e6b1','a33fab','ac13df','abe131','a7cf35','a4c651','a1a9bc','e4926b','a3e617','a99b4c','ada1db','a2b161','a4380a','ac26d6','a12feb','4caca4','a1c9e6','adda17','a2e22b','a33254','a9deae','a4abff','7810bc','a604f9','a19865','a257e2','a41b17','ad732a','ad4781','a518c4','a45f76','a4e604','a25a74','a01df8','aad116','a5e5a2','780da9','400c21','a30fb9','a22927','a572e5','a39a9f','485785','a507f0','4076c5','4aa443','adffb2','ad5d21','a294f2','a4c82e','4050c0','a36b6d','a08937','7c4050','aa24a8','780730','ab11d8','a1ffde','a14b8f','710355','4851f6','896523','a4459a','a9e8f7','a0ccb3','a4748f','a7059e','3e7256','43ec2b','48455f','a7ca42','a06de1','a3edaa','a07c1a','ab282f','a649d1','a14499','ab83f8','71c321','a41890','7c0810','ad77d7','a45929','ab6af3','a61cc0','a705c5','a60c2e','a1e062','abf696','ae54de','a08b0a','a5d2dd','4d21a6','aa1929','a563cf','7c0f30','aa3908','ace49b','a37c9a','a72dfd','a7753a','ace893','7803f6','49d108','a825d9','c053de','40066c','ab7543','a10638','a0e76f','a54713','ac73b1','7c718c','488d84','a4cace','a973da','710311','7c5bf8','7c4e82','adfebd','a6f403','758525','ab82e2','a0fbcf','a97daf','a7f2a2','a9f644','ae58ea','a82fb8','c008f0','a24b25','a063a6','c06908','a76c27','a3e9b2','c0771e','abbb00','7c7c3e','ae1ec8','aa80c0','ad369d','a58a19','3444cf','a83477','a2ee12','a1c386','7c1758','a991ad','a6da8b','00d170','a954ef','a8090f','aa3836','aa935a','a6936e','ab05ac','a7f6f8','ac3078','a710df','a94e1b','a1b22d','a0ae55','a84f93','7809e8','a37cfc','a7b7b3','ae0467','a9c79a','44029d','a221cd','aa252e','48ca6b','4b2cca','484204','38469a','ac8a7b','a1a55e','7c2a1c','a68d2d','abac24','a16479','42441b','152ad7','a092a2','a97a7b','a8d717','a88e4f','a6cc29','4bb1e6','a3a967','ad5973','aca455','a9891e','894080','aa19cf','396406','a7dba2','a0876d','a04506','ae6470','a1bbde','a19738','ab7a0c','a83e9f','aa40a7','a747ac','a0f133','491463','ab968a','a6f68f','a4f233','ae1167','a4df89','c04e70','ae11c2','add9cf','a7860c','a3e08d','a8b58f','a6191c','a64900','a00179','a08a96','a0b617','43ebdf','a3db99','a1f0ae','3d7f8a','a77a51','a65509','a4f261','a5e726','ac9747','a9a8e3','3d6502','a062ca','3d2104','a4bb8a','aba28e','a709db','a4e6a2','a0dadb','a0856a','800b05','acd4f5','a54941','ab5ad8','a05dae','a68fef','896e94','a4335d','a16710','7c443f','aa262a','a307e7','780bfc','abf67f','a7610e','a69326','406a9f','a0de5c','ad2636','a6a215','47c43a','a10912','ad260e','ac6f38','a6beaa','a4ef90','ae1fd7','a8a6cd','4b1904','a4fc13','a1df65','485d30','a81efa','aa1d82','e4814b','a545a9','a3cbc3','a0bbe3','7c7534','4ca8d8','adc7aa','c8276e','a857fd','ae1ea3','a64933','acdd6a','aa3527','a76051','ad3474','a7767b','a53c39','a88d55','c06b76','acfe50','a032d8','a55b09','a33475','a434fa','ac9938','aa1078','a6593e','ad3869','a7b1af','a43050','a120fe','a32c23','a6b183','aa1846','a63bd4','78109b','3b7b64','a4c811','49d436','a5cc8b','adef94','a565ba','a493a2','ad2dd2','a496cb','a30850','4242b4','a08ec1','7807b1','a9a82f','7c6785','ae5dfd','ae1f3f','ab4bb5','8964ea','a1248e','ab05e1','a356fd','a07645','abfef3','aaa5ff','508020','a7bf4a','501f3a','aaec57','a39b22','a90b05','a657fc','70605c','48476c','424782','a3e846','c0832a','a0f642','a05254','adf69a','aa781f','aa6ae8','4b8565','ae6238','ae29d8','a0cb9e','3ff16a','78920c','4076ca','39ca71','ab9723','a89aed','7807c6','c02986','3c70af','ab213c','a96f07','a6d9c3','a9ea79','a36d89','710247','ab9073','a8fa6c','a4204e','a44453','ab7fee','a2226b','a030cf','3ff66b','ab9b0a','0b4030','a371a3','a22235','4b19a2','3d1548','ac4878','a18d6a','a12405','780ca0','a3133e','a1bc2e','3d2706','a57aa2','ad5a04','a55fd7','a552e1','a41ee0','a78dd2','4072e4','a5ef8e','aab5cf','780b64','47aa8d','c03171','a4f668','aacb6a','a6f82d','78096f','a6ddb9','a7d653','a7a91f','4249ca','a0b404','a0ef1f','a54bf7','3c9432','ab3cbb','a9da7e','a5bbf7','a4d2b5','a2c57c','c824f5','ad267d','a7de0b','49d2c4','a9e396','a7c4b2','e49680','a14052','a24931','a0f150','8e91bf','3d136c','a230d1','4d2237','a81bff','a2cb12','adcfa9','a28a34','a6469a','a998ff','ad7203','a80d47','a09239','ac2014','a817db','ad04f2','4249fe','3532c1','a05f92','c086bd','ac5402','a8243b','aa1baa','a57113','a94da4','899080','7c7c45','a70181','4b0b11','a8d9a0','a241e8','aa37d2','ac8d18','a38d20','a0ebd9','a1d634','ac7321','a28d4f','a6f7ba','a384a3','a0106a','7c0cbe','aa6dd8','a96122','a2e615','018078','7c75e4','888178','a6030e','155c08','aa6963','a153f8','a05e6b','406cbe','4242d0','ae2046','aae18d','a2fc52','a2eed5','a21cea','ab5033','a9dfcb','a5fad7','c038aa','a3f1d6','a72bbd','a1d28a','aaa818','a6513a','a651ec','7c6f68','a465ac','7c7c43','39ac49','a184c1','a15a16','a1663f','800579','a38527','8a080a','a1bfe1','a740f1','a49c7a','a06ff3','a8cada','a7b014','a43a58','a40623','aca978','a1f581','4d2239','8964df','3ecbb5','a2f35e','aa02d5','a28261','ce089e','32003c','a0e687','a3f4c4','484ace','ac916f','ae1117','7c6698','780ef1','a64955','a4190b','c81bcc','3ff1c9','aae322','a6008f','a9762b','acd181','4c3f2c','a10d4f','a34d28','a2c125','adda1b','a18fc7','a68a31','aada90','ae2edd','c04693','a010c4','a74815','894065','a35256','ac731b','a60cb7','7813bc','aadd85','a82f12','a6ec7b','a0956b','484846','a70a13','a559c2','49d3ce','ae05af','7c0838','a6778b','5083c0','add0b1','780e47','3d39b3','ae4eb8','abbea2','780c12','ab232e','3d1f23','a534ec','a4fbca','a52e4e','a005a6','49c217','48c583','ad26fd','a3d818','a95e5f','ab95f2','a05434','7c6286','a5d67a','a2a817','ae1f6e','a9dcc6','a2a455','ac4fca','a53ce0','a33bcf','a446a1','a52a5f','a0b265','a7ecc5','a54d3c','a00201','adabbb','ac2619','ac892e','ae0454','a45a94','a856f3','4405a6','aab935','a72bf9','a93fd0','a987c0','a8b744','ab8247','a6b61b','aa5321','a45edc','a2264a','a074ad','ae0db3','4b3ca2','ad1104','a449e5','780cff','a1fb23','7cf830','48d961','a2e907','ab5f29','a96deb','a29bc3','a23699','ae4bd7','a856ee','7c1c49','a1b024','a002db','a773f8','acd329','a22b1f','ad9ea0','a709a9','ae54d2','4407de','a67535','4a81f9','a40e59','a5e846','a6ac17','ae1110','a7fc9d','c81561','ab3647','896584','7c1fc9','a698f2','aa9bc1','c0725e','a1ac0c','406dec','780d2f','a9eaf4','abd685','484802','40312d','ac9923','a58c61','a4be4c','a2cabe','a642fa','a54973','a27e94','48c980','a31d18','496c2d','aa6d40','78070e','a20e32','a92b22','7c39b9','a72e6d','a95ba2','0d01f6','ac871b','a7b7ea','a4d521','0d0821','e8044b','a53ec4','4b43d0','adc943','a6b799','a69e98','a92043','3dd373','3d3cc9','a4fa6d','7c2a15','a2d5bf','a02fbe','a6b7bb','402d72','7c6684','ae01fa','a34fcc','a92942','a67706','a773c4','a0c8c0','a96321','3c4b2c','3d2298','ad2103','aa0824','ab1d02','a36493','a054db','ac7019','a809de','ab0841','aa40f5','a8a6ef','a416bb','aa7e86','a2f7d6','4b42f1','7c250e','a90ffc','ab680f','a00a70','a066eb','a751da','a1c0a4','ac5436','ab875c','ac94d4','a8bf5f','a54287','aae735','a4a539','a4955e','3d386c','7c3ced','a03510','a7a9fb','a05b59','a5dacc','a4de94','a1351a','a54f60','a010b3','781392','4843f3','ac9f38','adeb7b','a32890','a1e16e','484eed','a39c42','a55b0e','a1c00c','a184bd','a53154','400561','a9535c','aaf02e','a8b7e6','a0f357','a0ee9f','42435f','a0d6dd','a87bb8','abfbaf','780d50','ae6237','a95173','4850a1','a2536d','aa4d2f','a124ae','7c6045','ab4102','ac6882','780628','7c6c3d','8964a5','ab7cb0','468476','3d3f2f','142583','7c4441','a0c454','aadfad','780d78','a9c8e7','3ff7ea','a4caf8','a652df','a3944f','a1914d','031298','43ebc1','39e61c','a23a31','7c0a75','84622f','a4c244','a1ac38','adde4a','4b0bd5','aa67e1','a57100','aa1e04','aa70b8','a38281','a05903','adce0b','a16bf6','a121f2','a0418a','a5d32b','ac6f37','a6acba','440562','a89119','5113c4','a94937','a26f6f','a022b6','405477','a7f775','ad35bf','a80874','e80452','7c009f','acf399','a25637','a19da5','a9ef42','a9caa9','a9b37f','ad4652','ab5ed8','a455cd','4caa33','ab1859','7c7cb0','a83c1d','3ddc70','a426e0','ab087b','4607f8','a63f3a','a2a2df','a1d296','7c6ba1','a7d668','3951c2','ad40d6','a9ce0f','a29aff','a42f5d','3fefaf','a84307','a7ff9a','8963ac','aa84f8','a8e713','aabe2f','a3a21d','a1a74e','7c5be5','ad3594','a682b5','3ff297','aa0b93','86e5ed','ae47dc','a1226e','a4a41a','780a2b','ac9aec','a005c9','ab99da','aaabd3','7c4513','ad246c','a83184','4242b9','a9ee7e','7c3cf1','ae58f2','ad4a6c','345353','a08efd','ae174f','a2eac8','a2cb6d','a9727f','ac857d','a82b2b','a69cf0','a3c180','49d111','a6749f','a6cf94','a37b29','7c07c4','47c11b','780b05','780a3e','a53340','3ff094','abd203','a214de','a035db','a9e34f','a15360','7c19cd','a8a608','a442b2','88510e','ad0a0a','3fec12','a0b75b','458c34','a84a74','a744a7','4ca1c3','7c15e1','7807aa','ace7e7','506dff','a33376','a00901','ae5360','a69b70','a394f8','a25526','a1d61b','a04b89','e49163','44d1f9','a9ba57','a2c20f','a678e5','ac22d7','a49343','3ff762','a8006f','3e6c15','0c414a','406b1f','aadd95','4a81f4','4ab48e','acd15f','a7565b','adfd75','a5d7b2','ad2351','a95078','a2e78d','a78c44','a5092b','acc519','a63aa3','3e3a2c','a1ea5b','a549c8','a82213','ac2375','aca99d','a04b24','7c5bc1','aafd17','485000','aa24fc','4b4b5a','40293c','a4e39c','7503fe','ae648f','a087d8','ab082a','a61c8a','adf0a9','a4afe4','adff94','a1af39','ab25e7','aae984','a03820','aad9db','e494e8','a8b46c','a59ae0','a13b1e','ad0122','a91af0','a84f35','ab5da3','a5123f','a9e6cc','7810c5','aaaf2a','3d0888','a500b3','51012b','a63a27','a24d40','4d0345','a70522','ac76d2','a78445','a19d84','710128','aa843e','a1a0ba','a17455','7805ea','485cfd','ac8305','a1f669','4b2902','49d275','a89558','353508','c066f2','a5d26d','a448dd','ac9fed','a9a709','4041da','abb4e2','a09db5','ad0daa','aa9a7d','a32ffe','a7b444','abca65','a9e5c0','a073e0','3e6b63','a9d2b5','a9b3e6','a7fbf0','34210b','71c005','a12b89','ac1ddc','a1e189','440841','3e61d0','a6bce0','abc6c6','a3b8da','a32c39','300709','ae690b','a48416','a0d517','aa9cde','a1d0d8','add373','ab5c67','49d44f','4408f6','ad0343','a5b4b0','867fe8','aa0ef3','a996e1','a5dc18','a1b980','ab42b2','a92c25','adc375','a4eaa2','a1b063','7c3df7','a20a75','7c1b5f','406ecc','ad1c11','a9db8d','ade18c','a499a5','ac4e81','406de6','a26148','a20f02','a709ff','a5d4c3','ac9af5','ad6f3b','a05810','a16794','40345f','ad89ba','43ea47','c03d0f','a9c22c','a6c991','a22dec','3d2d85','acac2e','c005ea','a9eaec','a95ffb','a76b63','acaad2','a735ed','ab8d76','3fea33','ae4eab','aa38dc','a0b6ea','4b25b0','c02de8','a174f3','aba507','a62449','a14059','a22f57','a1aa85','ad6df2','a304bb','76cdba','a50e85','a76531','a1022f','7c355d','aa9107','abcd77','a5696d','4b43ac','ab4728','780d5c','a13624','ab8820','a6b074','498b24','7c0c43','a76f6c','06a13d','a000ca','a40d51','a3b393','86da5c','ab89c7','a797d9','a86f52','a183c4','ae4a11','44081c','48507c','43e947','a99a4a','a91315','a79158','a23882','3c458a','a7ff3c','888133','424a72','a31d82','a52bd9','a0ef02','3ff72b','ab2786','a3c603','a4d946','ae4e9d','a42645','a35ad3','3b76a5','a52933','39d71c','a638aa','a10095','abd520','adbdd9','a43a98','a7e627','a22e28','49c1b2','a73ea3','a5e550','880a76','a298b8','ae61fb','a0cd0c','4076fe','a50b4d','8990f6','a8be67','a7d159','ada4c1','a716e5','a103b8','ad6f73','a3313b','aa84f1','ad941f','a7c866','06a1c9','abec6c','aaed7f','a5707a','a296c2','a013fd','3d2390','a0cbdb','abc274','a81c5e','ad8f0a','4ca857','a0660f','c05940','aa0875','a9bd90','3c3f31','a40f1e','a1f2f9','ac7ac9','aade37','ada7c6','39e91c','47c59b','a9d2d9','ace605','7c79ec','780290','a9d103','a6cf4a','a0917e','aaadc3','a1d235','a65512','a5cc49','780f85','a3f6d9','3d0c39','a478cc','acb240','a44af9','a23e63','4b3a2e','a4e569','780bf6','3ff6f4','a1fcf9','c08568','a833bf','449052','4ca842','4ba94e','a1976b','a0e33c','a900dd','a697b0','a585c4','add7ab','abc253','a3d7aa','a3235f','ab5eb2','3d1f9f','a3f3c2','a4fc31','a11d22','a649e8','a24493','adf41a','498422','a5324b','aadead','a19153','7bb0bf','a4f8ab','7c8089','80087d','a955b4','a08038','a9dd21','7cf8eb','a2f2d8','a02b7c','abd3bd','440996','a63aab','88084c','ae044f','ad063c','a5ab00','a15a4b','ab5acb','505c8e','adee4d','a0d456','a6e441','a4c2f8','42454d','a29d11','459901','ad8261','a71724','4d2155','a683e0','a2a213','a5c01b','7802ad','a77b76','ad6b08','a4d5b8','a05d82','a1dc60','a1e051','aa1e41','a80467','a82467','a7fb8c','aae0db','a938a9','a432c1','440002','a963b6','a4b1a1','abb7c7','a1804c','a079fc','781410','a570d1','a261a4','aac6f5','780ad9','aa055d','a60ba5','7c7ae9','a6bbb0','a50ddc','ac2249','a1896c','a75c12','adc531','ac763e','a0aa48','a675d2','a697b4','86e780','adb590','a61a4b','ac710e','ab885b','4ca615','a6c340','a9be20','a35e66','7c669b','4b3377','a4d5da','a8c006','a530cb','75847a','a06b9e','c00db7','a57e98','780bd7','a7bee2','a24df0','3d33c8','ae0206','aa357c','abc318','a2c934','34508f','76cccf','add386','a86428','4a348d','a95d44','a04ecf','a56618','a876d5','a82ede','a5d7d8','ad0ff8','a6c586','781132','a808c2','ad4696','a788b5','a06d8c','abbad8','ac3c40','a4b30c','a113cc','c804d6','a2aef2','a1e42f','aaaba2','c01d1f','4403de','ac1b7b','ad73f1','a07ab8','acbd52','399483','a69d41','a9e5ea','a392fa','a33829','3d1005','c051e2','aa732e','ac5487','a815cb','a7fbb3','3e5d2b','aa4b63','a42f04','a860cb','ae11ad','a9568c','a818c6','ad0b5e','7c46f2','a18eed','a61abb','4b1920','a4f21b','a26964','ad8826','a00cc6','ad1747','ab396d','a3032c','a99401','406ea2','a23507','a0cd48','7c5cd2','a4f47f','a3e4d8','7c42da','a6cce2','78106a','a5e42b','3004c2','a55753','a33390','a20538','a5a690','a1b69e','a14706','7c62a5','ab0b56','a4f228','4b0897','a0ee6a','a325b2','405f49','a48e00','a40154','a73ac3','48fe04','a57773','a75a21','3c0e3d','479694','8a05c8','a0723d','e48f53','aca89a','a6b4e1','a1402f','a09a04','7c432c','ab3446','aca60d','a1c828','a5bcc9','70c0f9','a7c439','3e2bf6','a81022','8880ad','a599e6','aca35d','a66675','a92e1a','48c232','a67ccc','3935ae','a2c2d6','ae04fa','ad225b','ac420e','a6416c','a352cc','a0b1c5','a32648','a2a9ba','7c580a','a70ac6','a95bf7','a7e693','a16ce7','800d6c','ae5a78','7c71fe','4b4002','acc372','a83d28','ad53ea','ae5bd5','3ff396','a93140','ae58e8','a5214e','ad1aef','39cf00','a83c9b','a32312','a24d36','a2e25e','ab7e41','4ca782','a9d74a','a9c018','344211','780dd2','a52661','8a05a5','3ed60d','ab600b','acdf05','800892','abe94e','155bbf','3febfc','ab0229','885178','a9f2d5','a5fa72','ac3d09','4cc52a','aaa861','a50316','aa558a','a95ca2','4ca336','acde74','a7f186','a14d65','a7dc71','c819e9','3ff214','aab109','acbcfc','a555c5','4851ae','a1c4e3','ab45f7','a598d6','8a0633','a7b899','a39e55','45ce13','a311e5','a29a14','a9ba16','a3b69e','abe528','ad275d','a61584','ae07c6','800b63','39b486','06a1ca','ab4dcd','ab1b87','392462','a06a02','a00b64','aa1b39','7b3440','a5ec53','a9276d','47807f','49c241','acbcd1','aacabf','a2a436','a9ee42','a8214d','a46219','a174e5','400dcd','ae5f3a','ae03fb','a6aade','3ff1a8','ac86ee','ad5b2a','a8bc4f','ae0093','a5840b','a8dbc9','7813ff','a14dd3','a18304','a1336d','4b1917','a42650','a2e9cc','abca58','aba652','a7c668','a4f7fb','4aaa72','7c4516','a10e73','800b61','ab5db8','a08422','ac8b13','a24d6b','a0b580','a025fd','a6bd51','ab27c7','a0bc5d','a91b14','ad5b5a','ab3fbf','a63ecb','a0af28','a69538','a420db','ae2ed6','a2a2e3','a18cbf','a18d60','a4cbf1','a080aa','7c0ff2','a4f117','a34b8d','add25e','a7d37f','c0019e','780a8d','a50400','a997e5','44a7c0','3ff0f9','a0a9d6','ad3302','a9e8c8','a6aa0a','a960a9','a3c64d','a82258','a5bca5','ac1d36','a0452e','a70a47','a04fae','a143e7','ad8906','a539a6','781706','c06647','a479d8','398073','e2002a','a91859','a65aad','a20388','a7f05b','a46308','7c3382','a53ae8','a9229a','e49449','aaa02d','78922f','ae007f','a72bc6','a47184','7cf9fa','a26295','ae21de','ad4625','a9edfc','ab5e75','a4462e','aaf89b','a0477b','88819b','a63e9c','a28c86','44016d','a3e97e','aab340','a75eef','75862a','a409f3','a2afa6','424292','aaf936','a69e14','a22404','a3e326','c055f2','012abc','7c593d','4d0107','ae4e96','a7e6c7','4aaa12','47bf10','a175ce','c0575d','a42907','a87876','a33a2b','a50d23','a46f7d','485228','a36348','7c4794','a9ee5a','a73e4c','a34585','a042bd','a9e1ef','a51099','3ff71d','06a0aa','a5d421','896378','40754a','a69df3','a00324','a07423','a821ad','a371ec','a14ae0','ac6658','a6b3a8','7818c4','a5bb67','acc757','ae290c','781851','3d18b6','a84e07','a0891c','a36c97','a76aed','a18994','a27134','ac2a1f','a84c55','a69d23','a30736','a07046','a1021a','a2bdc6','a24cc0','a688fc','a267ec','aaf4d3','a80c05','a06b0b','0d0aec','a50c35','a88dbb','781843','a63748','c08399','a54819','ae114f','a4834a','a21e1f','a00d80','e48ca7','a7fd3f','4b3667','44a3c3','a23cde','a6c7e7','a30069','45f054','75049d','aa8419','3e0f76','ab0c73','a7f66c','780d01','a6c5b1','ae2bf0','a48666','a6946e','aa77f0','a0f09d','a0d4a0','a29d89','ab7574','a076a6','a54be3','a6eab3','a3c031','3ff255','abbeb7','4ca84e','a1cd10','a033ab','a1e0bc','ad5cbc','aa4089','a500fa','ae4ee3','a1ca5f','300736','780bf8','ad1d0f','44a017','86951c','a61ad5','a0031b','485b2f','ae638e','adaee2','a73a77','aae529','a1d5fa','4077c8','ab649a','ae58e0','a1de26','780d5a','ad002c','a4e38e','aa3abe','710315','a27706','a1e388','a04997','3d3e71','3ff34a','a4ce30','a6e9cd','a6be13','a64e13','aa3968','a81c98','a33bc3','a3b8ca','4bd243','c07ccc','aabcec','a45b33','a0bd33','b19752','a67aef','a03356','a153f0','a24528','ad6182','a6be17','a4047c','ad8c42','a559ca','a54e71','abc1db','a832be','a69aea','a6adaf','a1f875','a58d63','a12186','ae147a','a6ed0d','ad7579','ac6073','a986c4','a8f0dd','32003a','ad444c','a411b6','ad520a','a8522b','85c678','aaf2c7','a9da44','a6812d')

## si quiero consultar : tstzspan_out(timespan(ttext_in(callsign_array_str)  )   )  
## como no anda minValue para tomar del callsign el  unico valor presente, lo hago con regexp